# HT5974 · Exact raw-data provenance

This notebook reconstructs **HT5974_A25_P384_08250.csv**, submission **58180681**.
Public score: **462.414**. Final score **475.750**, reported by the participant.

**Only data input:** the unmodified `analistasr_competencia_2026.csv`.
Every feature function, native parameter, ordered feature list, seed, and reference
fingerprint is embedded below. Run all cells in order in the pinned **R** kernel.
The original feature banks, models, predictions, submitted CSV, and competition
service are not execution inputs.

### How the final model was obtained
1. HT5954's bounded five-family search selected rich multiclass configuration 9
   (`9165b1857eff259e`): 1,914 features and 192 boosting rounds.
2. HT5966 retained every eligible row, downweighting March 2020–March 2021 to
   one quarter before global mean-one normalization; three seeds form BASE.
3. HT5967 introduced strict-prior six-month robust funds/payroll features.
4. HT5973 rich HPO selected the frozen 384-round MAD configuration reproduced here.
5. HT5974 tested blend/contact policies; the submitted policy uses 75% BASE rank,
   25% MAD rank, and 8,250 contacts. This notebook freezes that selection.

### Exactness and resources
Exact mode requires **R 4.6.1**, the recorded package versions, and the recorded
LightGBM native-library SHA256. A generic same-version installation is not an
exact-runtime substitute. A working Rcpp C++ toolchain is required for the embedded
original kernels. No packages are installed by this notebook.

Use the existing ~247 GiB cloud host or an equivalent Linux environment. Dense
construction needs roughly 65 GiB for a matrix plus native/copy overhead; a
136 GiB available-memory admission check and a single constructor protect headroom.
At most three four-thread fits run concurrently. Monthly gzip caches are derived
from raw input and can be resumed in this notebook's own output directory.
Allow several hours for feature construction and six full fits. All expensive
stages write progress and committed checkpoints; a stale constructor lock needs
explicit inspection after an interrupted process.

| Setting | BASE | MAD |
|---|---:|---:|
| Ordered features | 1,914 | 1,920 |
| Boosting rounds | 192 | 384 |
| Learning rate | 0.07591 | 0.04 |
| Leaves | 48 | 71 |
| Minimum leaf rows | 914 | 201 |
| Feature fraction | 0.815 | 0.6683411362336381 |
| Bagging fraction | 0.702 | 0.8541457988583311 |
| L2 penalty | 28.7807 | 8.809972383953268 |
| Blend weight after percentile ranking | 75% | 25% |

Both families use multiclass loss, 31 bins, four native threads, deterministic
column-wise training, and seeds **333803, 333903, 334003**. The embedded native
parameter lists below are authoritative, including every seed and missing-value
setting. Quarter-period weights apply inclusively from **202003 through 202103**
and are normalized over all **4,547,505** training rows.

The final cell asserts SHA256
`756d4e512e29f47565e93db4412f1f6ee77c5f0ddf3e134f9e2b40cf2fce505a`.
Only that successful assertion establishes end-to-end reproduction.


In [ ]:
# Edit these two paths before Run All. No project/bucket dependency is embedded.
RAW_DATA <- Sys.getenv("HT5976_RAW", "../analistasr_competencia_2026.csv")
OUTPUT_DIR <- Sys.getenv("HT5976_OUTPUT", "./ht5974_raw_reproduction")
SCRATCH_DIR <- Sys.getenv("HT5976_SCRATCH", "./ht5974_raw_reproduction_scratch")
RUN_NOTEBOOK <- identical(Sys.getenv("HT5976_RUN", "TRUE"), "TRUE")
RUN_TRAINING <- identical(Sys.getenv("HT5976_TRAIN", "TRUE"), "TRUE")
WORKERS <- as.integer(Sys.getenv("HT5976_WORKERS", "3"))
Sys.setenv(OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", OMP_NUM_THREADS="4")
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(lightgbm))


## Frozen specification and reference fingerprints

The metadata below was exported read-only from the six original signed models.
Original paths are retained as provenance labels only; no code opens them.
Reference monthly fingerprints were independently calculated from signed original
frames, including IDs, training labels, and all 1,920 feature values in exact order.
They are assertions, not stored feature data.


In [ ]:
recipe <- jsonlite::fromJSON(r"---({
  "dataset_sha256": "a97d955b59917ea0df831264e5b50dc2c407580e26adb08d2cbc798c32b199aa",
  "final_csv_sha256": "756d4e512e29f47565e93db4412f1f6ee77c5f0ddf3e134f9e2b40cf2fce505a",
  "final_filename": "HT5974_A25_P384_08250.csv",
  "public_score": 462.414,
  "user_reported_final_score": 475.75,
  "submission": "58180681",
  "rows": 4877363,
  "customers": 186788,
  "training_rows": 4547505,
  "prediction_rows": 165076,
  "train_months": [
    201901,
    201902,
    201903,
    201904,
    201905,
    201906,
    201907,
    201908,
    201909,
    201910,
    201911,
    201912,
    202001,
    202002,
    202003,
    202004,
    202005,
    202006,
    202007,
    202008,
    202009,
    202010,
    202011,
    202012,
    202101,
    202102,
    202103,
    202104,
    202105,
    202106,
    202107
  ],
  "target_month": 202109,
  "seeds": [
    333803,
    333903,
    334003
  ],
  "raw_fields": [
    "active_quarter",
    "cliente_vip",
    "internet",
    "cliente_edad",
    "cliente_antiguedad",
    "mrentabilidad",
    "mrentabilidad_annual",
    "mcomisiones",
    "mactivos_margen",
    "mpasivos_margen",
    "cproductos",
    "tcuentas",
    "ccuenta_corriente",
    "mcuenta_corriente_adicional",
    "mcuenta_corriente",
    "ccaja_ahorro",
    "mcaja_ahorro",
    "mcaja_ahorro_adicional",
    "mcaja_ahorro_dolares",
    "cdescubierto_preacordado",
    "mcuentas_saldo",
    "ctarjeta_debito",
    "ctarjeta_debito_transacciones",
    "mautoservicio",
    "ctarjeta_visa",
    "ctarjeta_visa_transacciones",
    "mtarjeta_visa_consumo",
    "ctarjeta_master",
    "ctarjeta_master_transacciones",
    "mtarjeta_master_consumo",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_prendarios",
    "mprestamos_prendarios",
    "cprestamos_hipotecarios",
    "mprestamos_hipotecarios",
    "cplazo_fijo",
    "mplazo_fijo_dolares",
    "mplazo_fijo_pesos",
    "cinversion1",
    "minversion1_pesos",
    "minversion1_dolares",
    "cinversion2",
    "minversion2",
    "cseguro_vida",
    "cseguro_auto",
    "cseguro_vivienda",
    "cseguro_accidentes_personales",
    "ccaja_seguridad",
    "cpayroll_trx",
    "mpayroll",
    "mpayroll2",
    "cpayroll2_trx",
    "ccuenta_debitos_automaticos",
    "mcuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos",
    "mttarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos",
    "mttarjeta_master_debitos_automaticos",
    "cpagodeservicios",
    "mpagodeservicios",
    "cpagomiscuentas",
    "mpagomiscuentas",
    "ccajeros_propios_descuentos",
    "mcajeros_propios_descuentos",
    "ctarjeta_visa_descuentos",
    "mtarjeta_visa_descuentos",
    "ctarjeta_master_descuentos",
    "mtarjeta_master_descuentos",
    "ccomisiones_mantenimiento",
    "mcomisiones_mantenimiento",
    "ccomisiones_otras",
    "mcomisiones_otras",
    "cforex",
    "cforex_buy",
    "mforex_buy",
    "cforex_sell",
    "mforex_sell",
    "ctransferencias_recibidas",
    "mtransferencias_recibidas",
    "ctransferencias_emitidas",
    "mtransferencias_emitidas",
    "cextraccion_autoservicio",
    "mextraccion_autoservicio",
    "ccheques_depositados",
    "mcheques_depositados",
    "ccheques_emitidos",
    "mcheques_emitidos",
    "ccheques_depositados_rechazados",
    "mcheques_depositados_rechazados",
    "ccheques_emitidos_rechazados",
    "mcheques_emitidos_rechazados",
    "tcallcenter",
    "ccallcenter_transacciones",
    "thomebanking",
    "chomebanking_transacciones",
    "ccajas_transacciones",
    "ccajas_consultas",
    "ccajas_depositos",
    "ccajas_extracciones",
    "ccajas_otras",
    "catm_trx",
    "matm",
    "catm_trx_other",
    "matm_other",
    "ctrx_quarter",
    "tmobile_app",
    "cmobile_app_trx",
    "Master_delinquency",
    "Master_status",
    "Master_mfinanciacion_limite",
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_msaldototal",
    "Master_msaldopesos",
    "Master_msaldodolares",
    "Master_mconsumospesos",
    "Master_mconsumosdolares",
    "Master_mlimitecompra",
    "Master_madelantopesos",
    "Master_madelantodolares",
    "Master_fultimo_cierre",
    "Master_mpagado",
    "Master_mpagospesos",
    "Master_mpagosdolares",
    "Master_fechaalta",
    "Master_mconsumototal",
    "Master_cconsumos",
    "Master_cadelantosefectivo",
    "Master_mpagominimo",
    "Visa_delinquency",
    "Visa_status",
    "Visa_mfinanciacion_limite",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_msaldototal",
    "Visa_msaldopesos",
    "Visa_msaldodolares",
    "Visa_mconsumospesos",
    "Visa_mconsumosdolares",
    "Visa_mlimitecompra",
    "Visa_madelantopesos",
    "Visa_madelantodolares",
    "Visa_fultimo_cierre",
    "Visa_mpagado",
    "Visa_mpagospesos",
    "Visa_mpagosdolares",
    "Visa_fechaalta",
    "Visa_mconsumototal",
    "Visa_cconsumos",
    "Visa_cadelantosefectivo",
    "Visa_mpagominimo"
  ],
  "history_fields": [
    "mcuentas_saldo",
    "mcuenta_corriente",
    "mcaja_ahorro",
    "mpayroll",
    "mtarjeta_visa_consumo",
    "mtarjeta_master_consumo",
    "ctrx_quarter",
    "ctarjeta_debito_transacciones",
    "mtransferencias_recibidas",
    "mtransferencias_emitidas",
    "mprestamos_personales",
    "mrentabilidad"
  ],
  "base_columns": [
    "active_quarter",
    "cliente_vip",
    "internet",
    "cliente_edad",
    "cliente_antiguedad",
    "mrentabilidad",
    "mrentabilidad_annual",
    "mcomisiones",
    "mactivos_margen",
    "mpasivos_margen",
    "cproductos",
    "tcuentas",
    "ccuenta_corriente",
    "mcuenta_corriente_adicional",
    "mcuenta_corriente",
    "ccaja_ahorro",
    "mcaja_ahorro",
    "mcaja_ahorro_adicional",
    "mcaja_ahorro_dolares",
    "cdescubierto_preacordado",
    "mcuentas_saldo",
    "ctarjeta_debito",
    "ctarjeta_debito_transacciones",
    "mautoservicio",
    "ctarjeta_visa",
    "ctarjeta_visa_transacciones",
    "mtarjeta_visa_consumo",
    "ctarjeta_master",
    "ctarjeta_master_transacciones",
    "mtarjeta_master_consumo",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_prendarios",
    "mprestamos_prendarios",
    "cprestamos_hipotecarios",
    "mprestamos_hipotecarios",
    "cplazo_fijo",
    "mplazo_fijo_dolares",
    "mplazo_fijo_pesos",
    "cinversion1",
    "minversion1_pesos",
    "minversion1_dolares",
    "cinversion2",
    "minversion2",
    "cseguro_vida",
    "cseguro_auto",
    "cseguro_vivienda",
    "cseguro_accidentes_personales",
    "ccaja_seguridad",
    "cpayroll_trx",
    "mpayroll",
    "mpayroll2",
    "cpayroll2_trx",
    "ccuenta_debitos_automaticos",
    "mcuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos",
    "mttarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos",
    "mttarjeta_master_debitos_automaticos",
    "cpagodeservicios",
    "mpagodeservicios",
    "cpagomiscuentas",
    "mpagomiscuentas",
    "ccajeros_propios_descuentos",
    "mcajeros_propios_descuentos",
    "ctarjeta_visa_descuentos",
    "mtarjeta_visa_descuentos",
    "ctarjeta_master_descuentos",
    "mtarjeta_master_descuentos",
    "ccomisiones_mantenimiento",
    "mcomisiones_mantenimiento",
    "ccomisiones_otras",
    "mcomisiones_otras",
    "cforex",
    "cforex_buy",
    "mforex_buy",
    "cforex_sell",
    "mforex_sell",
    "ctransferencias_recibidas",
    "mtransferencias_recibidas",
    "ctransferencias_emitidas",
    "mtransferencias_emitidas",
    "cextraccion_autoservicio",
    "mextraccion_autoservicio",
    "ccheques_depositados",
    "mcheques_depositados",
    "ccheques_emitidos",
    "mcheques_emitidos",
    "ccheques_depositados_rechazados",
    "mcheques_depositados_rechazados",
    "ccheques_emitidos_rechazados",
    "mcheques_emitidos_rechazados",
    "tcallcenter",
    "ccallcenter_transacciones",
    "thomebanking",
    "chomebanking_transacciones",
    "ccajas_transacciones",
    "ccajas_consultas",
    "ccajas_depositos",
    "ccajas_extracciones",
    "ccajas_otras",
    "catm_trx",
    "matm",
    "catm_trx_other",
    "matm_other",
    "ctrx_quarter",
    "tmobile_app",
    "cmobile_app_trx",
    "Master_delinquency",
    "Master_status",
    "Master_mfinanciacion_limite",
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_msaldototal",
    "Master_msaldopesos",
    "Master_msaldodolares",
    "Master_mconsumospesos",
    "Master_mconsumosdolares",
    "Master_mlimitecompra",
    "Master_madelantopesos",
    "Master_madelantodolares",
    "Master_fultimo_cierre",
    "Master_mpagado",
    "Master_mpagospesos",
    "Master_mpagosdolares",
    "Master_fechaalta",
    "Master_mconsumototal",
    "Master_cconsumos",
    "Master_cadelantosefectivo",
    "Master_mpagominimo",
    "Visa_delinquency",
    "Visa_status",
    "Visa_mfinanciacion_limite",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_msaldototal",
    "Visa_msaldopesos",
    "Visa_msaldodolares",
    "Visa_mconsumospesos",
    "Visa_mconsumosdolares",
    "Visa_mlimitecompra",
    "Visa_madelantopesos",
    "Visa_madelantodolares",
    "Visa_fultimo_cierre",
    "Visa_mpagado",
    "Visa_mpagospesos",
    "Visa_mpagosdolares",
    "Visa_fechaalta",
    "Visa_mconsumototal",
    "Visa_cconsumos",
    "Visa_cadelantosefectivo",
    "Visa_mpagominimo",
    "history_month1_present",
    "history_month2_present",
    "history_month3_present",
    "history_month4_present",
    "history_month5_present",
    "history_month6_present",
    "mcuentas_saldo_lag1",
    "mcuentas_saldo_lag2",
    "mcuentas_saldo_lag3",
    "mcuentas_saldo_lag6",
    "mcuentas_saldo_delta1",
    "mcuentas_saldo_delta3",
    "mcuentas_saldo_prior3_mean",
    "mcuentas_saldo_prior3_observed",
    "mcuentas_saldo_prior6_mean",
    "mcuentas_saldo_prior6_observed",
    "mcuenta_corriente_lag1",
    "mcuenta_corriente_lag2",
    "mcuenta_corriente_lag3",
    "mcuenta_corriente_lag6",
    "mcuenta_corriente_delta1",
    "mcuenta_corriente_delta3",
    "mcuenta_corriente_prior3_mean",
    "mcuenta_corriente_prior3_observed",
    "mcuenta_corriente_prior6_mean",
    "mcuenta_corriente_prior6_observed",
    "mcaja_ahorro_lag1",
    "mcaja_ahorro_lag2",
    "mcaja_ahorro_lag3",
    "mcaja_ahorro_lag6",
    "mcaja_ahorro_delta1",
    "mcaja_ahorro_delta3",
    "mcaja_ahorro_prior3_mean",
    "mcaja_ahorro_prior3_observed",
    "mcaja_ahorro_prior6_mean",
    "mcaja_ahorro_prior6_observed",
    "mpayroll_lag1",
    "mpayroll_lag2",
    "mpayroll_lag3",
    "mpayroll_lag6",
    "mpayroll_delta1",
    "mpayroll_delta3",
    "mpayroll_prior3_mean",
    "mpayroll_prior3_observed",
    "mpayroll_prior6_mean",
    "mpayroll_prior6_observed",
    "mtarjeta_visa_consumo_lag1",
    "mtarjeta_visa_consumo_lag2",
    "mtarjeta_visa_consumo_lag3",
    "mtarjeta_visa_consumo_lag6",
    "mtarjeta_visa_consumo_delta1",
    "mtarjeta_visa_consumo_delta3",
    "mtarjeta_visa_consumo_prior3_mean",
    "mtarjeta_visa_consumo_prior3_observed",
    "mtarjeta_visa_consumo_prior6_mean",
    "mtarjeta_visa_consumo_prior6_observed",
    "mtarjeta_master_consumo_lag1",
    "mtarjeta_master_consumo_lag2",
    "mtarjeta_master_consumo_lag3",
    "mtarjeta_master_consumo_lag6",
    "mtarjeta_master_consumo_delta1",
    "mtarjeta_master_consumo_delta3",
    "mtarjeta_master_consumo_prior3_mean",
    "mtarjeta_master_consumo_prior3_observed",
    "mtarjeta_master_consumo_prior6_mean",
    "mtarjeta_master_consumo_prior6_observed",
    "ctrx_quarter_lag1",
    "ctrx_quarter_lag2",
    "ctrx_quarter_lag3",
    "ctrx_quarter_lag6",
    "ctrx_quarter_delta1",
    "ctrx_quarter_delta3",
    "ctrx_quarter_prior3_mean",
    "ctrx_quarter_prior3_observed",
    "ctrx_quarter_prior6_mean",
    "ctrx_quarter_prior6_observed",
    "ctarjeta_debito_transacciones_lag1",
    "ctarjeta_debito_transacciones_lag2",
    "ctarjeta_debito_transacciones_lag3",
    "ctarjeta_debito_transacciones_lag6",
    "ctarjeta_debito_transacciones_delta1",
    "ctarjeta_debito_transacciones_delta3",
    "ctarjeta_debito_transacciones_prior3_mean",
    "ctarjeta_debito_transacciones_prior3_observed",
    "ctarjeta_debito_transacciones_prior6_mean",
    "ctarjeta_debito_transacciones_prior6_observed",
    "mtransferencias_recibidas_lag1",
    "mtransferencias_recibidas_lag2",
    "mtransferencias_recibidas_lag3",
    "mtransferencias_recibidas_lag6",
    "mtransferencias_recibidas_delta1",
    "mtransferencias_recibidas_delta3",
    "mtransferencias_recibidas_prior3_mean",
    "mtransferencias_recibidas_prior3_observed",
    "mtransferencias_recibidas_prior6_mean",
    "mtransferencias_recibidas_prior6_observed",
    "mtransferencias_emitidas_lag1",
    "mtransferencias_emitidas_lag2",
    "mtransferencias_emitidas_lag3",
    "mtransferencias_emitidas_lag6",
    "mtransferencias_emitidas_delta1",
    "mtransferencias_emitidas_delta3",
    "mtransferencias_emitidas_prior3_mean",
    "mtransferencias_emitidas_prior3_observed",
    "mtransferencias_emitidas_prior6_mean",
    "mtransferencias_emitidas_prior6_observed",
    "mprestamos_personales_lag1",
    "mprestamos_personales_lag2",
    "mprestamos_personales_lag3",
    "mprestamos_personales_lag6",
    "mprestamos_personales_delta1",
    "mprestamos_personales_delta3",
    "mprestamos_personales_prior3_mean",
    "mprestamos_personales_prior3_observed",
    "mprestamos_personales_prior6_mean",
    "mprestamos_personales_prior6_observed",
    "mrentabilidad_lag1",
    "mrentabilidad_lag2",
    "mrentabilidad_lag3",
    "mrentabilidad_lag6",
    "mrentabilidad_delta1",
    "mrentabilidad_delta3",
    "mrentabilidad_prior3_mean",
    "mrentabilidad_prior3_observed",
    "mrentabilidad_prior6_mean",
    "mrentabilidad_prior6_observed",
    "active_quarter_lag1",
    "active_quarter_delta1",
    "active_quarter_lag2",
    "active_quarter_delta2",
    "active_quarter_ols6",
    "active_quarter_sma3",
    "active_quarter_sma6",
    "active_quarter_min6",
    "active_quarter_max6",
    "active_quarter_ema3",
    "active_quarter_ema6",
    "cliente_vip_lag1",
    "cliente_vip_delta1",
    "cliente_vip_lag2",
    "cliente_vip_delta2",
    "cliente_vip_ols6",
    "cliente_vip_sma3",
    "cliente_vip_sma6",
    "cliente_vip_min6",
    "cliente_vip_max6",
    "cliente_vip_ema3",
    "cliente_vip_ema6",
    "internet_lag1",
    "internet_delta1",
    "internet_lag2",
    "internet_delta2",
    "internet_ols6",
    "internet_sma3",
    "internet_sma6",
    "internet_min6",
    "internet_max6",
    "internet_ema3",
    "internet_ema6",
    "cliente_edad_lag1",
    "cliente_edad_delta1",
    "cliente_edad_lag2",
    "cliente_edad_delta2",
    "cliente_edad_ols6",
    "cliente_edad_sma3",
    "cliente_edad_sma6",
    "cliente_edad_min6",
    "cliente_edad_max6",
    "cliente_edad_ema3",
    "cliente_edad_ema6",
    "cliente_antiguedad_lag1",
    "cliente_antiguedad_delta1",
    "cliente_antiguedad_lag2",
    "cliente_antiguedad_delta2",
    "cliente_antiguedad_ols6",
    "cliente_antiguedad_sma3",
    "cliente_antiguedad_sma6",
    "cliente_antiguedad_min6",
    "cliente_antiguedad_max6",
    "cliente_antiguedad_ema3",
    "cliente_antiguedad_ema6",
    "mrentabilidad_delta2",
    "mrentabilidad_ols6",
    "mrentabilidad_sma3",
    "mrentabilidad_sma6",
    "mrentabilidad_min6",
    "mrentabilidad_max6",
    "mrentabilidad_ema3",
    "mrentabilidad_ema6",
    "mrentabilidad_annual_lag1",
    "mrentabilidad_annual_delta1",
    "mrentabilidad_annual_lag2",
    "mrentabilidad_annual_delta2",
    "mrentabilidad_annual_ols6",
    "mrentabilidad_annual_sma3",
    "mrentabilidad_annual_sma6",
    "mrentabilidad_annual_min6",
    "mrentabilidad_annual_max6",
    "mrentabilidad_annual_ema3",
    "mrentabilidad_annual_ema6",
    "mcomisiones_lag1",
    "mcomisiones_delta1",
    "mcomisiones_lag2",
    "mcomisiones_delta2",
    "mcomisiones_ols6",
    "mcomisiones_sma3",
    "mcomisiones_sma6",
    "mcomisiones_min6",
    "mcomisiones_max6",
    "mcomisiones_ema3",
    "mcomisiones_ema6",
    "mactivos_margen_lag1",
    "mactivos_margen_delta1",
    "mactivos_margen_lag2",
    "mactivos_margen_delta2",
    "mactivos_margen_ols6",
    "mactivos_margen_sma3",
    "mactivos_margen_sma6",
    "mactivos_margen_min6",
    "mactivos_margen_max6",
    "mactivos_margen_ema3",
    "mactivos_margen_ema6",
    "mpasivos_margen_lag1",
    "mpasivos_margen_delta1",
    "mpasivos_margen_lag2",
    "mpasivos_margen_delta2",
    "mpasivos_margen_ols6",
    "mpasivos_margen_sma3",
    "mpasivos_margen_sma6",
    "mpasivos_margen_min6",
    "mpasivos_margen_max6",
    "mpasivos_margen_ema3",
    "mpasivos_margen_ema6",
    "cproductos_lag1",
    "cproductos_delta1",
    "cproductos_lag2",
    "cproductos_delta2",
    "cproductos_ols6",
    "cproductos_sma3",
    "cproductos_sma6",
    "cproductos_min6",
    "cproductos_max6",
    "cproductos_ema3",
    "cproductos_ema6",
    "tcuentas_lag1",
    "tcuentas_delta1",
    "tcuentas_lag2",
    "tcuentas_delta2",
    "tcuentas_ols6",
    "tcuentas_sma3",
    "tcuentas_sma6",
    "tcuentas_min6",
    "tcuentas_max6",
    "tcuentas_ema3",
    "tcuentas_ema6",
    "ccuenta_corriente_lag1",
    "ccuenta_corriente_delta1",
    "ccuenta_corriente_lag2",
    "ccuenta_corriente_delta2",
    "ccuenta_corriente_ols6",
    "ccuenta_corriente_sma3",
    "ccuenta_corriente_sma6",
    "ccuenta_corriente_min6",
    "ccuenta_corriente_max6",
    "ccuenta_corriente_ema3",
    "ccuenta_corriente_ema6",
    "mcuenta_corriente_adicional_lag1",
    "mcuenta_corriente_adicional_delta1",
    "mcuenta_corriente_adicional_lag2",
    "mcuenta_corriente_adicional_delta2",
    "mcuenta_corriente_adicional_ols6",
    "mcuenta_corriente_adicional_sma3",
    "mcuenta_corriente_adicional_sma6",
    "mcuenta_corriente_adicional_min6",
    "mcuenta_corriente_adicional_max6",
    "mcuenta_corriente_adicional_ema3",
    "mcuenta_corriente_adicional_ema6",
    "mcuenta_corriente_delta2",
    "mcuenta_corriente_ols6",
    "mcuenta_corriente_sma3",
    "mcuenta_corriente_sma6",
    "mcuenta_corriente_min6",
    "mcuenta_corriente_max6",
    "mcuenta_corriente_ema3",
    "mcuenta_corriente_ema6",
    "ccaja_ahorro_lag1",
    "ccaja_ahorro_delta1",
    "ccaja_ahorro_lag2",
    "ccaja_ahorro_delta2",
    "ccaja_ahorro_ols6",
    "ccaja_ahorro_sma3",
    "ccaja_ahorro_sma6",
    "ccaja_ahorro_min6",
    "ccaja_ahorro_max6",
    "ccaja_ahorro_ema3",
    "ccaja_ahorro_ema6",
    "mcaja_ahorro_delta2",
    "mcaja_ahorro_ols6",
    "mcaja_ahorro_sma3",
    "mcaja_ahorro_sma6",
    "mcaja_ahorro_min6",
    "mcaja_ahorro_max6",
    "mcaja_ahorro_ema3",
    "mcaja_ahorro_ema6",
    "mcaja_ahorro_adicional_lag1",
    "mcaja_ahorro_adicional_delta1",
    "mcaja_ahorro_adicional_lag2",
    "mcaja_ahorro_adicional_delta2",
    "mcaja_ahorro_adicional_ols6",
    "mcaja_ahorro_adicional_sma3",
    "mcaja_ahorro_adicional_sma6",
    "mcaja_ahorro_adicional_min6",
    "mcaja_ahorro_adicional_max6",
    "mcaja_ahorro_adicional_ema3",
    "mcaja_ahorro_adicional_ema6",
    "mcaja_ahorro_dolares_lag1",
    "mcaja_ahorro_dolares_delta1",
    "mcaja_ahorro_dolares_lag2",
    "mcaja_ahorro_dolares_delta2",
    "mcaja_ahorro_dolares_ols6",
    "mcaja_ahorro_dolares_sma3",
    "mcaja_ahorro_dolares_sma6",
    "mcaja_ahorro_dolares_min6",
    "mcaja_ahorro_dolares_max6",
    "mcaja_ahorro_dolares_ema3",
    "mcaja_ahorro_dolares_ema6",
    "cdescubierto_preacordado_lag1",
    "cdescubierto_preacordado_delta1",
    "cdescubierto_preacordado_lag2",
    "cdescubierto_preacordado_delta2",
    "cdescubierto_preacordado_ols6",
    "cdescubierto_preacordado_sma3",
    "cdescubierto_preacordado_sma6",
    "cdescubierto_preacordado_min6",
    "cdescubierto_preacordado_max6",
    "cdescubierto_preacordado_ema3",
    "cdescubierto_preacordado_ema6",
    "mcuentas_saldo_delta2",
    "mcuentas_saldo_ols6",
    "mcuentas_saldo_sma3",
    "mcuentas_saldo_sma6",
    "mcuentas_saldo_min6",
    "mcuentas_saldo_max6",
    "mcuentas_saldo_ema3",
    "mcuentas_saldo_ema6",
    "ctarjeta_debito_lag1",
    "ctarjeta_debito_delta1",
    "ctarjeta_debito_lag2",
    "ctarjeta_debito_delta2",
    "ctarjeta_debito_ols6",
    "ctarjeta_debito_sma3",
    "ctarjeta_debito_sma6",
    "ctarjeta_debito_min6",
    "ctarjeta_debito_max6",
    "ctarjeta_debito_ema3",
    "ctarjeta_debito_ema6",
    "ctarjeta_debito_transacciones_delta2",
    "ctarjeta_debito_transacciones_ols6",
    "ctarjeta_debito_transacciones_sma3",
    "ctarjeta_debito_transacciones_sma6",
    "ctarjeta_debito_transacciones_min6",
    "ctarjeta_debito_transacciones_max6",
    "ctarjeta_debito_transacciones_ema3",
    "ctarjeta_debito_transacciones_ema6",
    "mautoservicio_lag1",
    "mautoservicio_delta1",
    "mautoservicio_lag2",
    "mautoservicio_delta2",
    "mautoservicio_ols6",
    "mautoservicio_sma3",
    "mautoservicio_sma6",
    "mautoservicio_min6",
    "mautoservicio_max6",
    "mautoservicio_ema3",
    "mautoservicio_ema6",
    "ctarjeta_visa_lag1",
    "ctarjeta_visa_delta1",
    "ctarjeta_visa_lag2",
    "ctarjeta_visa_delta2",
    "ctarjeta_visa_ols6",
    "ctarjeta_visa_sma3",
    "ctarjeta_visa_sma6",
    "ctarjeta_visa_min6",
    "ctarjeta_visa_max6",
    "ctarjeta_visa_ema3",
    "ctarjeta_visa_ema6",
    "ctarjeta_visa_transacciones_lag1",
    "ctarjeta_visa_transacciones_delta1",
    "ctarjeta_visa_transacciones_lag2",
    "ctarjeta_visa_transacciones_delta2",
    "ctarjeta_visa_transacciones_ols6",
    "ctarjeta_visa_transacciones_sma3",
    "ctarjeta_visa_transacciones_sma6",
    "ctarjeta_visa_transacciones_min6",
    "ctarjeta_visa_transacciones_max6",
    "ctarjeta_visa_transacciones_ema3",
    "ctarjeta_visa_transacciones_ema6",
    "mtarjeta_visa_consumo_delta2",
    "mtarjeta_visa_consumo_ols6",
    "mtarjeta_visa_consumo_sma3",
    "mtarjeta_visa_consumo_sma6",
    "mtarjeta_visa_consumo_min6",
    "mtarjeta_visa_consumo_max6",
    "mtarjeta_visa_consumo_ema3",
    "mtarjeta_visa_consumo_ema6",
    "ctarjeta_master_lag1",
    "ctarjeta_master_delta1",
    "ctarjeta_master_lag2",
    "ctarjeta_master_delta2",
    "ctarjeta_master_ols6",
    "ctarjeta_master_sma3",
    "ctarjeta_master_sma6",
    "ctarjeta_master_min6",
    "ctarjeta_master_max6",
    "ctarjeta_master_ema3",
    "ctarjeta_master_ema6",
    "ctarjeta_master_transacciones_lag1",
    "ctarjeta_master_transacciones_delta1",
    "ctarjeta_master_transacciones_lag2",
    "ctarjeta_master_transacciones_delta2",
    "ctarjeta_master_transacciones_ols6",
    "ctarjeta_master_transacciones_sma3",
    "ctarjeta_master_transacciones_sma6",
    "ctarjeta_master_transacciones_min6",
    "ctarjeta_master_transacciones_max6",
    "ctarjeta_master_transacciones_ema3",
    "ctarjeta_master_transacciones_ema6",
    "mtarjeta_master_consumo_delta2",
    "mtarjeta_master_consumo_ols6",
    "mtarjeta_master_consumo_sma3",
    "mtarjeta_master_consumo_sma6",
    "mtarjeta_master_consumo_min6",
    "mtarjeta_master_consumo_max6",
    "mtarjeta_master_consumo_ema3",
    "mtarjeta_master_consumo_ema6",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "cprestamos_personales_lag2",
    "cprestamos_personales_delta2",
    "cprestamos_personales_ols6",
    "cprestamos_personales_sma3",
    "cprestamos_personales_sma6",
    "cprestamos_personales_min6",
    "cprestamos_personales_max6",
    "cprestamos_personales_ema3",
    "cprestamos_personales_ema6",
    "mprestamos_personales_delta2",
    "mprestamos_personales_ols6",
    "mprestamos_personales_sma3",
    "mprestamos_personales_sma6",
    "mprestamos_personales_min6",
    "mprestamos_personales_max6",
    "mprestamos_personales_ema3",
    "mprestamos_personales_ema6",
    "cprestamos_prendarios_lag1",
    "cprestamos_prendarios_delta1",
    "cprestamos_prendarios_lag2",
    "cprestamos_prendarios_delta2",
    "cprestamos_prendarios_ols6",
    "cprestamos_prendarios_sma3",
    "cprestamos_prendarios_sma6",
    "cprestamos_prendarios_min6",
    "cprestamos_prendarios_max6",
    "cprestamos_prendarios_ema3",
    "cprestamos_prendarios_ema6",
    "mprestamos_prendarios_lag1",
    "mprestamos_prendarios_delta1",
    "mprestamos_prendarios_lag2",
    "mprestamos_prendarios_delta2",
    "mprestamos_prendarios_ols6",
    "mprestamos_prendarios_sma3",
    "mprestamos_prendarios_sma6",
    "mprestamos_prendarios_min6",
    "mprestamos_prendarios_max6",
    "mprestamos_prendarios_ema3",
    "mprestamos_prendarios_ema6",
    "cprestamos_hipotecarios_lag1",
    "cprestamos_hipotecarios_delta1",
    "cprestamos_hipotecarios_lag2",
    "cprestamos_hipotecarios_delta2",
    "cprestamos_hipotecarios_ols6",
    "cprestamos_hipotecarios_sma3",
    "cprestamos_hipotecarios_sma6",
    "cprestamos_hipotecarios_min6",
    "cprestamos_hipotecarios_max6",
    "cprestamos_hipotecarios_ema3",
    "cprestamos_hipotecarios_ema6",
    "mprestamos_hipotecarios_lag1",
    "mprestamos_hipotecarios_delta1",
    "mprestamos_hipotecarios_lag2",
    "mprestamos_hipotecarios_delta2",
    "mprestamos_hipotecarios_ols6",
    "mprestamos_hipotecarios_sma3",
    "mprestamos_hipotecarios_sma6",
    "mprestamos_hipotecarios_min6",
    "mprestamos_hipotecarios_max6",
    "mprestamos_hipotecarios_ema3",
    "mprestamos_hipotecarios_ema6",
    "cplazo_fijo_lag1",
    "cplazo_fijo_delta1",
    "cplazo_fijo_lag2",
    "cplazo_fijo_delta2",
    "cplazo_fijo_ols6",
    "cplazo_fijo_sma3",
    "cplazo_fijo_sma6",
    "cplazo_fijo_min6",
    "cplazo_fijo_max6",
    "cplazo_fijo_ema3",
    "cplazo_fijo_ema6",
    "mplazo_fijo_dolares_lag1",
    "mplazo_fijo_dolares_delta1",
    "mplazo_fijo_dolares_lag2",
    "mplazo_fijo_dolares_delta2",
    "mplazo_fijo_dolares_ols6",
    "mplazo_fijo_dolares_sma3",
    "mplazo_fijo_dolares_sma6",
    "mplazo_fijo_dolares_min6",
    "mplazo_fijo_dolares_max6",
    "mplazo_fijo_dolares_ema3",
    "mplazo_fijo_dolares_ema6",
    "mplazo_fijo_pesos_lag1",
    "mplazo_fijo_pesos_delta1",
    "mplazo_fijo_pesos_lag2",
    "mplazo_fijo_pesos_delta2",
    "mplazo_fijo_pesos_ols6",
    "mplazo_fijo_pesos_sma3",
    "mplazo_fijo_pesos_sma6",
    "mplazo_fijo_pesos_min6",
    "mplazo_fijo_pesos_max6",
    "mplazo_fijo_pesos_ema3",
    "mplazo_fijo_pesos_ema6",
    "cinversion1_lag1",
    "cinversion1_delta1",
    "cinversion1_lag2",
    "cinversion1_delta2",
    "cinversion1_ols6",
    "cinversion1_sma3",
    "cinversion1_sma6",
    "cinversion1_min6",
    "cinversion1_max6",
    "cinversion1_ema3",
    "cinversion1_ema6",
    "minversion1_pesos_lag1",
    "minversion1_pesos_delta1",
    "minversion1_pesos_lag2",
    "minversion1_pesos_delta2",
    "minversion1_pesos_ols6",
    "minversion1_pesos_sma3",
    "minversion1_pesos_sma6",
    "minversion1_pesos_min6",
    "minversion1_pesos_max6",
    "minversion1_pesos_ema3",
    "minversion1_pesos_ema6",
    "minversion1_dolares_lag1",
    "minversion1_dolares_delta1",
    "minversion1_dolares_lag2",
    "minversion1_dolares_delta2",
    "minversion1_dolares_ols6",
    "minversion1_dolares_sma3",
    "minversion1_dolares_sma6",
    "minversion1_dolares_min6",
    "minversion1_dolares_max6",
    "minversion1_dolares_ema3",
    "minversion1_dolares_ema6",
    "cinversion2_lag1",
    "cinversion2_delta1",
    "cinversion2_lag2",
    "cinversion2_delta2",
    "cinversion2_ols6",
    "cinversion2_sma3",
    "cinversion2_sma6",
    "cinversion2_min6",
    "cinversion2_max6",
    "cinversion2_ema3",
    "cinversion2_ema6",
    "minversion2_lag1",
    "minversion2_delta1",
    "minversion2_lag2",
    "minversion2_delta2",
    "minversion2_ols6",
    "minversion2_sma3",
    "minversion2_sma6",
    "minversion2_min6",
    "minversion2_max6",
    "minversion2_ema3",
    "minversion2_ema6",
    "cseguro_vida_lag1",
    "cseguro_vida_delta1",
    "cseguro_vida_lag2",
    "cseguro_vida_delta2",
    "cseguro_vida_ols6",
    "cseguro_vida_sma3",
    "cseguro_vida_sma6",
    "cseguro_vida_min6",
    "cseguro_vida_max6",
    "cseguro_vida_ema3",
    "cseguro_vida_ema6",
    "cseguro_auto_lag1",
    "cseguro_auto_delta1",
    "cseguro_auto_lag2",
    "cseguro_auto_delta2",
    "cseguro_auto_ols6",
    "cseguro_auto_sma3",
    "cseguro_auto_sma6",
    "cseguro_auto_min6",
    "cseguro_auto_max6",
    "cseguro_auto_ema3",
    "cseguro_auto_ema6",
    "cseguro_vivienda_lag1",
    "cseguro_vivienda_delta1",
    "cseguro_vivienda_lag2",
    "cseguro_vivienda_delta2",
    "cseguro_vivienda_ols6",
    "cseguro_vivienda_sma3",
    "cseguro_vivienda_sma6",
    "cseguro_vivienda_min6",
    "cseguro_vivienda_max6",
    "cseguro_vivienda_ema3",
    "cseguro_vivienda_ema6",
    "cseguro_accidentes_personales_lag1",
    "cseguro_accidentes_personales_delta1",
    "cseguro_accidentes_personales_lag2",
    "cseguro_accidentes_personales_delta2",
    "cseguro_accidentes_personales_ols6",
    "cseguro_accidentes_personales_sma3",
    "cseguro_accidentes_personales_sma6",
    "cseguro_accidentes_personales_min6",
    "cseguro_accidentes_personales_max6",
    "cseguro_accidentes_personales_ema3",
    "cseguro_accidentes_personales_ema6",
    "ccaja_seguridad_lag1",
    "ccaja_seguridad_delta1",
    "ccaja_seguridad_lag2",
    "ccaja_seguridad_delta2",
    "ccaja_seguridad_ols6",
    "ccaja_seguridad_sma3",
    "ccaja_seguridad_sma6",
    "ccaja_seguridad_min6",
    "ccaja_seguridad_max6",
    "ccaja_seguridad_ema3",
    "ccaja_seguridad_ema6",
    "cpayroll_trx_lag1",
    "cpayroll_trx_delta1",
    "cpayroll_trx_lag2",
    "cpayroll_trx_delta2",
    "cpayroll_trx_ols6",
    "cpayroll_trx_sma3",
    "cpayroll_trx_sma6",
    "cpayroll_trx_min6",
    "cpayroll_trx_max6",
    "cpayroll_trx_ema3",
    "cpayroll_trx_ema6",
    "mpayroll_delta2",
    "mpayroll_ols6",
    "mpayroll_sma3",
    "mpayroll_sma6",
    "mpayroll_min6",
    "mpayroll_max6",
    "mpayroll_ema3",
    "mpayroll_ema6",
    "mpayroll2_lag1",
    "mpayroll2_delta1",
    "mpayroll2_lag2",
    "mpayroll2_delta2",
    "mpayroll2_ols6",
    "mpayroll2_sma3",
    "mpayroll2_sma6",
    "mpayroll2_min6",
    "mpayroll2_max6",
    "mpayroll2_ema3",
    "mpayroll2_ema6",
    "cpayroll2_trx_lag1",
    "cpayroll2_trx_delta1",
    "cpayroll2_trx_lag2",
    "cpayroll2_trx_delta2",
    "cpayroll2_trx_ols6",
    "cpayroll2_trx_sma3",
    "cpayroll2_trx_sma6",
    "cpayroll2_trx_min6",
    "cpayroll2_trx_max6",
    "cpayroll2_trx_ema3",
    "cpayroll2_trx_ema6",
    "ccuenta_debitos_automaticos_lag1",
    "ccuenta_debitos_automaticos_delta1",
    "ccuenta_debitos_automaticos_lag2",
    "ccuenta_debitos_automaticos_delta2",
    "ccuenta_debitos_automaticos_ols6",
    "ccuenta_debitos_automaticos_sma3",
    "ccuenta_debitos_automaticos_sma6",
    "ccuenta_debitos_automaticos_min6",
    "ccuenta_debitos_automaticos_max6",
    "ccuenta_debitos_automaticos_ema3",
    "ccuenta_debitos_automaticos_ema6",
    "mcuenta_debitos_automaticos_lag1",
    "mcuenta_debitos_automaticos_delta1",
    "mcuenta_debitos_automaticos_lag2",
    "mcuenta_debitos_automaticos_delta2",
    "mcuenta_debitos_automaticos_ols6",
    "mcuenta_debitos_automaticos_sma3",
    "mcuenta_debitos_automaticos_sma6",
    "mcuenta_debitos_automaticos_min6",
    "mcuenta_debitos_automaticos_max6",
    "mcuenta_debitos_automaticos_ema3",
    "mcuenta_debitos_automaticos_ema6",
    "ctarjeta_visa_debitos_automaticos_lag1",
    "ctarjeta_visa_debitos_automaticos_delta1",
    "ctarjeta_visa_debitos_automaticos_lag2",
    "ctarjeta_visa_debitos_automaticos_delta2",
    "ctarjeta_visa_debitos_automaticos_ols6",
    "ctarjeta_visa_debitos_automaticos_sma3",
    "ctarjeta_visa_debitos_automaticos_sma6",
    "ctarjeta_visa_debitos_automaticos_min6",
    "ctarjeta_visa_debitos_automaticos_max6",
    "ctarjeta_visa_debitos_automaticos_ema3",
    "ctarjeta_visa_debitos_automaticos_ema6",
    "mttarjeta_visa_debitos_automaticos_lag1",
    "mttarjeta_visa_debitos_automaticos_delta1",
    "mttarjeta_visa_debitos_automaticos_lag2",
    "mttarjeta_visa_debitos_automaticos_delta2",
    "mttarjeta_visa_debitos_automaticos_ols6",
    "mttarjeta_visa_debitos_automaticos_sma3",
    "mttarjeta_visa_debitos_automaticos_sma6",
    "mttarjeta_visa_debitos_automaticos_min6",
    "mttarjeta_visa_debitos_automaticos_max6",
    "mttarjeta_visa_debitos_automaticos_ema3",
    "mttarjeta_visa_debitos_automaticos_ema6",
    "ctarjeta_master_debitos_automaticos_lag1",
    "ctarjeta_master_debitos_automaticos_delta1",
    "ctarjeta_master_debitos_automaticos_lag2",
    "ctarjeta_master_debitos_automaticos_delta2",
    "ctarjeta_master_debitos_automaticos_ols6",
    "ctarjeta_master_debitos_automaticos_sma3",
    "ctarjeta_master_debitos_automaticos_sma6",
    "ctarjeta_master_debitos_automaticos_min6",
    "ctarjeta_master_debitos_automaticos_max6",
    "ctarjeta_master_debitos_automaticos_ema3",
    "ctarjeta_master_debitos_automaticos_ema6",
    "mttarjeta_master_debitos_automaticos_lag1",
    "mttarjeta_master_debitos_automaticos_delta1",
    "mttarjeta_master_debitos_automaticos_lag2",
    "mttarjeta_master_debitos_automaticos_delta2",
    "mttarjeta_master_debitos_automaticos_ols6",
    "mttarjeta_master_debitos_automaticos_sma3",
    "mttarjeta_master_debitos_automaticos_sma6",
    "mttarjeta_master_debitos_automaticos_min6",
    "mttarjeta_master_debitos_automaticos_max6",
    "mttarjeta_master_debitos_automaticos_ema3",
    "mttarjeta_master_debitos_automaticos_ema6",
    "cpagodeservicios_lag1",
    "cpagodeservicios_delta1",
    "cpagodeservicios_lag2",
    "cpagodeservicios_delta2",
    "cpagodeservicios_ols6",
    "cpagodeservicios_sma3",
    "cpagodeservicios_sma6",
    "cpagodeservicios_min6",
    "cpagodeservicios_max6",
    "cpagodeservicios_ema3",
    "cpagodeservicios_ema6",
    "mpagodeservicios_lag1",
    "mpagodeservicios_delta1",
    "mpagodeservicios_lag2",
    "mpagodeservicios_delta2",
    "mpagodeservicios_ols6",
    "mpagodeservicios_sma3",
    "mpagodeservicios_sma6",
    "mpagodeservicios_min6",
    "mpagodeservicios_max6",
    "mpagodeservicios_ema3",
    "mpagodeservicios_ema6",
    "cpagomiscuentas_lag1",
    "cpagomiscuentas_delta1",
    "cpagomiscuentas_lag2",
    "cpagomiscuentas_delta2",
    "cpagomiscuentas_ols6",
    "cpagomiscuentas_sma3",
    "cpagomiscuentas_sma6",
    "cpagomiscuentas_min6",
    "cpagomiscuentas_max6",
    "cpagomiscuentas_ema3",
    "cpagomiscuentas_ema6",
    "mpagomiscuentas_lag1",
    "mpagomiscuentas_delta1",
    "mpagomiscuentas_lag2",
    "mpagomiscuentas_delta2",
    "mpagomiscuentas_ols6",
    "mpagomiscuentas_sma3",
    "mpagomiscuentas_sma6",
    "mpagomiscuentas_min6",
    "mpagomiscuentas_max6",
    "mpagomiscuentas_ema3",
    "mpagomiscuentas_ema6",
    "ccajeros_propios_descuentos_lag1",
    "ccajeros_propios_descuentos_delta1",
    "ccajeros_propios_descuentos_lag2",
    "ccajeros_propios_descuentos_delta2",
    "ccajeros_propios_descuentos_ols6",
    "ccajeros_propios_descuentos_sma3",
    "ccajeros_propios_descuentos_sma6",
    "ccajeros_propios_descuentos_min6",
    "ccajeros_propios_descuentos_max6",
    "ccajeros_propios_descuentos_ema3",
    "ccajeros_propios_descuentos_ema6",
    "mcajeros_propios_descuentos_lag1",
    "mcajeros_propios_descuentos_delta1",
    "mcajeros_propios_descuentos_lag2",
    "mcajeros_propios_descuentos_delta2",
    "mcajeros_propios_descuentos_ols6",
    "mcajeros_propios_descuentos_sma3",
    "mcajeros_propios_descuentos_sma6",
    "mcajeros_propios_descuentos_min6",
    "mcajeros_propios_descuentos_max6",
    "mcajeros_propios_descuentos_ema3",
    "mcajeros_propios_descuentos_ema6",
    "ctarjeta_visa_descuentos_lag1",
    "ctarjeta_visa_descuentos_delta1",
    "ctarjeta_visa_descuentos_lag2",
    "ctarjeta_visa_descuentos_delta2",
    "ctarjeta_visa_descuentos_ols6",
    "ctarjeta_visa_descuentos_sma3",
    "ctarjeta_visa_descuentos_sma6",
    "ctarjeta_visa_descuentos_min6",
    "ctarjeta_visa_descuentos_max6",
    "ctarjeta_visa_descuentos_ema3",
    "ctarjeta_visa_descuentos_ema6",
    "mtarjeta_visa_descuentos_lag1",
    "mtarjeta_visa_descuentos_delta1",
    "mtarjeta_visa_descuentos_lag2",
    "mtarjeta_visa_descuentos_delta2",
    "mtarjeta_visa_descuentos_ols6",
    "mtarjeta_visa_descuentos_sma3",
    "mtarjeta_visa_descuentos_sma6",
    "mtarjeta_visa_descuentos_min6",
    "mtarjeta_visa_descuentos_max6",
    "mtarjeta_visa_descuentos_ema3",
    "mtarjeta_visa_descuentos_ema6",
    "ctarjeta_master_descuentos_lag1",
    "ctarjeta_master_descuentos_delta1",
    "ctarjeta_master_descuentos_lag2",
    "ctarjeta_master_descuentos_delta2",
    "ctarjeta_master_descuentos_ols6",
    "ctarjeta_master_descuentos_sma3",
    "ctarjeta_master_descuentos_sma6",
    "ctarjeta_master_descuentos_min6",
    "ctarjeta_master_descuentos_max6",
    "ctarjeta_master_descuentos_ema3",
    "ctarjeta_master_descuentos_ema6",
    "mtarjeta_master_descuentos_lag1",
    "mtarjeta_master_descuentos_delta1",
    "mtarjeta_master_descuentos_lag2",
    "mtarjeta_master_descuentos_delta2",
    "mtarjeta_master_descuentos_ols6",
    "mtarjeta_master_descuentos_sma3",
    "mtarjeta_master_descuentos_sma6",
    "mtarjeta_master_descuentos_min6",
    "mtarjeta_master_descuentos_max6",
    "mtarjeta_master_descuentos_ema3",
    "mtarjeta_master_descuentos_ema6",
    "ccomisiones_mantenimiento_lag1",
    "ccomisiones_mantenimiento_delta1",
    "ccomisiones_mantenimiento_lag2",
    "ccomisiones_mantenimiento_delta2",
    "ccomisiones_mantenimiento_ols6",
    "ccomisiones_mantenimiento_sma3",
    "ccomisiones_mantenimiento_sma6",
    "ccomisiones_mantenimiento_min6",
    "ccomisiones_mantenimiento_max6",
    "ccomisiones_mantenimiento_ema3",
    "ccomisiones_mantenimiento_ema6",
    "mcomisiones_mantenimiento_lag1",
    "mcomisiones_mantenimiento_delta1",
    "mcomisiones_mantenimiento_lag2",
    "mcomisiones_mantenimiento_delta2",
    "mcomisiones_mantenimiento_ols6",
    "mcomisiones_mantenimiento_sma3",
    "mcomisiones_mantenimiento_sma6",
    "mcomisiones_mantenimiento_min6",
    "mcomisiones_mantenimiento_max6",
    "mcomisiones_mantenimiento_ema3",
    "mcomisiones_mantenimiento_ema6",
    "ccomisiones_otras_lag1",
    "ccomisiones_otras_delta1",
    "ccomisiones_otras_lag2",
    "ccomisiones_otras_delta2",
    "ccomisiones_otras_ols6",
    "ccomisiones_otras_sma3",
    "ccomisiones_otras_sma6",
    "ccomisiones_otras_min6",
    "ccomisiones_otras_max6",
    "ccomisiones_otras_ema3",
    "ccomisiones_otras_ema6",
    "mcomisiones_otras_lag1",
    "mcomisiones_otras_delta1",
    "mcomisiones_otras_lag2",
    "mcomisiones_otras_delta2",
    "mcomisiones_otras_ols6",
    "mcomisiones_otras_sma3",
    "mcomisiones_otras_sma6",
    "mcomisiones_otras_min6",
    "mcomisiones_otras_max6",
    "mcomisiones_otras_ema3",
    "mcomisiones_otras_ema6",
    "cforex_lag1",
    "cforex_delta1",
    "cforex_lag2",
    "cforex_delta2",
    "cforex_ols6",
    "cforex_sma3",
    "cforex_sma6",
    "cforex_min6",
    "cforex_max6",
    "cforex_ema3",
    "cforex_ema6",
    "cforex_buy_lag1",
    "cforex_buy_delta1",
    "cforex_buy_lag2",
    "cforex_buy_delta2",
    "cforex_buy_ols6",
    "cforex_buy_sma3",
    "cforex_buy_sma6",
    "cforex_buy_min6",
    "cforex_buy_max6",
    "cforex_buy_ema3",
    "cforex_buy_ema6",
    "mforex_buy_lag1",
    "mforex_buy_delta1",
    "mforex_buy_lag2",
    "mforex_buy_delta2",
    "mforex_buy_ols6",
    "mforex_buy_sma3",
    "mforex_buy_sma6",
    "mforex_buy_min6",
    "mforex_buy_max6",
    "mforex_buy_ema3",
    "mforex_buy_ema6",
    "cforex_sell_lag1",
    "cforex_sell_delta1",
    "cforex_sell_lag2",
    "cforex_sell_delta2",
    "cforex_sell_ols6",
    "cforex_sell_sma3",
    "cforex_sell_sma6",
    "cforex_sell_min6",
    "cforex_sell_max6",
    "cforex_sell_ema3",
    "cforex_sell_ema6",
    "mforex_sell_lag1",
    "mforex_sell_delta1",
    "mforex_sell_lag2",
    "mforex_sell_delta2",
    "mforex_sell_ols6",
    "mforex_sell_sma3",
    "mforex_sell_sma6",
    "mforex_sell_min6",
    "mforex_sell_max6",
    "mforex_sell_ema3",
    "mforex_sell_ema6",
    "ctransferencias_recibidas_lag1",
    "ctransferencias_recibidas_delta1",
    "ctransferencias_recibidas_lag2",
    "ctransferencias_recibidas_delta2",
    "ctransferencias_recibidas_ols6",
    "ctransferencias_recibidas_sma3",
    "ctransferencias_recibidas_sma6",
    "ctransferencias_recibidas_min6",
    "ctransferencias_recibidas_max6",
    "ctransferencias_recibidas_ema3",
    "ctransferencias_recibidas_ema6",
    "mtransferencias_recibidas_delta2",
    "mtransferencias_recibidas_ols6",
    "mtransferencias_recibidas_sma3",
    "mtransferencias_recibidas_sma6",
    "mtransferencias_recibidas_min6",
    "mtransferencias_recibidas_max6",
    "mtransferencias_recibidas_ema3",
    "mtransferencias_recibidas_ema6",
    "ctransferencias_emitidas_lag1",
    "ctransferencias_emitidas_delta1",
    "ctransferencias_emitidas_lag2",
    "ctransferencias_emitidas_delta2",
    "ctransferencias_emitidas_ols6",
    "ctransferencias_emitidas_sma3",
    "ctransferencias_emitidas_sma6",
    "ctransferencias_emitidas_min6",
    "ctransferencias_emitidas_max6",
    "ctransferencias_emitidas_ema3",
    "ctransferencias_emitidas_ema6",
    "mtransferencias_emitidas_delta2",
    "mtransferencias_emitidas_ols6",
    "mtransferencias_emitidas_sma3",
    "mtransferencias_emitidas_sma6",
    "mtransferencias_emitidas_min6",
    "mtransferencias_emitidas_max6",
    "mtransferencias_emitidas_ema3",
    "mtransferencias_emitidas_ema6",
    "cextraccion_autoservicio_lag1",
    "cextraccion_autoservicio_delta1",
    "cextraccion_autoservicio_lag2",
    "cextraccion_autoservicio_delta2",
    "cextraccion_autoservicio_ols6",
    "cextraccion_autoservicio_sma3",
    "cextraccion_autoservicio_sma6",
    "cextraccion_autoservicio_min6",
    "cextraccion_autoservicio_max6",
    "cextraccion_autoservicio_ema3",
    "cextraccion_autoservicio_ema6",
    "mextraccion_autoservicio_lag1",
    "mextraccion_autoservicio_delta1",
    "mextraccion_autoservicio_lag2",
    "mextraccion_autoservicio_delta2",
    "mextraccion_autoservicio_ols6",
    "mextraccion_autoservicio_sma3",
    "mextraccion_autoservicio_sma6",
    "mextraccion_autoservicio_min6",
    "mextraccion_autoservicio_max6",
    "mextraccion_autoservicio_ema3",
    "mextraccion_autoservicio_ema6",
    "ccheques_depositados_lag1",
    "ccheques_depositados_delta1",
    "ccheques_depositados_lag2",
    "ccheques_depositados_delta2",
    "ccheques_depositados_ols6",
    "ccheques_depositados_sma3",
    "ccheques_depositados_sma6",
    "ccheques_depositados_min6",
    "ccheques_depositados_max6",
    "ccheques_depositados_ema3",
    "ccheques_depositados_ema6",
    "mcheques_depositados_lag1",
    "mcheques_depositados_delta1",
    "mcheques_depositados_lag2",
    "mcheques_depositados_delta2",
    "mcheques_depositados_ols6",
    "mcheques_depositados_sma3",
    "mcheques_depositados_sma6",
    "mcheques_depositados_min6",
    "mcheques_depositados_max6",
    "mcheques_depositados_ema3",
    "mcheques_depositados_ema6",
    "ccheques_emitidos_lag1",
    "ccheques_emitidos_delta1",
    "ccheques_emitidos_lag2",
    "ccheques_emitidos_delta2",
    "ccheques_emitidos_ols6",
    "ccheques_emitidos_sma3",
    "ccheques_emitidos_sma6",
    "ccheques_emitidos_min6",
    "ccheques_emitidos_max6",
    "ccheques_emitidos_ema3",
    "ccheques_emitidos_ema6",
    "mcheques_emitidos_lag1",
    "mcheques_emitidos_delta1",
    "mcheques_emitidos_lag2",
    "mcheques_emitidos_delta2",
    "mcheques_emitidos_ols6",
    "mcheques_emitidos_sma3",
    "mcheques_emitidos_sma6",
    "mcheques_emitidos_min6",
    "mcheques_emitidos_max6",
    "mcheques_emitidos_ema3",
    "mcheques_emitidos_ema6",
    "ccheques_depositados_rechazados_lag1",
    "ccheques_depositados_rechazados_delta1",
    "ccheques_depositados_rechazados_lag2",
    "ccheques_depositados_rechazados_delta2",
    "ccheques_depositados_rechazados_ols6",
    "ccheques_depositados_rechazados_sma3",
    "ccheques_depositados_rechazados_sma6",
    "ccheques_depositados_rechazados_min6",
    "ccheques_depositados_rechazados_max6",
    "ccheques_depositados_rechazados_ema3",
    "ccheques_depositados_rechazados_ema6",
    "mcheques_depositados_rechazados_lag1",
    "mcheques_depositados_rechazados_delta1",
    "mcheques_depositados_rechazados_lag2",
    "mcheques_depositados_rechazados_delta2",
    "mcheques_depositados_rechazados_ols6",
    "mcheques_depositados_rechazados_sma3",
    "mcheques_depositados_rechazados_sma6",
    "mcheques_depositados_rechazados_min6",
    "mcheques_depositados_rechazados_max6",
    "mcheques_depositados_rechazados_ema3",
    "mcheques_depositados_rechazados_ema6",
    "ccheques_emitidos_rechazados_lag1",
    "ccheques_emitidos_rechazados_delta1",
    "ccheques_emitidos_rechazados_lag2",
    "ccheques_emitidos_rechazados_delta2",
    "ccheques_emitidos_rechazados_ols6",
    "ccheques_emitidos_rechazados_sma3",
    "ccheques_emitidos_rechazados_sma6",
    "ccheques_emitidos_rechazados_min6",
    "ccheques_emitidos_rechazados_max6",
    "ccheques_emitidos_rechazados_ema3",
    "ccheques_emitidos_rechazados_ema6",
    "mcheques_emitidos_rechazados_lag1",
    "mcheques_emitidos_rechazados_delta1",
    "mcheques_emitidos_rechazados_lag2",
    "mcheques_emitidos_rechazados_delta2",
    "mcheques_emitidos_rechazados_ols6",
    "mcheques_emitidos_rechazados_sma3",
    "mcheques_emitidos_rechazados_sma6",
    "mcheques_emitidos_rechazados_min6",
    "mcheques_emitidos_rechazados_max6",
    "mcheques_emitidos_rechazados_ema3",
    "mcheques_emitidos_rechazados_ema6",
    "tcallcenter_lag1",
    "tcallcenter_delta1",
    "tcallcenter_lag2",
    "tcallcenter_delta2",
    "tcallcenter_ols6",
    "tcallcenter_sma3",
    "tcallcenter_sma6",
    "tcallcenter_min6",
    "tcallcenter_max6",
    "tcallcenter_ema3",
    "tcallcenter_ema6",
    "ccallcenter_transacciones_lag1",
    "ccallcenter_transacciones_delta1",
    "ccallcenter_transacciones_lag2",
    "ccallcenter_transacciones_delta2",
    "ccallcenter_transacciones_ols6",
    "ccallcenter_transacciones_sma3",
    "ccallcenter_transacciones_sma6",
    "ccallcenter_transacciones_min6",
    "ccallcenter_transacciones_max6",
    "ccallcenter_transacciones_ema3",
    "ccallcenter_transacciones_ema6",
    "thomebanking_lag1",
    "thomebanking_delta1",
    "thomebanking_lag2",
    "thomebanking_delta2",
    "thomebanking_ols6",
    "thomebanking_sma3",
    "thomebanking_sma6",
    "thomebanking_min6",
    "thomebanking_max6",
    "thomebanking_ema3",
    "thomebanking_ema6",
    "chomebanking_transacciones_lag1",
    "chomebanking_transacciones_delta1",
    "chomebanking_transacciones_lag2",
    "chomebanking_transacciones_delta2",
    "chomebanking_transacciones_ols6",
    "chomebanking_transacciones_sma3",
    "chomebanking_transacciones_sma6",
    "chomebanking_transacciones_min6",
    "chomebanking_transacciones_max6",
    "chomebanking_transacciones_ema3",
    "chomebanking_transacciones_ema6",
    "ccajas_transacciones_lag1",
    "ccajas_transacciones_delta1",
    "ccajas_transacciones_lag2",
    "ccajas_transacciones_delta2",
    "ccajas_transacciones_ols6",
    "ccajas_transacciones_sma3",
    "ccajas_transacciones_sma6",
    "ccajas_transacciones_min6",
    "ccajas_transacciones_max6",
    "ccajas_transacciones_ema3",
    "ccajas_transacciones_ema6",
    "ccajas_consultas_lag1",
    "ccajas_consultas_delta1",
    "ccajas_consultas_lag2",
    "ccajas_consultas_delta2",
    "ccajas_consultas_ols6",
    "ccajas_consultas_sma3",
    "ccajas_consultas_sma6",
    "ccajas_consultas_min6",
    "ccajas_consultas_max6",
    "ccajas_consultas_ema3",
    "ccajas_consultas_ema6",
    "ccajas_depositos_lag1",
    "ccajas_depositos_delta1",
    "ccajas_depositos_lag2",
    "ccajas_depositos_delta2",
    "ccajas_depositos_ols6",
    "ccajas_depositos_sma3",
    "ccajas_depositos_sma6",
    "ccajas_depositos_min6",
    "ccajas_depositos_max6",
    "ccajas_depositos_ema3",
    "ccajas_depositos_ema6",
    "ccajas_extracciones_lag1",
    "ccajas_extracciones_delta1",
    "ccajas_extracciones_lag2",
    "ccajas_extracciones_delta2",
    "ccajas_extracciones_ols6",
    "ccajas_extracciones_sma3",
    "ccajas_extracciones_sma6",
    "ccajas_extracciones_min6",
    "ccajas_extracciones_max6",
    "ccajas_extracciones_ema3",
    "ccajas_extracciones_ema6",
    "ccajas_otras_lag1",
    "ccajas_otras_delta1",
    "ccajas_otras_lag2",
    "ccajas_otras_delta2",
    "ccajas_otras_ols6",
    "ccajas_otras_sma3",
    "ccajas_otras_sma6",
    "ccajas_otras_min6",
    "ccajas_otras_max6",
    "ccajas_otras_ema3",
    "ccajas_otras_ema6",
    "catm_trx_lag1",
    "catm_trx_delta1",
    "catm_trx_lag2",
    "catm_trx_delta2",
    "catm_trx_ols6",
    "catm_trx_sma3",
    "catm_trx_sma6",
    "catm_trx_min6",
    "catm_trx_max6",
    "catm_trx_ema3",
    "catm_trx_ema6",
    "matm_lag1",
    "matm_delta1",
    "matm_lag2",
    "matm_delta2",
    "matm_ols6",
    "matm_sma3",
    "matm_sma6",
    "matm_min6",
    "matm_max6",
    "matm_ema3",
    "matm_ema6",
    "catm_trx_other_lag1",
    "catm_trx_other_delta1",
    "catm_trx_other_lag2",
    "catm_trx_other_delta2",
    "catm_trx_other_ols6",
    "catm_trx_other_sma3",
    "catm_trx_other_sma6",
    "catm_trx_other_min6",
    "catm_trx_other_max6",
    "catm_trx_other_ema3",
    "catm_trx_other_ema6",
    "matm_other_lag1",
    "matm_other_delta1",
    "matm_other_lag2",
    "matm_other_delta2",
    "matm_other_ols6",
    "matm_other_sma3",
    "matm_other_sma6",
    "matm_other_min6",
    "matm_other_max6",
    "matm_other_ema3",
    "matm_other_ema6",
    "ctrx_quarter_delta2",
    "ctrx_quarter_ols6",
    "ctrx_quarter_sma3",
    "ctrx_quarter_sma6",
    "ctrx_quarter_min6",
    "ctrx_quarter_max6",
    "ctrx_quarter_ema3",
    "ctrx_quarter_ema6",
    "tmobile_app_lag1",
    "tmobile_app_delta1",
    "tmobile_app_lag2",
    "tmobile_app_delta2",
    "tmobile_app_ols6",
    "tmobile_app_sma3",
    "tmobile_app_sma6",
    "tmobile_app_min6",
    "tmobile_app_max6",
    "tmobile_app_ema3",
    "tmobile_app_ema6",
    "cmobile_app_trx_lag1",
    "cmobile_app_trx_delta1",
    "cmobile_app_trx_lag2",
    "cmobile_app_trx_delta2",
    "cmobile_app_trx_ols6",
    "cmobile_app_trx_sma3",
    "cmobile_app_trx_sma6",
    "cmobile_app_trx_min6",
    "cmobile_app_trx_max6",
    "cmobile_app_trx_ema3",
    "cmobile_app_trx_ema6",
    "Master_delinquency_lag1",
    "Master_delinquency_delta1",
    "Master_delinquency_lag2",
    "Master_delinquency_delta2",
    "Master_delinquency_ols6",
    "Master_delinquency_sma3",
    "Master_delinquency_sma6",
    "Master_delinquency_min6",
    "Master_delinquency_max6",
    "Master_delinquency_ema3",
    "Master_delinquency_ema6",
    "Master_status_lag1",
    "Master_status_delta1",
    "Master_status_lag2",
    "Master_status_delta2",
    "Master_status_ols6",
    "Master_status_sma3",
    "Master_status_sma6",
    "Master_status_min6",
    "Master_status_max6",
    "Master_status_ema3",
    "Master_status_ema6",
    "Master_mfinanciacion_limite_lag1",
    "Master_mfinanciacion_limite_delta1",
    "Master_mfinanciacion_limite_lag2",
    "Master_mfinanciacion_limite_delta2",
    "Master_mfinanciacion_limite_ols6",
    "Master_mfinanciacion_limite_sma3",
    "Master_mfinanciacion_limite_sma6",
    "Master_mfinanciacion_limite_min6",
    "Master_mfinanciacion_limite_max6",
    "Master_mfinanciacion_limite_ema3",
    "Master_mfinanciacion_limite_ema6",
    "Master_Fvencimiento_lag1",
    "Master_Fvencimiento_delta1",
    "Master_Fvencimiento_lag2",
    "Master_Fvencimiento_delta2",
    "Master_Fvencimiento_ols6",
    "Master_Fvencimiento_sma3",
    "Master_Fvencimiento_sma6",
    "Master_Fvencimiento_min6",
    "Master_Fvencimiento_max6",
    "Master_Fvencimiento_ema3",
    "Master_Fvencimiento_ema6",
    "Master_Finiciomora_lag1",
    "Master_Finiciomora_delta1",
    "Master_Finiciomora_lag2",
    "Master_Finiciomora_delta2",
    "Master_Finiciomora_ols6",
    "Master_Finiciomora_sma3",
    "Master_Finiciomora_sma6",
    "Master_Finiciomora_min6",
    "Master_Finiciomora_max6",
    "Master_Finiciomora_ema3",
    "Master_Finiciomora_ema6",
    "Master_msaldototal_lag1",
    "Master_msaldototal_delta1",
    "Master_msaldototal_lag2",
    "Master_msaldototal_delta2",
    "Master_msaldototal_ols6",
    "Master_msaldototal_sma3",
    "Master_msaldototal_sma6",
    "Master_msaldototal_min6",
    "Master_msaldototal_max6",
    "Master_msaldototal_ema3",
    "Master_msaldototal_ema6",
    "Master_msaldopesos_lag1",
    "Master_msaldopesos_delta1",
    "Master_msaldopesos_lag2",
    "Master_msaldopesos_delta2",
    "Master_msaldopesos_ols6",
    "Master_msaldopesos_sma3",
    "Master_msaldopesos_sma6",
    "Master_msaldopesos_min6",
    "Master_msaldopesos_max6",
    "Master_msaldopesos_ema3",
    "Master_msaldopesos_ema6",
    "Master_msaldodolares_lag1",
    "Master_msaldodolares_delta1",
    "Master_msaldodolares_lag2",
    "Master_msaldodolares_delta2",
    "Master_msaldodolares_ols6",
    "Master_msaldodolares_sma3",
    "Master_msaldodolares_sma6",
    "Master_msaldodolares_min6",
    "Master_msaldodolares_max6",
    "Master_msaldodolares_ema3",
    "Master_msaldodolares_ema6",
    "Master_mconsumospesos_lag1",
    "Master_mconsumospesos_delta1",
    "Master_mconsumospesos_lag2",
    "Master_mconsumospesos_delta2",
    "Master_mconsumospesos_ols6",
    "Master_mconsumospesos_sma3",
    "Master_mconsumospesos_sma6",
    "Master_mconsumospesos_min6",
    "Master_mconsumospesos_max6",
    "Master_mconsumospesos_ema3",
    "Master_mconsumospesos_ema6",
    "Master_mconsumosdolares_lag1",
    "Master_mconsumosdolares_delta1",
    "Master_mconsumosdolares_lag2",
    "Master_mconsumosdolares_delta2",
    "Master_mconsumosdolares_ols6",
    "Master_mconsumosdolares_sma3",
    "Master_mconsumosdolares_sma6",
    "Master_mconsumosdolares_min6",
    "Master_mconsumosdolares_max6",
    "Master_mconsumosdolares_ema3",
    "Master_mconsumosdolares_ema6",
    "Master_mlimitecompra_lag1",
    "Master_mlimitecompra_delta1",
    "Master_mlimitecompra_lag2",
    "Master_mlimitecompra_delta2",
    "Master_mlimitecompra_ols6",
    "Master_mlimitecompra_sma3",
    "Master_mlimitecompra_sma6",
    "Master_mlimitecompra_min6",
    "Master_mlimitecompra_max6",
    "Master_mlimitecompra_ema3",
    "Master_mlimitecompra_ema6",
    "Master_madelantopesos_lag1",
    "Master_madelantopesos_delta1",
    "Master_madelantopesos_lag2",
    "Master_madelantopesos_delta2",
    "Master_madelantopesos_ols6",
    "Master_madelantopesos_sma3",
    "Master_madelantopesos_sma6",
    "Master_madelantopesos_min6",
    "Master_madelantopesos_max6",
    "Master_madelantopesos_ema3",
    "Master_madelantopesos_ema6",
    "Master_madelantodolares_lag1",
    "Master_madelantodolares_delta1",
    "Master_madelantodolares_lag2",
    "Master_madelantodolares_delta2",
    "Master_madelantodolares_ols6",
    "Master_madelantodolares_sma3",
    "Master_madelantodolares_sma6",
    "Master_madelantodolares_min6",
    "Master_madelantodolares_max6",
    "Master_madelantodolares_ema3",
    "Master_madelantodolares_ema6",
    "Master_fultimo_cierre_lag1",
    "Master_fultimo_cierre_delta1",
    "Master_fultimo_cierre_lag2",
    "Master_fultimo_cierre_delta2",
    "Master_fultimo_cierre_ols6",
    "Master_fultimo_cierre_sma3",
    "Master_fultimo_cierre_sma6",
    "Master_fultimo_cierre_min6",
    "Master_fultimo_cierre_max6",
    "Master_fultimo_cierre_ema3",
    "Master_fultimo_cierre_ema6",
    "Master_mpagado_lag1",
    "Master_mpagado_delta1",
    "Master_mpagado_lag2",
    "Master_mpagado_delta2",
    "Master_mpagado_ols6",
    "Master_mpagado_sma3",
    "Master_mpagado_sma6",
    "Master_mpagado_min6",
    "Master_mpagado_max6",
    "Master_mpagado_ema3",
    "Master_mpagado_ema6",
    "Master_mpagospesos_lag1",
    "Master_mpagospesos_delta1",
    "Master_mpagospesos_lag2",
    "Master_mpagospesos_delta2",
    "Master_mpagospesos_ols6",
    "Master_mpagospesos_sma3",
    "Master_mpagospesos_sma6",
    "Master_mpagospesos_min6",
    "Master_mpagospesos_max6",
    "Master_mpagospesos_ema3",
    "Master_mpagospesos_ema6",
    "Master_mpagosdolares_lag1",
    "Master_mpagosdolares_delta1",
    "Master_mpagosdolares_lag2",
    "Master_mpagosdolares_delta2",
    "Master_mpagosdolares_ols6",
    "Master_mpagosdolares_sma3",
    "Master_mpagosdolares_sma6",
    "Master_mpagosdolares_min6",
    "Master_mpagosdolares_max6",
    "Master_mpagosdolares_ema3",
    "Master_mpagosdolares_ema6",
    "Master_fechaalta_lag1",
    "Master_fechaalta_delta1",
    "Master_fechaalta_lag2",
    "Master_fechaalta_delta2",
    "Master_fechaalta_ols6",
    "Master_fechaalta_sma3",
    "Master_fechaalta_sma6",
    "Master_fechaalta_min6",
    "Master_fechaalta_max6",
    "Master_fechaalta_ema3",
    "Master_fechaalta_ema6",
    "Master_mconsumototal_lag1",
    "Master_mconsumototal_delta1",
    "Master_mconsumototal_lag2",
    "Master_mconsumototal_delta2",
    "Master_mconsumototal_ols6",
    "Master_mconsumototal_sma3",
    "Master_mconsumototal_sma6",
    "Master_mconsumototal_min6",
    "Master_mconsumototal_max6",
    "Master_mconsumototal_ema3",
    "Master_mconsumototal_ema6",
    "Master_cconsumos_lag1",
    "Master_cconsumos_delta1",
    "Master_cconsumos_lag2",
    "Master_cconsumos_delta2",
    "Master_cconsumos_ols6",
    "Master_cconsumos_sma3",
    "Master_cconsumos_sma6",
    "Master_cconsumos_min6",
    "Master_cconsumos_max6",
    "Master_cconsumos_ema3",
    "Master_cconsumos_ema6",
    "Master_cadelantosefectivo_lag1",
    "Master_cadelantosefectivo_delta1",
    "Master_cadelantosefectivo_lag2",
    "Master_cadelantosefectivo_delta2",
    "Master_cadelantosefectivo_ols6",
    "Master_cadelantosefectivo_sma3",
    "Master_cadelantosefectivo_sma6",
    "Master_cadelantosefectivo_min6",
    "Master_cadelantosefectivo_max6",
    "Master_cadelantosefectivo_ema3",
    "Master_cadelantosefectivo_ema6",
    "Master_mpagominimo_lag1",
    "Master_mpagominimo_delta1",
    "Master_mpagominimo_lag2",
    "Master_mpagominimo_delta2",
    "Master_mpagominimo_ols6",
    "Master_mpagominimo_sma3",
    "Master_mpagominimo_sma6",
    "Master_mpagominimo_min6",
    "Master_mpagominimo_max6",
    "Master_mpagominimo_ema3",
    "Master_mpagominimo_ema6",
    "Visa_delinquency_lag1",
    "Visa_delinquency_delta1",
    "Visa_delinquency_lag2",
    "Visa_delinquency_delta2",
    "Visa_delinquency_ols6",
    "Visa_delinquency_sma3",
    "Visa_delinquency_sma6",
    "Visa_delinquency_min6",
    "Visa_delinquency_max6",
    "Visa_delinquency_ema3",
    "Visa_delinquency_ema6",
    "Visa_status_lag1",
    "Visa_status_delta1",
    "Visa_status_lag2",
    "Visa_status_delta2",
    "Visa_status_ols6",
    "Visa_status_sma3",
    "Visa_status_sma6",
    "Visa_status_min6",
    "Visa_status_max6",
    "Visa_status_ema3",
    "Visa_status_ema6",
    "Visa_mfinanciacion_limite_lag1",
    "Visa_mfinanciacion_limite_delta1",
    "Visa_mfinanciacion_limite_lag2",
    "Visa_mfinanciacion_limite_delta2",
    "Visa_mfinanciacion_limite_ols6",
    "Visa_mfinanciacion_limite_sma3",
    "Visa_mfinanciacion_limite_sma6",
    "Visa_mfinanciacion_limite_min6",
    "Visa_mfinanciacion_limite_max6",
    "Visa_mfinanciacion_limite_ema3",
    "Visa_mfinanciacion_limite_ema6",
    "Visa_Fvencimiento_lag1",
    "Visa_Fvencimiento_delta1",
    "Visa_Fvencimiento_lag2",
    "Visa_Fvencimiento_delta2",
    "Visa_Fvencimiento_ols6",
    "Visa_Fvencimiento_sma3",
    "Visa_Fvencimiento_sma6",
    "Visa_Fvencimiento_min6",
    "Visa_Fvencimiento_max6",
    "Visa_Fvencimiento_ema3",
    "Visa_Fvencimiento_ema6",
    "Visa_Finiciomora_lag1",
    "Visa_Finiciomora_delta1",
    "Visa_Finiciomora_lag2",
    "Visa_Finiciomora_delta2",
    "Visa_Finiciomora_ols6",
    "Visa_Finiciomora_sma3",
    "Visa_Finiciomora_sma6",
    "Visa_Finiciomora_min6",
    "Visa_Finiciomora_max6",
    "Visa_Finiciomora_ema3",
    "Visa_Finiciomora_ema6",
    "Visa_msaldototal_lag1",
    "Visa_msaldototal_delta1",
    "Visa_msaldototal_lag2",
    "Visa_msaldototal_delta2",
    "Visa_msaldototal_ols6",
    "Visa_msaldototal_sma3",
    "Visa_msaldototal_sma6",
    "Visa_msaldototal_min6",
    "Visa_msaldototal_max6",
    "Visa_msaldototal_ema3",
    "Visa_msaldototal_ema6",
    "Visa_msaldopesos_lag1",
    "Visa_msaldopesos_delta1",
    "Visa_msaldopesos_lag2",
    "Visa_msaldopesos_delta2",
    "Visa_msaldopesos_ols6",
    "Visa_msaldopesos_sma3",
    "Visa_msaldopesos_sma6",
    "Visa_msaldopesos_min6",
    "Visa_msaldopesos_max6",
    "Visa_msaldopesos_ema3",
    "Visa_msaldopesos_ema6",
    "Visa_msaldodolares_lag1",
    "Visa_msaldodolares_delta1",
    "Visa_msaldodolares_lag2",
    "Visa_msaldodolares_delta2",
    "Visa_msaldodolares_ols6",
    "Visa_msaldodolares_sma3",
    "Visa_msaldodolares_sma6",
    "Visa_msaldodolares_min6",
    "Visa_msaldodolares_max6",
    "Visa_msaldodolares_ema3",
    "Visa_msaldodolares_ema6",
    "Visa_mconsumospesos_lag1",
    "Visa_mconsumospesos_delta1",
    "Visa_mconsumospesos_lag2",
    "Visa_mconsumospesos_delta2",
    "Visa_mconsumospesos_ols6",
    "Visa_mconsumospesos_sma3",
    "Visa_mconsumospesos_sma6",
    "Visa_mconsumospesos_min6",
    "Visa_mconsumospesos_max6",
    "Visa_mconsumospesos_ema3",
    "Visa_mconsumospesos_ema6",
    "Visa_mconsumosdolares_lag1",
    "Visa_mconsumosdolares_delta1",
    "Visa_mconsumosdolares_lag2",
    "Visa_mconsumosdolares_delta2",
    "Visa_mconsumosdolares_ols6",
    "Visa_mconsumosdolares_sma3",
    "Visa_mconsumosdolares_sma6",
    "Visa_mconsumosdolares_min6",
    "Visa_mconsumosdolares_max6",
    "Visa_mconsumosdolares_ema3",
    "Visa_mconsumosdolares_ema6",
    "Visa_mlimitecompra_lag1",
    "Visa_mlimitecompra_delta1",
    "Visa_mlimitecompra_lag2",
    "Visa_mlimitecompra_delta2",
    "Visa_mlimitecompra_ols6",
    "Visa_mlimitecompra_sma3",
    "Visa_mlimitecompra_sma6",
    "Visa_mlimitecompra_min6",
    "Visa_mlimitecompra_max6",
    "Visa_mlimitecompra_ema3",
    "Visa_mlimitecompra_ema6",
    "Visa_madelantopesos_lag1",
    "Visa_madelantopesos_delta1",
    "Visa_madelantopesos_lag2",
    "Visa_madelantopesos_delta2",
    "Visa_madelantopesos_ols6",
    "Visa_madelantopesos_sma3",
    "Visa_madelantopesos_sma6",
    "Visa_madelantopesos_min6",
    "Visa_madelantopesos_max6",
    "Visa_madelantopesos_ema3",
    "Visa_madelantopesos_ema6",
    "Visa_madelantodolares_lag1",
    "Visa_madelantodolares_delta1",
    "Visa_madelantodolares_lag2",
    "Visa_madelantodolares_delta2",
    "Visa_madelantodolares_ols6",
    "Visa_madelantodolares_sma3",
    "Visa_madelantodolares_sma6",
    "Visa_madelantodolares_min6",
    "Visa_madelantodolares_max6",
    "Visa_madelantodolares_ema3",
    "Visa_madelantodolares_ema6",
    "Visa_fultimo_cierre_lag1",
    "Visa_fultimo_cierre_delta1",
    "Visa_fultimo_cierre_lag2",
    "Visa_fultimo_cierre_delta2",
    "Visa_fultimo_cierre_ols6",
    "Visa_fultimo_cierre_sma3",
    "Visa_fultimo_cierre_sma6",
    "Visa_fultimo_cierre_min6",
    "Visa_fultimo_cierre_max6",
    "Visa_fultimo_cierre_ema3",
    "Visa_fultimo_cierre_ema6",
    "Visa_mpagado_lag1",
    "Visa_mpagado_delta1",
    "Visa_mpagado_lag2",
    "Visa_mpagado_delta2",
    "Visa_mpagado_ols6",
    "Visa_mpagado_sma3",
    "Visa_mpagado_sma6",
    "Visa_mpagado_min6",
    "Visa_mpagado_max6",
    "Visa_mpagado_ema3",
    "Visa_mpagado_ema6",
    "Visa_mpagospesos_lag1",
    "Visa_mpagospesos_delta1",
    "Visa_mpagospesos_lag2",
    "Visa_mpagospesos_delta2",
    "Visa_mpagospesos_ols6",
    "Visa_mpagospesos_sma3",
    "Visa_mpagospesos_sma6",
    "Visa_mpagospesos_min6",
    "Visa_mpagospesos_max6",
    "Visa_mpagospesos_ema3",
    "Visa_mpagospesos_ema6",
    "Visa_mpagosdolares_lag1",
    "Visa_mpagosdolares_delta1",
    "Visa_mpagosdolares_lag2",
    "Visa_mpagosdolares_delta2",
    "Visa_mpagosdolares_ols6",
    "Visa_mpagosdolares_sma3",
    "Visa_mpagosdolares_sma6",
    "Visa_mpagosdolares_min6",
    "Visa_mpagosdolares_max6",
    "Visa_mpagosdolares_ema3",
    "Visa_mpagosdolares_ema6",
    "Visa_fechaalta_lag1",
    "Visa_fechaalta_delta1",
    "Visa_fechaalta_lag2",
    "Visa_fechaalta_delta2",
    "Visa_fechaalta_ols6",
    "Visa_fechaalta_sma3",
    "Visa_fechaalta_sma6",
    "Visa_fechaalta_min6",
    "Visa_fechaalta_max6",
    "Visa_fechaalta_ema3",
    "Visa_fechaalta_ema6",
    "Visa_mconsumototal_lag1",
    "Visa_mconsumototal_delta1",
    "Visa_mconsumototal_lag2",
    "Visa_mconsumototal_delta2",
    "Visa_mconsumototal_ols6",
    "Visa_mconsumototal_sma3",
    "Visa_mconsumototal_sma6",
    "Visa_mconsumototal_min6",
    "Visa_mconsumototal_max6",
    "Visa_mconsumototal_ema3",
    "Visa_mconsumototal_ema6",
    "Visa_cconsumos_lag1",
    "Visa_cconsumos_delta1",
    "Visa_cconsumos_lag2",
    "Visa_cconsumos_delta2",
    "Visa_cconsumos_ols6",
    "Visa_cconsumos_sma3",
    "Visa_cconsumos_sma6",
    "Visa_cconsumos_min6",
    "Visa_cconsumos_max6",
    "Visa_cconsumos_ema3",
    "Visa_cconsumos_ema6",
    "Visa_cadelantosefectivo_lag1",
    "Visa_cadelantosefectivo_delta1",
    "Visa_cadelantosefectivo_lag2",
    "Visa_cadelantosefectivo_delta2",
    "Visa_cadelantosefectivo_ols6",
    "Visa_cadelantosefectivo_sma3",
    "Visa_cadelantosefectivo_sma6",
    "Visa_cadelantosefectivo_min6",
    "Visa_cadelantosefectivo_max6",
    "Visa_cadelantosefectivo_ema3",
    "Visa_cadelantosefectivo_ema6",
    "Visa_mpagominimo_lag1",
    "Visa_mpagominimo_delta1",
    "Visa_mpagominimo_lag2",
    "Visa_mpagominimo_delta2",
    "Visa_mpagominimo_ols6",
    "Visa_mpagominimo_sma3",
    "Visa_mpagominimo_sma6",
    "Visa_mpagominimo_min6",
    "Visa_mpagominimo_max6",
    "Visa_mpagominimo_ema3",
    "Visa_mpagominimo_ema6"
  ],
  "mad_columns": [
    "active_quarter",
    "cliente_vip",
    "internet",
    "cliente_edad",
    "cliente_antiguedad",
    "mrentabilidad",
    "mrentabilidad_annual",
    "mcomisiones",
    "mactivos_margen",
    "mpasivos_margen",
    "cproductos",
    "tcuentas",
    "ccuenta_corriente",
    "mcuenta_corriente_adicional",
    "mcuenta_corriente",
    "ccaja_ahorro",
    "mcaja_ahorro",
    "mcaja_ahorro_adicional",
    "mcaja_ahorro_dolares",
    "cdescubierto_preacordado",
    "mcuentas_saldo",
    "ctarjeta_debito",
    "ctarjeta_debito_transacciones",
    "mautoservicio",
    "ctarjeta_visa",
    "ctarjeta_visa_transacciones",
    "mtarjeta_visa_consumo",
    "ctarjeta_master",
    "ctarjeta_master_transacciones",
    "mtarjeta_master_consumo",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_prendarios",
    "mprestamos_prendarios",
    "cprestamos_hipotecarios",
    "mprestamos_hipotecarios",
    "cplazo_fijo",
    "mplazo_fijo_dolares",
    "mplazo_fijo_pesos",
    "cinversion1",
    "minversion1_pesos",
    "minversion1_dolares",
    "cinversion2",
    "minversion2",
    "cseguro_vida",
    "cseguro_auto",
    "cseguro_vivienda",
    "cseguro_accidentes_personales",
    "ccaja_seguridad",
    "cpayroll_trx",
    "mpayroll",
    "mpayroll2",
    "cpayroll2_trx",
    "ccuenta_debitos_automaticos",
    "mcuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos",
    "mttarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos",
    "mttarjeta_master_debitos_automaticos",
    "cpagodeservicios",
    "mpagodeservicios",
    "cpagomiscuentas",
    "mpagomiscuentas",
    "ccajeros_propios_descuentos",
    "mcajeros_propios_descuentos",
    "ctarjeta_visa_descuentos",
    "mtarjeta_visa_descuentos",
    "ctarjeta_master_descuentos",
    "mtarjeta_master_descuentos",
    "ccomisiones_mantenimiento",
    "mcomisiones_mantenimiento",
    "ccomisiones_otras",
    "mcomisiones_otras",
    "cforex",
    "cforex_buy",
    "mforex_buy",
    "cforex_sell",
    "mforex_sell",
    "ctransferencias_recibidas",
    "mtransferencias_recibidas",
    "ctransferencias_emitidas",
    "mtransferencias_emitidas",
    "cextraccion_autoservicio",
    "mextraccion_autoservicio",
    "ccheques_depositados",
    "mcheques_depositados",
    "ccheques_emitidos",
    "mcheques_emitidos",
    "ccheques_depositados_rechazados",
    "mcheques_depositados_rechazados",
    "ccheques_emitidos_rechazados",
    "mcheques_emitidos_rechazados",
    "tcallcenter",
    "ccallcenter_transacciones",
    "thomebanking",
    "chomebanking_transacciones",
    "ccajas_transacciones",
    "ccajas_consultas",
    "ccajas_depositos",
    "ccajas_extracciones",
    "ccajas_otras",
    "catm_trx",
    "matm",
    "catm_trx_other",
    "matm_other",
    "ctrx_quarter",
    "tmobile_app",
    "cmobile_app_trx",
    "Master_delinquency",
    "Master_status",
    "Master_mfinanciacion_limite",
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_msaldototal",
    "Master_msaldopesos",
    "Master_msaldodolares",
    "Master_mconsumospesos",
    "Master_mconsumosdolares",
    "Master_mlimitecompra",
    "Master_madelantopesos",
    "Master_madelantodolares",
    "Master_fultimo_cierre",
    "Master_mpagado",
    "Master_mpagospesos",
    "Master_mpagosdolares",
    "Master_fechaalta",
    "Master_mconsumototal",
    "Master_cconsumos",
    "Master_cadelantosefectivo",
    "Master_mpagominimo",
    "Visa_delinquency",
    "Visa_status",
    "Visa_mfinanciacion_limite",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_msaldototal",
    "Visa_msaldopesos",
    "Visa_msaldodolares",
    "Visa_mconsumospesos",
    "Visa_mconsumosdolares",
    "Visa_mlimitecompra",
    "Visa_madelantopesos",
    "Visa_madelantodolares",
    "Visa_fultimo_cierre",
    "Visa_mpagado",
    "Visa_mpagospesos",
    "Visa_mpagosdolares",
    "Visa_fechaalta",
    "Visa_mconsumototal",
    "Visa_cconsumos",
    "Visa_cadelantosefectivo",
    "Visa_mpagominimo",
    "history_month1_present",
    "history_month2_present",
    "history_month3_present",
    "history_month4_present",
    "history_month5_present",
    "history_month6_present",
    "mcuentas_saldo_lag1",
    "mcuentas_saldo_lag2",
    "mcuentas_saldo_lag3",
    "mcuentas_saldo_lag6",
    "mcuentas_saldo_delta1",
    "mcuentas_saldo_delta3",
    "mcuentas_saldo_prior3_mean",
    "mcuentas_saldo_prior3_observed",
    "mcuentas_saldo_prior6_mean",
    "mcuentas_saldo_prior6_observed",
    "mcuenta_corriente_lag1",
    "mcuenta_corriente_lag2",
    "mcuenta_corriente_lag3",
    "mcuenta_corriente_lag6",
    "mcuenta_corriente_delta1",
    "mcuenta_corriente_delta3",
    "mcuenta_corriente_prior3_mean",
    "mcuenta_corriente_prior3_observed",
    "mcuenta_corriente_prior6_mean",
    "mcuenta_corriente_prior6_observed",
    "mcaja_ahorro_lag1",
    "mcaja_ahorro_lag2",
    "mcaja_ahorro_lag3",
    "mcaja_ahorro_lag6",
    "mcaja_ahorro_delta1",
    "mcaja_ahorro_delta3",
    "mcaja_ahorro_prior3_mean",
    "mcaja_ahorro_prior3_observed",
    "mcaja_ahorro_prior6_mean",
    "mcaja_ahorro_prior6_observed",
    "mpayroll_lag1",
    "mpayroll_lag2",
    "mpayroll_lag3",
    "mpayroll_lag6",
    "mpayroll_delta1",
    "mpayroll_delta3",
    "mpayroll_prior3_mean",
    "mpayroll_prior3_observed",
    "mpayroll_prior6_mean",
    "mpayroll_prior6_observed",
    "mtarjeta_visa_consumo_lag1",
    "mtarjeta_visa_consumo_lag2",
    "mtarjeta_visa_consumo_lag3",
    "mtarjeta_visa_consumo_lag6",
    "mtarjeta_visa_consumo_delta1",
    "mtarjeta_visa_consumo_delta3",
    "mtarjeta_visa_consumo_prior3_mean",
    "mtarjeta_visa_consumo_prior3_observed",
    "mtarjeta_visa_consumo_prior6_mean",
    "mtarjeta_visa_consumo_prior6_observed",
    "mtarjeta_master_consumo_lag1",
    "mtarjeta_master_consumo_lag2",
    "mtarjeta_master_consumo_lag3",
    "mtarjeta_master_consumo_lag6",
    "mtarjeta_master_consumo_delta1",
    "mtarjeta_master_consumo_delta3",
    "mtarjeta_master_consumo_prior3_mean",
    "mtarjeta_master_consumo_prior3_observed",
    "mtarjeta_master_consumo_prior6_mean",
    "mtarjeta_master_consumo_prior6_observed",
    "ctrx_quarter_lag1",
    "ctrx_quarter_lag2",
    "ctrx_quarter_lag3",
    "ctrx_quarter_lag6",
    "ctrx_quarter_delta1",
    "ctrx_quarter_delta3",
    "ctrx_quarter_prior3_mean",
    "ctrx_quarter_prior3_observed",
    "ctrx_quarter_prior6_mean",
    "ctrx_quarter_prior6_observed",
    "ctarjeta_debito_transacciones_lag1",
    "ctarjeta_debito_transacciones_lag2",
    "ctarjeta_debito_transacciones_lag3",
    "ctarjeta_debito_transacciones_lag6",
    "ctarjeta_debito_transacciones_delta1",
    "ctarjeta_debito_transacciones_delta3",
    "ctarjeta_debito_transacciones_prior3_mean",
    "ctarjeta_debito_transacciones_prior3_observed",
    "ctarjeta_debito_transacciones_prior6_mean",
    "ctarjeta_debito_transacciones_prior6_observed",
    "mtransferencias_recibidas_lag1",
    "mtransferencias_recibidas_lag2",
    "mtransferencias_recibidas_lag3",
    "mtransferencias_recibidas_lag6",
    "mtransferencias_recibidas_delta1",
    "mtransferencias_recibidas_delta3",
    "mtransferencias_recibidas_prior3_mean",
    "mtransferencias_recibidas_prior3_observed",
    "mtransferencias_recibidas_prior6_mean",
    "mtransferencias_recibidas_prior6_observed",
    "mtransferencias_emitidas_lag1",
    "mtransferencias_emitidas_lag2",
    "mtransferencias_emitidas_lag3",
    "mtransferencias_emitidas_lag6",
    "mtransferencias_emitidas_delta1",
    "mtransferencias_emitidas_delta3",
    "mtransferencias_emitidas_prior3_mean",
    "mtransferencias_emitidas_prior3_observed",
    "mtransferencias_emitidas_prior6_mean",
    "mtransferencias_emitidas_prior6_observed",
    "mprestamos_personales_lag1",
    "mprestamos_personales_lag2",
    "mprestamos_personales_lag3",
    "mprestamos_personales_lag6",
    "mprestamos_personales_delta1",
    "mprestamos_personales_delta3",
    "mprestamos_personales_prior3_mean",
    "mprestamos_personales_prior3_observed",
    "mprestamos_personales_prior6_mean",
    "mprestamos_personales_prior6_observed",
    "mrentabilidad_lag1",
    "mrentabilidad_lag2",
    "mrentabilidad_lag3",
    "mrentabilidad_lag6",
    "mrentabilidad_delta1",
    "mrentabilidad_delta3",
    "mrentabilidad_prior3_mean",
    "mrentabilidad_prior3_observed",
    "mrentabilidad_prior6_mean",
    "mrentabilidad_prior6_observed",
    "active_quarter_lag1",
    "active_quarter_delta1",
    "active_quarter_lag2",
    "active_quarter_delta2",
    "active_quarter_ols6",
    "active_quarter_sma3",
    "active_quarter_sma6",
    "active_quarter_min6",
    "active_quarter_max6",
    "active_quarter_ema3",
    "active_quarter_ema6",
    "cliente_vip_lag1",
    "cliente_vip_delta1",
    "cliente_vip_lag2",
    "cliente_vip_delta2",
    "cliente_vip_ols6",
    "cliente_vip_sma3",
    "cliente_vip_sma6",
    "cliente_vip_min6",
    "cliente_vip_max6",
    "cliente_vip_ema3",
    "cliente_vip_ema6",
    "internet_lag1",
    "internet_delta1",
    "internet_lag2",
    "internet_delta2",
    "internet_ols6",
    "internet_sma3",
    "internet_sma6",
    "internet_min6",
    "internet_max6",
    "internet_ema3",
    "internet_ema6",
    "cliente_edad_lag1",
    "cliente_edad_delta1",
    "cliente_edad_lag2",
    "cliente_edad_delta2",
    "cliente_edad_ols6",
    "cliente_edad_sma3",
    "cliente_edad_sma6",
    "cliente_edad_min6",
    "cliente_edad_max6",
    "cliente_edad_ema3",
    "cliente_edad_ema6",
    "cliente_antiguedad_lag1",
    "cliente_antiguedad_delta1",
    "cliente_antiguedad_lag2",
    "cliente_antiguedad_delta2",
    "cliente_antiguedad_ols6",
    "cliente_antiguedad_sma3",
    "cliente_antiguedad_sma6",
    "cliente_antiguedad_min6",
    "cliente_antiguedad_max6",
    "cliente_antiguedad_ema3",
    "cliente_antiguedad_ema6",
    "mrentabilidad_delta2",
    "mrentabilidad_ols6",
    "mrentabilidad_sma3",
    "mrentabilidad_sma6",
    "mrentabilidad_min6",
    "mrentabilidad_max6",
    "mrentabilidad_ema3",
    "mrentabilidad_ema6",
    "mrentabilidad_annual_lag1",
    "mrentabilidad_annual_delta1",
    "mrentabilidad_annual_lag2",
    "mrentabilidad_annual_delta2",
    "mrentabilidad_annual_ols6",
    "mrentabilidad_annual_sma3",
    "mrentabilidad_annual_sma6",
    "mrentabilidad_annual_min6",
    "mrentabilidad_annual_max6",
    "mrentabilidad_annual_ema3",
    "mrentabilidad_annual_ema6",
    "mcomisiones_lag1",
    "mcomisiones_delta1",
    "mcomisiones_lag2",
    "mcomisiones_delta2",
    "mcomisiones_ols6",
    "mcomisiones_sma3",
    "mcomisiones_sma6",
    "mcomisiones_min6",
    "mcomisiones_max6",
    "mcomisiones_ema3",
    "mcomisiones_ema6",
    "mactivos_margen_lag1",
    "mactivos_margen_delta1",
    "mactivos_margen_lag2",
    "mactivos_margen_delta2",
    "mactivos_margen_ols6",
    "mactivos_margen_sma3",
    "mactivos_margen_sma6",
    "mactivos_margen_min6",
    "mactivos_margen_max6",
    "mactivos_margen_ema3",
    "mactivos_margen_ema6",
    "mpasivos_margen_lag1",
    "mpasivos_margen_delta1",
    "mpasivos_margen_lag2",
    "mpasivos_margen_delta2",
    "mpasivos_margen_ols6",
    "mpasivos_margen_sma3",
    "mpasivos_margen_sma6",
    "mpasivos_margen_min6",
    "mpasivos_margen_max6",
    "mpasivos_margen_ema3",
    "mpasivos_margen_ema6",
    "cproductos_lag1",
    "cproductos_delta1",
    "cproductos_lag2",
    "cproductos_delta2",
    "cproductos_ols6",
    "cproductos_sma3",
    "cproductos_sma6",
    "cproductos_min6",
    "cproductos_max6",
    "cproductos_ema3",
    "cproductos_ema6",
    "tcuentas_lag1",
    "tcuentas_delta1",
    "tcuentas_lag2",
    "tcuentas_delta2",
    "tcuentas_ols6",
    "tcuentas_sma3",
    "tcuentas_sma6",
    "tcuentas_min6",
    "tcuentas_max6",
    "tcuentas_ema3",
    "tcuentas_ema6",
    "ccuenta_corriente_lag1",
    "ccuenta_corriente_delta1",
    "ccuenta_corriente_lag2",
    "ccuenta_corriente_delta2",
    "ccuenta_corriente_ols6",
    "ccuenta_corriente_sma3",
    "ccuenta_corriente_sma6",
    "ccuenta_corriente_min6",
    "ccuenta_corriente_max6",
    "ccuenta_corriente_ema3",
    "ccuenta_corriente_ema6",
    "mcuenta_corriente_adicional_lag1",
    "mcuenta_corriente_adicional_delta1",
    "mcuenta_corriente_adicional_lag2",
    "mcuenta_corriente_adicional_delta2",
    "mcuenta_corriente_adicional_ols6",
    "mcuenta_corriente_adicional_sma3",
    "mcuenta_corriente_adicional_sma6",
    "mcuenta_corriente_adicional_min6",
    "mcuenta_corriente_adicional_max6",
    "mcuenta_corriente_adicional_ema3",
    "mcuenta_corriente_adicional_ema6",
    "mcuenta_corriente_delta2",
    "mcuenta_corriente_ols6",
    "mcuenta_corriente_sma3",
    "mcuenta_corriente_sma6",
    "mcuenta_corriente_min6",
    "mcuenta_corriente_max6",
    "mcuenta_corriente_ema3",
    "mcuenta_corriente_ema6",
    "ccaja_ahorro_lag1",
    "ccaja_ahorro_delta1",
    "ccaja_ahorro_lag2",
    "ccaja_ahorro_delta2",
    "ccaja_ahorro_ols6",
    "ccaja_ahorro_sma3",
    "ccaja_ahorro_sma6",
    "ccaja_ahorro_min6",
    "ccaja_ahorro_max6",
    "ccaja_ahorro_ema3",
    "ccaja_ahorro_ema6",
    "mcaja_ahorro_delta2",
    "mcaja_ahorro_ols6",
    "mcaja_ahorro_sma3",
    "mcaja_ahorro_sma6",
    "mcaja_ahorro_min6",
    "mcaja_ahorro_max6",
    "mcaja_ahorro_ema3",
    "mcaja_ahorro_ema6",
    "mcaja_ahorro_adicional_lag1",
    "mcaja_ahorro_adicional_delta1",
    "mcaja_ahorro_adicional_lag2",
    "mcaja_ahorro_adicional_delta2",
    "mcaja_ahorro_adicional_ols6",
    "mcaja_ahorro_adicional_sma3",
    "mcaja_ahorro_adicional_sma6",
    "mcaja_ahorro_adicional_min6",
    "mcaja_ahorro_adicional_max6",
    "mcaja_ahorro_adicional_ema3",
    "mcaja_ahorro_adicional_ema6",
    "mcaja_ahorro_dolares_lag1",
    "mcaja_ahorro_dolares_delta1",
    "mcaja_ahorro_dolares_lag2",
    "mcaja_ahorro_dolares_delta2",
    "mcaja_ahorro_dolares_ols6",
    "mcaja_ahorro_dolares_sma3",
    "mcaja_ahorro_dolares_sma6",
    "mcaja_ahorro_dolares_min6",
    "mcaja_ahorro_dolares_max6",
    "mcaja_ahorro_dolares_ema3",
    "mcaja_ahorro_dolares_ema6",
    "cdescubierto_preacordado_lag1",
    "cdescubierto_preacordado_delta1",
    "cdescubierto_preacordado_lag2",
    "cdescubierto_preacordado_delta2",
    "cdescubierto_preacordado_ols6",
    "cdescubierto_preacordado_sma3",
    "cdescubierto_preacordado_sma6",
    "cdescubierto_preacordado_min6",
    "cdescubierto_preacordado_max6",
    "cdescubierto_preacordado_ema3",
    "cdescubierto_preacordado_ema6",
    "mcuentas_saldo_delta2",
    "mcuentas_saldo_ols6",
    "mcuentas_saldo_sma3",
    "mcuentas_saldo_sma6",
    "mcuentas_saldo_min6",
    "mcuentas_saldo_max6",
    "mcuentas_saldo_ema3",
    "mcuentas_saldo_ema6",
    "ctarjeta_debito_lag1",
    "ctarjeta_debito_delta1",
    "ctarjeta_debito_lag2",
    "ctarjeta_debito_delta2",
    "ctarjeta_debito_ols6",
    "ctarjeta_debito_sma3",
    "ctarjeta_debito_sma6",
    "ctarjeta_debito_min6",
    "ctarjeta_debito_max6",
    "ctarjeta_debito_ema3",
    "ctarjeta_debito_ema6",
    "ctarjeta_debito_transacciones_delta2",
    "ctarjeta_debito_transacciones_ols6",
    "ctarjeta_debito_transacciones_sma3",
    "ctarjeta_debito_transacciones_sma6",
    "ctarjeta_debito_transacciones_min6",
    "ctarjeta_debito_transacciones_max6",
    "ctarjeta_debito_transacciones_ema3",
    "ctarjeta_debito_transacciones_ema6",
    "mautoservicio_lag1",
    "mautoservicio_delta1",
    "mautoservicio_lag2",
    "mautoservicio_delta2",
    "mautoservicio_ols6",
    "mautoservicio_sma3",
    "mautoservicio_sma6",
    "mautoservicio_min6",
    "mautoservicio_max6",
    "mautoservicio_ema3",
    "mautoservicio_ema6",
    "ctarjeta_visa_lag1",
    "ctarjeta_visa_delta1",
    "ctarjeta_visa_lag2",
    "ctarjeta_visa_delta2",
    "ctarjeta_visa_ols6",
    "ctarjeta_visa_sma3",
    "ctarjeta_visa_sma6",
    "ctarjeta_visa_min6",
    "ctarjeta_visa_max6",
    "ctarjeta_visa_ema3",
    "ctarjeta_visa_ema6",
    "ctarjeta_visa_transacciones_lag1",
    "ctarjeta_visa_transacciones_delta1",
    "ctarjeta_visa_transacciones_lag2",
    "ctarjeta_visa_transacciones_delta2",
    "ctarjeta_visa_transacciones_ols6",
    "ctarjeta_visa_transacciones_sma3",
    "ctarjeta_visa_transacciones_sma6",
    "ctarjeta_visa_transacciones_min6",
    "ctarjeta_visa_transacciones_max6",
    "ctarjeta_visa_transacciones_ema3",
    "ctarjeta_visa_transacciones_ema6",
    "mtarjeta_visa_consumo_delta2",
    "mtarjeta_visa_consumo_ols6",
    "mtarjeta_visa_consumo_sma3",
    "mtarjeta_visa_consumo_sma6",
    "mtarjeta_visa_consumo_min6",
    "mtarjeta_visa_consumo_max6",
    "mtarjeta_visa_consumo_ema3",
    "mtarjeta_visa_consumo_ema6",
    "ctarjeta_master_lag1",
    "ctarjeta_master_delta1",
    "ctarjeta_master_lag2",
    "ctarjeta_master_delta2",
    "ctarjeta_master_ols6",
    "ctarjeta_master_sma3",
    "ctarjeta_master_sma6",
    "ctarjeta_master_min6",
    "ctarjeta_master_max6",
    "ctarjeta_master_ema3",
    "ctarjeta_master_ema6",
    "ctarjeta_master_transacciones_lag1",
    "ctarjeta_master_transacciones_delta1",
    "ctarjeta_master_transacciones_lag2",
    "ctarjeta_master_transacciones_delta2",
    "ctarjeta_master_transacciones_ols6",
    "ctarjeta_master_transacciones_sma3",
    "ctarjeta_master_transacciones_sma6",
    "ctarjeta_master_transacciones_min6",
    "ctarjeta_master_transacciones_max6",
    "ctarjeta_master_transacciones_ema3",
    "ctarjeta_master_transacciones_ema6",
    "mtarjeta_master_consumo_delta2",
    "mtarjeta_master_consumo_ols6",
    "mtarjeta_master_consumo_sma3",
    "mtarjeta_master_consumo_sma6",
    "mtarjeta_master_consumo_min6",
    "mtarjeta_master_consumo_max6",
    "mtarjeta_master_consumo_ema3",
    "mtarjeta_master_consumo_ema6",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "cprestamos_personales_lag2",
    "cprestamos_personales_delta2",
    "cprestamos_personales_ols6",
    "cprestamos_personales_sma3",
    "cprestamos_personales_sma6",
    "cprestamos_personales_min6",
    "cprestamos_personales_max6",
    "cprestamos_personales_ema3",
    "cprestamos_personales_ema6",
    "mprestamos_personales_delta2",
    "mprestamos_personales_ols6",
    "mprestamos_personales_sma3",
    "mprestamos_personales_sma6",
    "mprestamos_personales_min6",
    "mprestamos_personales_max6",
    "mprestamos_personales_ema3",
    "mprestamos_personales_ema6",
    "cprestamos_prendarios_lag1",
    "cprestamos_prendarios_delta1",
    "cprestamos_prendarios_lag2",
    "cprestamos_prendarios_delta2",
    "cprestamos_prendarios_ols6",
    "cprestamos_prendarios_sma3",
    "cprestamos_prendarios_sma6",
    "cprestamos_prendarios_min6",
    "cprestamos_prendarios_max6",
    "cprestamos_prendarios_ema3",
    "cprestamos_prendarios_ema6",
    "mprestamos_prendarios_lag1",
    "mprestamos_prendarios_delta1",
    "mprestamos_prendarios_lag2",
    "mprestamos_prendarios_delta2",
    "mprestamos_prendarios_ols6",
    "mprestamos_prendarios_sma3",
    "mprestamos_prendarios_sma6",
    "mprestamos_prendarios_min6",
    "mprestamos_prendarios_max6",
    "mprestamos_prendarios_ema3",
    "mprestamos_prendarios_ema6",
    "cprestamos_hipotecarios_lag1",
    "cprestamos_hipotecarios_delta1",
    "cprestamos_hipotecarios_lag2",
    "cprestamos_hipotecarios_delta2",
    "cprestamos_hipotecarios_ols6",
    "cprestamos_hipotecarios_sma3",
    "cprestamos_hipotecarios_sma6",
    "cprestamos_hipotecarios_min6",
    "cprestamos_hipotecarios_max6",
    "cprestamos_hipotecarios_ema3",
    "cprestamos_hipotecarios_ema6",
    "mprestamos_hipotecarios_lag1",
    "mprestamos_hipotecarios_delta1",
    "mprestamos_hipotecarios_lag2",
    "mprestamos_hipotecarios_delta2",
    "mprestamos_hipotecarios_ols6",
    "mprestamos_hipotecarios_sma3",
    "mprestamos_hipotecarios_sma6",
    "mprestamos_hipotecarios_min6",
    "mprestamos_hipotecarios_max6",
    "mprestamos_hipotecarios_ema3",
    "mprestamos_hipotecarios_ema6",
    "cplazo_fijo_lag1",
    "cplazo_fijo_delta1",
    "cplazo_fijo_lag2",
    "cplazo_fijo_delta2",
    "cplazo_fijo_ols6",
    "cplazo_fijo_sma3",
    "cplazo_fijo_sma6",
    "cplazo_fijo_min6",
    "cplazo_fijo_max6",
    "cplazo_fijo_ema3",
    "cplazo_fijo_ema6",
    "mplazo_fijo_dolares_lag1",
    "mplazo_fijo_dolares_delta1",
    "mplazo_fijo_dolares_lag2",
    "mplazo_fijo_dolares_delta2",
    "mplazo_fijo_dolares_ols6",
    "mplazo_fijo_dolares_sma3",
    "mplazo_fijo_dolares_sma6",
    "mplazo_fijo_dolares_min6",
    "mplazo_fijo_dolares_max6",
    "mplazo_fijo_dolares_ema3",
    "mplazo_fijo_dolares_ema6",
    "mplazo_fijo_pesos_lag1",
    "mplazo_fijo_pesos_delta1",
    "mplazo_fijo_pesos_lag2",
    "mplazo_fijo_pesos_delta2",
    "mplazo_fijo_pesos_ols6",
    "mplazo_fijo_pesos_sma3",
    "mplazo_fijo_pesos_sma6",
    "mplazo_fijo_pesos_min6",
    "mplazo_fijo_pesos_max6",
    "mplazo_fijo_pesos_ema3",
    "mplazo_fijo_pesos_ema6",
    "cinversion1_lag1",
    "cinversion1_delta1",
    "cinversion1_lag2",
    "cinversion1_delta2",
    "cinversion1_ols6",
    "cinversion1_sma3",
    "cinversion1_sma6",
    "cinversion1_min6",
    "cinversion1_max6",
    "cinversion1_ema3",
    "cinversion1_ema6",
    "minversion1_pesos_lag1",
    "minversion1_pesos_delta1",
    "minversion1_pesos_lag2",
    "minversion1_pesos_delta2",
    "minversion1_pesos_ols6",
    "minversion1_pesos_sma3",
    "minversion1_pesos_sma6",
    "minversion1_pesos_min6",
    "minversion1_pesos_max6",
    "minversion1_pesos_ema3",
    "minversion1_pesos_ema6",
    "minversion1_dolares_lag1",
    "minversion1_dolares_delta1",
    "minversion1_dolares_lag2",
    "minversion1_dolares_delta2",
    "minversion1_dolares_ols6",
    "minversion1_dolares_sma3",
    "minversion1_dolares_sma6",
    "minversion1_dolares_min6",
    "minversion1_dolares_max6",
    "minversion1_dolares_ema3",
    "minversion1_dolares_ema6",
    "cinversion2_lag1",
    "cinversion2_delta1",
    "cinversion2_lag2",
    "cinversion2_delta2",
    "cinversion2_ols6",
    "cinversion2_sma3",
    "cinversion2_sma6",
    "cinversion2_min6",
    "cinversion2_max6",
    "cinversion2_ema3",
    "cinversion2_ema6",
    "minversion2_lag1",
    "minversion2_delta1",
    "minversion2_lag2",
    "minversion2_delta2",
    "minversion2_ols6",
    "minversion2_sma3",
    "minversion2_sma6",
    "minversion2_min6",
    "minversion2_max6",
    "minversion2_ema3",
    "minversion2_ema6",
    "cseguro_vida_lag1",
    "cseguro_vida_delta1",
    "cseguro_vida_lag2",
    "cseguro_vida_delta2",
    "cseguro_vida_ols6",
    "cseguro_vida_sma3",
    "cseguro_vida_sma6",
    "cseguro_vida_min6",
    "cseguro_vida_max6",
    "cseguro_vida_ema3",
    "cseguro_vida_ema6",
    "cseguro_auto_lag1",
    "cseguro_auto_delta1",
    "cseguro_auto_lag2",
    "cseguro_auto_delta2",
    "cseguro_auto_ols6",
    "cseguro_auto_sma3",
    "cseguro_auto_sma6",
    "cseguro_auto_min6",
    "cseguro_auto_max6",
    "cseguro_auto_ema3",
    "cseguro_auto_ema6",
    "cseguro_vivienda_lag1",
    "cseguro_vivienda_delta1",
    "cseguro_vivienda_lag2",
    "cseguro_vivienda_delta2",
    "cseguro_vivienda_ols6",
    "cseguro_vivienda_sma3",
    "cseguro_vivienda_sma6",
    "cseguro_vivienda_min6",
    "cseguro_vivienda_max6",
    "cseguro_vivienda_ema3",
    "cseguro_vivienda_ema6",
    "cseguro_accidentes_personales_lag1",
    "cseguro_accidentes_personales_delta1",
    "cseguro_accidentes_personales_lag2",
    "cseguro_accidentes_personales_delta2",
    "cseguro_accidentes_personales_ols6",
    "cseguro_accidentes_personales_sma3",
    "cseguro_accidentes_personales_sma6",
    "cseguro_accidentes_personales_min6",
    "cseguro_accidentes_personales_max6",
    "cseguro_accidentes_personales_ema3",
    "cseguro_accidentes_personales_ema6",
    "ccaja_seguridad_lag1",
    "ccaja_seguridad_delta1",
    "ccaja_seguridad_lag2",
    "ccaja_seguridad_delta2",
    "ccaja_seguridad_ols6",
    "ccaja_seguridad_sma3",
    "ccaja_seguridad_sma6",
    "ccaja_seguridad_min6",
    "ccaja_seguridad_max6",
    "ccaja_seguridad_ema3",
    "ccaja_seguridad_ema6",
    "cpayroll_trx_lag1",
    "cpayroll_trx_delta1",
    "cpayroll_trx_lag2",
    "cpayroll_trx_delta2",
    "cpayroll_trx_ols6",
    "cpayroll_trx_sma3",
    "cpayroll_trx_sma6",
    "cpayroll_trx_min6",
    "cpayroll_trx_max6",
    "cpayroll_trx_ema3",
    "cpayroll_trx_ema6",
    "mpayroll_delta2",
    "mpayroll_ols6",
    "mpayroll_sma3",
    "mpayroll_sma6",
    "mpayroll_min6",
    "mpayroll_max6",
    "mpayroll_ema3",
    "mpayroll_ema6",
    "mpayroll2_lag1",
    "mpayroll2_delta1",
    "mpayroll2_lag2",
    "mpayroll2_delta2",
    "mpayroll2_ols6",
    "mpayroll2_sma3",
    "mpayroll2_sma6",
    "mpayroll2_min6",
    "mpayroll2_max6",
    "mpayroll2_ema3",
    "mpayroll2_ema6",
    "cpayroll2_trx_lag1",
    "cpayroll2_trx_delta1",
    "cpayroll2_trx_lag2",
    "cpayroll2_trx_delta2",
    "cpayroll2_trx_ols6",
    "cpayroll2_trx_sma3",
    "cpayroll2_trx_sma6",
    "cpayroll2_trx_min6",
    "cpayroll2_trx_max6",
    "cpayroll2_trx_ema3",
    "cpayroll2_trx_ema6",
    "ccuenta_debitos_automaticos_lag1",
    "ccuenta_debitos_automaticos_delta1",
    "ccuenta_debitos_automaticos_lag2",
    "ccuenta_debitos_automaticos_delta2",
    "ccuenta_debitos_automaticos_ols6",
    "ccuenta_debitos_automaticos_sma3",
    "ccuenta_debitos_automaticos_sma6",
    "ccuenta_debitos_automaticos_min6",
    "ccuenta_debitos_automaticos_max6",
    "ccuenta_debitos_automaticos_ema3",
    "ccuenta_debitos_automaticos_ema6",
    "mcuenta_debitos_automaticos_lag1",
    "mcuenta_debitos_automaticos_delta1",
    "mcuenta_debitos_automaticos_lag2",
    "mcuenta_debitos_automaticos_delta2",
    "mcuenta_debitos_automaticos_ols6",
    "mcuenta_debitos_automaticos_sma3",
    "mcuenta_debitos_automaticos_sma6",
    "mcuenta_debitos_automaticos_min6",
    "mcuenta_debitos_automaticos_max6",
    "mcuenta_debitos_automaticos_ema3",
    "mcuenta_debitos_automaticos_ema6",
    "ctarjeta_visa_debitos_automaticos_lag1",
    "ctarjeta_visa_debitos_automaticos_delta1",
    "ctarjeta_visa_debitos_automaticos_lag2",
    "ctarjeta_visa_debitos_automaticos_delta2",
    "ctarjeta_visa_debitos_automaticos_ols6",
    "ctarjeta_visa_debitos_automaticos_sma3",
    "ctarjeta_visa_debitos_automaticos_sma6",
    "ctarjeta_visa_debitos_automaticos_min6",
    "ctarjeta_visa_debitos_automaticos_max6",
    "ctarjeta_visa_debitos_automaticos_ema3",
    "ctarjeta_visa_debitos_automaticos_ema6",
    "mttarjeta_visa_debitos_automaticos_lag1",
    "mttarjeta_visa_debitos_automaticos_delta1",
    "mttarjeta_visa_debitos_automaticos_lag2",
    "mttarjeta_visa_debitos_automaticos_delta2",
    "mttarjeta_visa_debitos_automaticos_ols6",
    "mttarjeta_visa_debitos_automaticos_sma3",
    "mttarjeta_visa_debitos_automaticos_sma6",
    "mttarjeta_visa_debitos_automaticos_min6",
    "mttarjeta_visa_debitos_automaticos_max6",
    "mttarjeta_visa_debitos_automaticos_ema3",
    "mttarjeta_visa_debitos_automaticos_ema6",
    "ctarjeta_master_debitos_automaticos_lag1",
    "ctarjeta_master_debitos_automaticos_delta1",
    "ctarjeta_master_debitos_automaticos_lag2",
    "ctarjeta_master_debitos_automaticos_delta2",
    "ctarjeta_master_debitos_automaticos_ols6",
    "ctarjeta_master_debitos_automaticos_sma3",
    "ctarjeta_master_debitos_automaticos_sma6",
    "ctarjeta_master_debitos_automaticos_min6",
    "ctarjeta_master_debitos_automaticos_max6",
    "ctarjeta_master_debitos_automaticos_ema3",
    "ctarjeta_master_debitos_automaticos_ema6",
    "mttarjeta_master_debitos_automaticos_lag1",
    "mttarjeta_master_debitos_automaticos_delta1",
    "mttarjeta_master_debitos_automaticos_lag2",
    "mttarjeta_master_debitos_automaticos_delta2",
    "mttarjeta_master_debitos_automaticos_ols6",
    "mttarjeta_master_debitos_automaticos_sma3",
    "mttarjeta_master_debitos_automaticos_sma6",
    "mttarjeta_master_debitos_automaticos_min6",
    "mttarjeta_master_debitos_automaticos_max6",
    "mttarjeta_master_debitos_automaticos_ema3",
    "mttarjeta_master_debitos_automaticos_ema6",
    "cpagodeservicios_lag1",
    "cpagodeservicios_delta1",
    "cpagodeservicios_lag2",
    "cpagodeservicios_delta2",
    "cpagodeservicios_ols6",
    "cpagodeservicios_sma3",
    "cpagodeservicios_sma6",
    "cpagodeservicios_min6",
    "cpagodeservicios_max6",
    "cpagodeservicios_ema3",
    "cpagodeservicios_ema6",
    "mpagodeservicios_lag1",
    "mpagodeservicios_delta1",
    "mpagodeservicios_lag2",
    "mpagodeservicios_delta2",
    "mpagodeservicios_ols6",
    "mpagodeservicios_sma3",
    "mpagodeservicios_sma6",
    "mpagodeservicios_min6",
    "mpagodeservicios_max6",
    "mpagodeservicios_ema3",
    "mpagodeservicios_ema6",
    "cpagomiscuentas_lag1",
    "cpagomiscuentas_delta1",
    "cpagomiscuentas_lag2",
    "cpagomiscuentas_delta2",
    "cpagomiscuentas_ols6",
    "cpagomiscuentas_sma3",
    "cpagomiscuentas_sma6",
    "cpagomiscuentas_min6",
    "cpagomiscuentas_max6",
    "cpagomiscuentas_ema3",
    "cpagomiscuentas_ema6",
    "mpagomiscuentas_lag1",
    "mpagomiscuentas_delta1",
    "mpagomiscuentas_lag2",
    "mpagomiscuentas_delta2",
    "mpagomiscuentas_ols6",
    "mpagomiscuentas_sma3",
    "mpagomiscuentas_sma6",
    "mpagomiscuentas_min6",
    "mpagomiscuentas_max6",
    "mpagomiscuentas_ema3",
    "mpagomiscuentas_ema6",
    "ccajeros_propios_descuentos_lag1",
    "ccajeros_propios_descuentos_delta1",
    "ccajeros_propios_descuentos_lag2",
    "ccajeros_propios_descuentos_delta2",
    "ccajeros_propios_descuentos_ols6",
    "ccajeros_propios_descuentos_sma3",
    "ccajeros_propios_descuentos_sma6",
    "ccajeros_propios_descuentos_min6",
    "ccajeros_propios_descuentos_max6",
    "ccajeros_propios_descuentos_ema3",
    "ccajeros_propios_descuentos_ema6",
    "mcajeros_propios_descuentos_lag1",
    "mcajeros_propios_descuentos_delta1",
    "mcajeros_propios_descuentos_lag2",
    "mcajeros_propios_descuentos_delta2",
    "mcajeros_propios_descuentos_ols6",
    "mcajeros_propios_descuentos_sma3",
    "mcajeros_propios_descuentos_sma6",
    "mcajeros_propios_descuentos_min6",
    "mcajeros_propios_descuentos_max6",
    "mcajeros_propios_descuentos_ema3",
    "mcajeros_propios_descuentos_ema6",
    "ctarjeta_visa_descuentos_lag1",
    "ctarjeta_visa_descuentos_delta1",
    "ctarjeta_visa_descuentos_lag2",
    "ctarjeta_visa_descuentos_delta2",
    "ctarjeta_visa_descuentos_ols6",
    "ctarjeta_visa_descuentos_sma3",
    "ctarjeta_visa_descuentos_sma6",
    "ctarjeta_visa_descuentos_min6",
    "ctarjeta_visa_descuentos_max6",
    "ctarjeta_visa_descuentos_ema3",
    "ctarjeta_visa_descuentos_ema6",
    "mtarjeta_visa_descuentos_lag1",
    "mtarjeta_visa_descuentos_delta1",
    "mtarjeta_visa_descuentos_lag2",
    "mtarjeta_visa_descuentos_delta2",
    "mtarjeta_visa_descuentos_ols6",
    "mtarjeta_visa_descuentos_sma3",
    "mtarjeta_visa_descuentos_sma6",
    "mtarjeta_visa_descuentos_min6",
    "mtarjeta_visa_descuentos_max6",
    "mtarjeta_visa_descuentos_ema3",
    "mtarjeta_visa_descuentos_ema6",
    "ctarjeta_master_descuentos_lag1",
    "ctarjeta_master_descuentos_delta1",
    "ctarjeta_master_descuentos_lag2",
    "ctarjeta_master_descuentos_delta2",
    "ctarjeta_master_descuentos_ols6",
    "ctarjeta_master_descuentos_sma3",
    "ctarjeta_master_descuentos_sma6",
    "ctarjeta_master_descuentos_min6",
    "ctarjeta_master_descuentos_max6",
    "ctarjeta_master_descuentos_ema3",
    "ctarjeta_master_descuentos_ema6",
    "mtarjeta_master_descuentos_lag1",
    "mtarjeta_master_descuentos_delta1",
    "mtarjeta_master_descuentos_lag2",
    "mtarjeta_master_descuentos_delta2",
    "mtarjeta_master_descuentos_ols6",
    "mtarjeta_master_descuentos_sma3",
    "mtarjeta_master_descuentos_sma6",
    "mtarjeta_master_descuentos_min6",
    "mtarjeta_master_descuentos_max6",
    "mtarjeta_master_descuentos_ema3",
    "mtarjeta_master_descuentos_ema6",
    "ccomisiones_mantenimiento_lag1",
    "ccomisiones_mantenimiento_delta1",
    "ccomisiones_mantenimiento_lag2",
    "ccomisiones_mantenimiento_delta2",
    "ccomisiones_mantenimiento_ols6",
    "ccomisiones_mantenimiento_sma3",
    "ccomisiones_mantenimiento_sma6",
    "ccomisiones_mantenimiento_min6",
    "ccomisiones_mantenimiento_max6",
    "ccomisiones_mantenimiento_ema3",
    "ccomisiones_mantenimiento_ema6",
    "mcomisiones_mantenimiento_lag1",
    "mcomisiones_mantenimiento_delta1",
    "mcomisiones_mantenimiento_lag2",
    "mcomisiones_mantenimiento_delta2",
    "mcomisiones_mantenimiento_ols6",
    "mcomisiones_mantenimiento_sma3",
    "mcomisiones_mantenimiento_sma6",
    "mcomisiones_mantenimiento_min6",
    "mcomisiones_mantenimiento_max6",
    "mcomisiones_mantenimiento_ema3",
    "mcomisiones_mantenimiento_ema6",
    "ccomisiones_otras_lag1",
    "ccomisiones_otras_delta1",
    "ccomisiones_otras_lag2",
    "ccomisiones_otras_delta2",
    "ccomisiones_otras_ols6",
    "ccomisiones_otras_sma3",
    "ccomisiones_otras_sma6",
    "ccomisiones_otras_min6",
    "ccomisiones_otras_max6",
    "ccomisiones_otras_ema3",
    "ccomisiones_otras_ema6",
    "mcomisiones_otras_lag1",
    "mcomisiones_otras_delta1",
    "mcomisiones_otras_lag2",
    "mcomisiones_otras_delta2",
    "mcomisiones_otras_ols6",
    "mcomisiones_otras_sma3",
    "mcomisiones_otras_sma6",
    "mcomisiones_otras_min6",
    "mcomisiones_otras_max6",
    "mcomisiones_otras_ema3",
    "mcomisiones_otras_ema6",
    "cforex_lag1",
    "cforex_delta1",
    "cforex_lag2",
    "cforex_delta2",
    "cforex_ols6",
    "cforex_sma3",
    "cforex_sma6",
    "cforex_min6",
    "cforex_max6",
    "cforex_ema3",
    "cforex_ema6",
    "cforex_buy_lag1",
    "cforex_buy_delta1",
    "cforex_buy_lag2",
    "cforex_buy_delta2",
    "cforex_buy_ols6",
    "cforex_buy_sma3",
    "cforex_buy_sma6",
    "cforex_buy_min6",
    "cforex_buy_max6",
    "cforex_buy_ema3",
    "cforex_buy_ema6",
    "mforex_buy_lag1",
    "mforex_buy_delta1",
    "mforex_buy_lag2",
    "mforex_buy_delta2",
    "mforex_buy_ols6",
    "mforex_buy_sma3",
    "mforex_buy_sma6",
    "mforex_buy_min6",
    "mforex_buy_max6",
    "mforex_buy_ema3",
    "mforex_buy_ema6",
    "cforex_sell_lag1",
    "cforex_sell_delta1",
    "cforex_sell_lag2",
    "cforex_sell_delta2",
    "cforex_sell_ols6",
    "cforex_sell_sma3",
    "cforex_sell_sma6",
    "cforex_sell_min6",
    "cforex_sell_max6",
    "cforex_sell_ema3",
    "cforex_sell_ema6",
    "mforex_sell_lag1",
    "mforex_sell_delta1",
    "mforex_sell_lag2",
    "mforex_sell_delta2",
    "mforex_sell_ols6",
    "mforex_sell_sma3",
    "mforex_sell_sma6",
    "mforex_sell_min6",
    "mforex_sell_max6",
    "mforex_sell_ema3",
    "mforex_sell_ema6",
    "ctransferencias_recibidas_lag1",
    "ctransferencias_recibidas_delta1",
    "ctransferencias_recibidas_lag2",
    "ctransferencias_recibidas_delta2",
    "ctransferencias_recibidas_ols6",
    "ctransferencias_recibidas_sma3",
    "ctransferencias_recibidas_sma6",
    "ctransferencias_recibidas_min6",
    "ctransferencias_recibidas_max6",
    "ctransferencias_recibidas_ema3",
    "ctransferencias_recibidas_ema6",
    "mtransferencias_recibidas_delta2",
    "mtransferencias_recibidas_ols6",
    "mtransferencias_recibidas_sma3",
    "mtransferencias_recibidas_sma6",
    "mtransferencias_recibidas_min6",
    "mtransferencias_recibidas_max6",
    "mtransferencias_recibidas_ema3",
    "mtransferencias_recibidas_ema6",
    "ctransferencias_emitidas_lag1",
    "ctransferencias_emitidas_delta1",
    "ctransferencias_emitidas_lag2",
    "ctransferencias_emitidas_delta2",
    "ctransferencias_emitidas_ols6",
    "ctransferencias_emitidas_sma3",
    "ctransferencias_emitidas_sma6",
    "ctransferencias_emitidas_min6",
    "ctransferencias_emitidas_max6",
    "ctransferencias_emitidas_ema3",
    "ctransferencias_emitidas_ema6",
    "mtransferencias_emitidas_delta2",
    "mtransferencias_emitidas_ols6",
    "mtransferencias_emitidas_sma3",
    "mtransferencias_emitidas_sma6",
    "mtransferencias_emitidas_min6",
    "mtransferencias_emitidas_max6",
    "mtransferencias_emitidas_ema3",
    "mtransferencias_emitidas_ema6",
    "cextraccion_autoservicio_lag1",
    "cextraccion_autoservicio_delta1",
    "cextraccion_autoservicio_lag2",
    "cextraccion_autoservicio_delta2",
    "cextraccion_autoservicio_ols6",
    "cextraccion_autoservicio_sma3",
    "cextraccion_autoservicio_sma6",
    "cextraccion_autoservicio_min6",
    "cextraccion_autoservicio_max6",
    "cextraccion_autoservicio_ema3",
    "cextraccion_autoservicio_ema6",
    "mextraccion_autoservicio_lag1",
    "mextraccion_autoservicio_delta1",
    "mextraccion_autoservicio_lag2",
    "mextraccion_autoservicio_delta2",
    "mextraccion_autoservicio_ols6",
    "mextraccion_autoservicio_sma3",
    "mextraccion_autoservicio_sma6",
    "mextraccion_autoservicio_min6",
    "mextraccion_autoservicio_max6",
    "mextraccion_autoservicio_ema3",
    "mextraccion_autoservicio_ema6",
    "ccheques_depositados_lag1",
    "ccheques_depositados_delta1",
    "ccheques_depositados_lag2",
    "ccheques_depositados_delta2",
    "ccheques_depositados_ols6",
    "ccheques_depositados_sma3",
    "ccheques_depositados_sma6",
    "ccheques_depositados_min6",
    "ccheques_depositados_max6",
    "ccheques_depositados_ema3",
    "ccheques_depositados_ema6",
    "mcheques_depositados_lag1",
    "mcheques_depositados_delta1",
    "mcheques_depositados_lag2",
    "mcheques_depositados_delta2",
    "mcheques_depositados_ols6",
    "mcheques_depositados_sma3",
    "mcheques_depositados_sma6",
    "mcheques_depositados_min6",
    "mcheques_depositados_max6",
    "mcheques_depositados_ema3",
    "mcheques_depositados_ema6",
    "ccheques_emitidos_lag1",
    "ccheques_emitidos_delta1",
    "ccheques_emitidos_lag2",
    "ccheques_emitidos_delta2",
    "ccheques_emitidos_ols6",
    "ccheques_emitidos_sma3",
    "ccheques_emitidos_sma6",
    "ccheques_emitidos_min6",
    "ccheques_emitidos_max6",
    "ccheques_emitidos_ema3",
    "ccheques_emitidos_ema6",
    "mcheques_emitidos_lag1",
    "mcheques_emitidos_delta1",
    "mcheques_emitidos_lag2",
    "mcheques_emitidos_delta2",
    "mcheques_emitidos_ols6",
    "mcheques_emitidos_sma3",
    "mcheques_emitidos_sma6",
    "mcheques_emitidos_min6",
    "mcheques_emitidos_max6",
    "mcheques_emitidos_ema3",
    "mcheques_emitidos_ema6",
    "ccheques_depositados_rechazados_lag1",
    "ccheques_depositados_rechazados_delta1",
    "ccheques_depositados_rechazados_lag2",
    "ccheques_depositados_rechazados_delta2",
    "ccheques_depositados_rechazados_ols6",
    "ccheques_depositados_rechazados_sma3",
    "ccheques_depositados_rechazados_sma6",
    "ccheques_depositados_rechazados_min6",
    "ccheques_depositados_rechazados_max6",
    "ccheques_depositados_rechazados_ema3",
    "ccheques_depositados_rechazados_ema6",
    "mcheques_depositados_rechazados_lag1",
    "mcheques_depositados_rechazados_delta1",
    "mcheques_depositados_rechazados_lag2",
    "mcheques_depositados_rechazados_delta2",
    "mcheques_depositados_rechazados_ols6",
    "mcheques_depositados_rechazados_sma3",
    "mcheques_depositados_rechazados_sma6",
    "mcheques_depositados_rechazados_min6",
    "mcheques_depositados_rechazados_max6",
    "mcheques_depositados_rechazados_ema3",
    "mcheques_depositados_rechazados_ema6",
    "ccheques_emitidos_rechazados_lag1",
    "ccheques_emitidos_rechazados_delta1",
    "ccheques_emitidos_rechazados_lag2",
    "ccheques_emitidos_rechazados_delta2",
    "ccheques_emitidos_rechazados_ols6",
    "ccheques_emitidos_rechazados_sma3",
    "ccheques_emitidos_rechazados_sma6",
    "ccheques_emitidos_rechazados_min6",
    "ccheques_emitidos_rechazados_max6",
    "ccheques_emitidos_rechazados_ema3",
    "ccheques_emitidos_rechazados_ema6",
    "mcheques_emitidos_rechazados_lag1",
    "mcheques_emitidos_rechazados_delta1",
    "mcheques_emitidos_rechazados_lag2",
    "mcheques_emitidos_rechazados_delta2",
    "mcheques_emitidos_rechazados_ols6",
    "mcheques_emitidos_rechazados_sma3",
    "mcheques_emitidos_rechazados_sma6",
    "mcheques_emitidos_rechazados_min6",
    "mcheques_emitidos_rechazados_max6",
    "mcheques_emitidos_rechazados_ema3",
    "mcheques_emitidos_rechazados_ema6",
    "tcallcenter_lag1",
    "tcallcenter_delta1",
    "tcallcenter_lag2",
    "tcallcenter_delta2",
    "tcallcenter_ols6",
    "tcallcenter_sma3",
    "tcallcenter_sma6",
    "tcallcenter_min6",
    "tcallcenter_max6",
    "tcallcenter_ema3",
    "tcallcenter_ema6",
    "ccallcenter_transacciones_lag1",
    "ccallcenter_transacciones_delta1",
    "ccallcenter_transacciones_lag2",
    "ccallcenter_transacciones_delta2",
    "ccallcenter_transacciones_ols6",
    "ccallcenter_transacciones_sma3",
    "ccallcenter_transacciones_sma6",
    "ccallcenter_transacciones_min6",
    "ccallcenter_transacciones_max6",
    "ccallcenter_transacciones_ema3",
    "ccallcenter_transacciones_ema6",
    "thomebanking_lag1",
    "thomebanking_delta1",
    "thomebanking_lag2",
    "thomebanking_delta2",
    "thomebanking_ols6",
    "thomebanking_sma3",
    "thomebanking_sma6",
    "thomebanking_min6",
    "thomebanking_max6",
    "thomebanking_ema3",
    "thomebanking_ema6",
    "chomebanking_transacciones_lag1",
    "chomebanking_transacciones_delta1",
    "chomebanking_transacciones_lag2",
    "chomebanking_transacciones_delta2",
    "chomebanking_transacciones_ols6",
    "chomebanking_transacciones_sma3",
    "chomebanking_transacciones_sma6",
    "chomebanking_transacciones_min6",
    "chomebanking_transacciones_max6",
    "chomebanking_transacciones_ema3",
    "chomebanking_transacciones_ema6",
    "ccajas_transacciones_lag1",
    "ccajas_transacciones_delta1",
    "ccajas_transacciones_lag2",
    "ccajas_transacciones_delta2",
    "ccajas_transacciones_ols6",
    "ccajas_transacciones_sma3",
    "ccajas_transacciones_sma6",
    "ccajas_transacciones_min6",
    "ccajas_transacciones_max6",
    "ccajas_transacciones_ema3",
    "ccajas_transacciones_ema6",
    "ccajas_consultas_lag1",
    "ccajas_consultas_delta1",
    "ccajas_consultas_lag2",
    "ccajas_consultas_delta2",
    "ccajas_consultas_ols6",
    "ccajas_consultas_sma3",
    "ccajas_consultas_sma6",
    "ccajas_consultas_min6",
    "ccajas_consultas_max6",
    "ccajas_consultas_ema3",
    "ccajas_consultas_ema6",
    "ccajas_depositos_lag1",
    "ccajas_depositos_delta1",
    "ccajas_depositos_lag2",
    "ccajas_depositos_delta2",
    "ccajas_depositos_ols6",
    "ccajas_depositos_sma3",
    "ccajas_depositos_sma6",
    "ccajas_depositos_min6",
    "ccajas_depositos_max6",
    "ccajas_depositos_ema3",
    "ccajas_depositos_ema6",
    "ccajas_extracciones_lag1",
    "ccajas_extracciones_delta1",
    "ccajas_extracciones_lag2",
    "ccajas_extracciones_delta2",
    "ccajas_extracciones_ols6",
    "ccajas_extracciones_sma3",
    "ccajas_extracciones_sma6",
    "ccajas_extracciones_min6",
    "ccajas_extracciones_max6",
    "ccajas_extracciones_ema3",
    "ccajas_extracciones_ema6",
    "ccajas_otras_lag1",
    "ccajas_otras_delta1",
    "ccajas_otras_lag2",
    "ccajas_otras_delta2",
    "ccajas_otras_ols6",
    "ccajas_otras_sma3",
    "ccajas_otras_sma6",
    "ccajas_otras_min6",
    "ccajas_otras_max6",
    "ccajas_otras_ema3",
    "ccajas_otras_ema6",
    "catm_trx_lag1",
    "catm_trx_delta1",
    "catm_trx_lag2",
    "catm_trx_delta2",
    "catm_trx_ols6",
    "catm_trx_sma3",
    "catm_trx_sma6",
    "catm_trx_min6",
    "catm_trx_max6",
    "catm_trx_ema3",
    "catm_trx_ema6",
    "matm_lag1",
    "matm_delta1",
    "matm_lag2",
    "matm_delta2",
    "matm_ols6",
    "matm_sma3",
    "matm_sma6",
    "matm_min6",
    "matm_max6",
    "matm_ema3",
    "matm_ema6",
    "catm_trx_other_lag1",
    "catm_trx_other_delta1",
    "catm_trx_other_lag2",
    "catm_trx_other_delta2",
    "catm_trx_other_ols6",
    "catm_trx_other_sma3",
    "catm_trx_other_sma6",
    "catm_trx_other_min6",
    "catm_trx_other_max6",
    "catm_trx_other_ema3",
    "catm_trx_other_ema6",
    "matm_other_lag1",
    "matm_other_delta1",
    "matm_other_lag2",
    "matm_other_delta2",
    "matm_other_ols6",
    "matm_other_sma3",
    "matm_other_sma6",
    "matm_other_min6",
    "matm_other_max6",
    "matm_other_ema3",
    "matm_other_ema6",
    "ctrx_quarter_delta2",
    "ctrx_quarter_ols6",
    "ctrx_quarter_sma3",
    "ctrx_quarter_sma6",
    "ctrx_quarter_min6",
    "ctrx_quarter_max6",
    "ctrx_quarter_ema3",
    "ctrx_quarter_ema6",
    "tmobile_app_lag1",
    "tmobile_app_delta1",
    "tmobile_app_lag2",
    "tmobile_app_delta2",
    "tmobile_app_ols6",
    "tmobile_app_sma3",
    "tmobile_app_sma6",
    "tmobile_app_min6",
    "tmobile_app_max6",
    "tmobile_app_ema3",
    "tmobile_app_ema6",
    "cmobile_app_trx_lag1",
    "cmobile_app_trx_delta1",
    "cmobile_app_trx_lag2",
    "cmobile_app_trx_delta2",
    "cmobile_app_trx_ols6",
    "cmobile_app_trx_sma3",
    "cmobile_app_trx_sma6",
    "cmobile_app_trx_min6",
    "cmobile_app_trx_max6",
    "cmobile_app_trx_ema3",
    "cmobile_app_trx_ema6",
    "Master_delinquency_lag1",
    "Master_delinquency_delta1",
    "Master_delinquency_lag2",
    "Master_delinquency_delta2",
    "Master_delinquency_ols6",
    "Master_delinquency_sma3",
    "Master_delinquency_sma6",
    "Master_delinquency_min6",
    "Master_delinquency_max6",
    "Master_delinquency_ema3",
    "Master_delinquency_ema6",
    "Master_status_lag1",
    "Master_status_delta1",
    "Master_status_lag2",
    "Master_status_delta2",
    "Master_status_ols6",
    "Master_status_sma3",
    "Master_status_sma6",
    "Master_status_min6",
    "Master_status_max6",
    "Master_status_ema3",
    "Master_status_ema6",
    "Master_mfinanciacion_limite_lag1",
    "Master_mfinanciacion_limite_delta1",
    "Master_mfinanciacion_limite_lag2",
    "Master_mfinanciacion_limite_delta2",
    "Master_mfinanciacion_limite_ols6",
    "Master_mfinanciacion_limite_sma3",
    "Master_mfinanciacion_limite_sma6",
    "Master_mfinanciacion_limite_min6",
    "Master_mfinanciacion_limite_max6",
    "Master_mfinanciacion_limite_ema3",
    "Master_mfinanciacion_limite_ema6",
    "Master_Fvencimiento_lag1",
    "Master_Fvencimiento_delta1",
    "Master_Fvencimiento_lag2",
    "Master_Fvencimiento_delta2",
    "Master_Fvencimiento_ols6",
    "Master_Fvencimiento_sma3",
    "Master_Fvencimiento_sma6",
    "Master_Fvencimiento_min6",
    "Master_Fvencimiento_max6",
    "Master_Fvencimiento_ema3",
    "Master_Fvencimiento_ema6",
    "Master_Finiciomora_lag1",
    "Master_Finiciomora_delta1",
    "Master_Finiciomora_lag2",
    "Master_Finiciomora_delta2",
    "Master_Finiciomora_ols6",
    "Master_Finiciomora_sma3",
    "Master_Finiciomora_sma6",
    "Master_Finiciomora_min6",
    "Master_Finiciomora_max6",
    "Master_Finiciomora_ema3",
    "Master_Finiciomora_ema6",
    "Master_msaldototal_lag1",
    "Master_msaldototal_delta1",
    "Master_msaldototal_lag2",
    "Master_msaldototal_delta2",
    "Master_msaldototal_ols6",
    "Master_msaldototal_sma3",
    "Master_msaldototal_sma6",
    "Master_msaldototal_min6",
    "Master_msaldototal_max6",
    "Master_msaldototal_ema3",
    "Master_msaldototal_ema6",
    "Master_msaldopesos_lag1",
    "Master_msaldopesos_delta1",
    "Master_msaldopesos_lag2",
    "Master_msaldopesos_delta2",
    "Master_msaldopesos_ols6",
    "Master_msaldopesos_sma3",
    "Master_msaldopesos_sma6",
    "Master_msaldopesos_min6",
    "Master_msaldopesos_max6",
    "Master_msaldopesos_ema3",
    "Master_msaldopesos_ema6",
    "Master_msaldodolares_lag1",
    "Master_msaldodolares_delta1",
    "Master_msaldodolares_lag2",
    "Master_msaldodolares_delta2",
    "Master_msaldodolares_ols6",
    "Master_msaldodolares_sma3",
    "Master_msaldodolares_sma6",
    "Master_msaldodolares_min6",
    "Master_msaldodolares_max6",
    "Master_msaldodolares_ema3",
    "Master_msaldodolares_ema6",
    "Master_mconsumospesos_lag1",
    "Master_mconsumospesos_delta1",
    "Master_mconsumospesos_lag2",
    "Master_mconsumospesos_delta2",
    "Master_mconsumospesos_ols6",
    "Master_mconsumospesos_sma3",
    "Master_mconsumospesos_sma6",
    "Master_mconsumospesos_min6",
    "Master_mconsumospesos_max6",
    "Master_mconsumospesos_ema3",
    "Master_mconsumospesos_ema6",
    "Master_mconsumosdolares_lag1",
    "Master_mconsumosdolares_delta1",
    "Master_mconsumosdolares_lag2",
    "Master_mconsumosdolares_delta2",
    "Master_mconsumosdolares_ols6",
    "Master_mconsumosdolares_sma3",
    "Master_mconsumosdolares_sma6",
    "Master_mconsumosdolares_min6",
    "Master_mconsumosdolares_max6",
    "Master_mconsumosdolares_ema3",
    "Master_mconsumosdolares_ema6",
    "Master_mlimitecompra_lag1",
    "Master_mlimitecompra_delta1",
    "Master_mlimitecompra_lag2",
    "Master_mlimitecompra_delta2",
    "Master_mlimitecompra_ols6",
    "Master_mlimitecompra_sma3",
    "Master_mlimitecompra_sma6",
    "Master_mlimitecompra_min6",
    "Master_mlimitecompra_max6",
    "Master_mlimitecompra_ema3",
    "Master_mlimitecompra_ema6",
    "Master_madelantopesos_lag1",
    "Master_madelantopesos_delta1",
    "Master_madelantopesos_lag2",
    "Master_madelantopesos_delta2",
    "Master_madelantopesos_ols6",
    "Master_madelantopesos_sma3",
    "Master_madelantopesos_sma6",
    "Master_madelantopesos_min6",
    "Master_madelantopesos_max6",
    "Master_madelantopesos_ema3",
    "Master_madelantopesos_ema6",
    "Master_madelantodolares_lag1",
    "Master_madelantodolares_delta1",
    "Master_madelantodolares_lag2",
    "Master_madelantodolares_delta2",
    "Master_madelantodolares_ols6",
    "Master_madelantodolares_sma3",
    "Master_madelantodolares_sma6",
    "Master_madelantodolares_min6",
    "Master_madelantodolares_max6",
    "Master_madelantodolares_ema3",
    "Master_madelantodolares_ema6",
    "Master_fultimo_cierre_lag1",
    "Master_fultimo_cierre_delta1",
    "Master_fultimo_cierre_lag2",
    "Master_fultimo_cierre_delta2",
    "Master_fultimo_cierre_ols6",
    "Master_fultimo_cierre_sma3",
    "Master_fultimo_cierre_sma6",
    "Master_fultimo_cierre_min6",
    "Master_fultimo_cierre_max6",
    "Master_fultimo_cierre_ema3",
    "Master_fultimo_cierre_ema6",
    "Master_mpagado_lag1",
    "Master_mpagado_delta1",
    "Master_mpagado_lag2",
    "Master_mpagado_delta2",
    "Master_mpagado_ols6",
    "Master_mpagado_sma3",
    "Master_mpagado_sma6",
    "Master_mpagado_min6",
    "Master_mpagado_max6",
    "Master_mpagado_ema3",
    "Master_mpagado_ema6",
    "Master_mpagospesos_lag1",
    "Master_mpagospesos_delta1",
    "Master_mpagospesos_lag2",
    "Master_mpagospesos_delta2",
    "Master_mpagospesos_ols6",
    "Master_mpagospesos_sma3",
    "Master_mpagospesos_sma6",
    "Master_mpagospesos_min6",
    "Master_mpagospesos_max6",
    "Master_mpagospesos_ema3",
    "Master_mpagospesos_ema6",
    "Master_mpagosdolares_lag1",
    "Master_mpagosdolares_delta1",
    "Master_mpagosdolares_lag2",
    "Master_mpagosdolares_delta2",
    "Master_mpagosdolares_ols6",
    "Master_mpagosdolares_sma3",
    "Master_mpagosdolares_sma6",
    "Master_mpagosdolares_min6",
    "Master_mpagosdolares_max6",
    "Master_mpagosdolares_ema3",
    "Master_mpagosdolares_ema6",
    "Master_fechaalta_lag1",
    "Master_fechaalta_delta1",
    "Master_fechaalta_lag2",
    "Master_fechaalta_delta2",
    "Master_fechaalta_ols6",
    "Master_fechaalta_sma3",
    "Master_fechaalta_sma6",
    "Master_fechaalta_min6",
    "Master_fechaalta_max6",
    "Master_fechaalta_ema3",
    "Master_fechaalta_ema6",
    "Master_mconsumototal_lag1",
    "Master_mconsumototal_delta1",
    "Master_mconsumototal_lag2",
    "Master_mconsumototal_delta2",
    "Master_mconsumototal_ols6",
    "Master_mconsumototal_sma3",
    "Master_mconsumototal_sma6",
    "Master_mconsumototal_min6",
    "Master_mconsumototal_max6",
    "Master_mconsumototal_ema3",
    "Master_mconsumototal_ema6",
    "Master_cconsumos_lag1",
    "Master_cconsumos_delta1",
    "Master_cconsumos_lag2",
    "Master_cconsumos_delta2",
    "Master_cconsumos_ols6",
    "Master_cconsumos_sma3",
    "Master_cconsumos_sma6",
    "Master_cconsumos_min6",
    "Master_cconsumos_max6",
    "Master_cconsumos_ema3",
    "Master_cconsumos_ema6",
    "Master_cadelantosefectivo_lag1",
    "Master_cadelantosefectivo_delta1",
    "Master_cadelantosefectivo_lag2",
    "Master_cadelantosefectivo_delta2",
    "Master_cadelantosefectivo_ols6",
    "Master_cadelantosefectivo_sma3",
    "Master_cadelantosefectivo_sma6",
    "Master_cadelantosefectivo_min6",
    "Master_cadelantosefectivo_max6",
    "Master_cadelantosefectivo_ema3",
    "Master_cadelantosefectivo_ema6",
    "Master_mpagominimo_lag1",
    "Master_mpagominimo_delta1",
    "Master_mpagominimo_lag2",
    "Master_mpagominimo_delta2",
    "Master_mpagominimo_ols6",
    "Master_mpagominimo_sma3",
    "Master_mpagominimo_sma6",
    "Master_mpagominimo_min6",
    "Master_mpagominimo_max6",
    "Master_mpagominimo_ema3",
    "Master_mpagominimo_ema6",
    "Visa_delinquency_lag1",
    "Visa_delinquency_delta1",
    "Visa_delinquency_lag2",
    "Visa_delinquency_delta2",
    "Visa_delinquency_ols6",
    "Visa_delinquency_sma3",
    "Visa_delinquency_sma6",
    "Visa_delinquency_min6",
    "Visa_delinquency_max6",
    "Visa_delinquency_ema3",
    "Visa_delinquency_ema6",
    "Visa_status_lag1",
    "Visa_status_delta1",
    "Visa_status_lag2",
    "Visa_status_delta2",
    "Visa_status_ols6",
    "Visa_status_sma3",
    "Visa_status_sma6",
    "Visa_status_min6",
    "Visa_status_max6",
    "Visa_status_ema3",
    "Visa_status_ema6",
    "Visa_mfinanciacion_limite_lag1",
    "Visa_mfinanciacion_limite_delta1",
    "Visa_mfinanciacion_limite_lag2",
    "Visa_mfinanciacion_limite_delta2",
    "Visa_mfinanciacion_limite_ols6",
    "Visa_mfinanciacion_limite_sma3",
    "Visa_mfinanciacion_limite_sma6",
    "Visa_mfinanciacion_limite_min6",
    "Visa_mfinanciacion_limite_max6",
    "Visa_mfinanciacion_limite_ema3",
    "Visa_mfinanciacion_limite_ema6",
    "Visa_Fvencimiento_lag1",
    "Visa_Fvencimiento_delta1",
    "Visa_Fvencimiento_lag2",
    "Visa_Fvencimiento_delta2",
    "Visa_Fvencimiento_ols6",
    "Visa_Fvencimiento_sma3",
    "Visa_Fvencimiento_sma6",
    "Visa_Fvencimiento_min6",
    "Visa_Fvencimiento_max6",
    "Visa_Fvencimiento_ema3",
    "Visa_Fvencimiento_ema6",
    "Visa_Finiciomora_lag1",
    "Visa_Finiciomora_delta1",
    "Visa_Finiciomora_lag2",
    "Visa_Finiciomora_delta2",
    "Visa_Finiciomora_ols6",
    "Visa_Finiciomora_sma3",
    "Visa_Finiciomora_sma6",
    "Visa_Finiciomora_min6",
    "Visa_Finiciomora_max6",
    "Visa_Finiciomora_ema3",
    "Visa_Finiciomora_ema6",
    "Visa_msaldototal_lag1",
    "Visa_msaldototal_delta1",
    "Visa_msaldototal_lag2",
    "Visa_msaldototal_delta2",
    "Visa_msaldototal_ols6",
    "Visa_msaldototal_sma3",
    "Visa_msaldototal_sma6",
    "Visa_msaldototal_min6",
    "Visa_msaldototal_max6",
    "Visa_msaldototal_ema3",
    "Visa_msaldototal_ema6",
    "Visa_msaldopesos_lag1",
    "Visa_msaldopesos_delta1",
    "Visa_msaldopesos_lag2",
    "Visa_msaldopesos_delta2",
    "Visa_msaldopesos_ols6",
    "Visa_msaldopesos_sma3",
    "Visa_msaldopesos_sma6",
    "Visa_msaldopesos_min6",
    "Visa_msaldopesos_max6",
    "Visa_msaldopesos_ema3",
    "Visa_msaldopesos_ema6",
    "Visa_msaldodolares_lag1",
    "Visa_msaldodolares_delta1",
    "Visa_msaldodolares_lag2",
    "Visa_msaldodolares_delta2",
    "Visa_msaldodolares_ols6",
    "Visa_msaldodolares_sma3",
    "Visa_msaldodolares_sma6",
    "Visa_msaldodolares_min6",
    "Visa_msaldodolares_max6",
    "Visa_msaldodolares_ema3",
    "Visa_msaldodolares_ema6",
    "Visa_mconsumospesos_lag1",
    "Visa_mconsumospesos_delta1",
    "Visa_mconsumospesos_lag2",
    "Visa_mconsumospesos_delta2",
    "Visa_mconsumospesos_ols6",
    "Visa_mconsumospesos_sma3",
    "Visa_mconsumospesos_sma6",
    "Visa_mconsumospesos_min6",
    "Visa_mconsumospesos_max6",
    "Visa_mconsumospesos_ema3",
    "Visa_mconsumospesos_ema6",
    "Visa_mconsumosdolares_lag1",
    "Visa_mconsumosdolares_delta1",
    "Visa_mconsumosdolares_lag2",
    "Visa_mconsumosdolares_delta2",
    "Visa_mconsumosdolares_ols6",
    "Visa_mconsumosdolares_sma3",
    "Visa_mconsumosdolares_sma6",
    "Visa_mconsumosdolares_min6",
    "Visa_mconsumosdolares_max6",
    "Visa_mconsumosdolares_ema3",
    "Visa_mconsumosdolares_ema6",
    "Visa_mlimitecompra_lag1",
    "Visa_mlimitecompra_delta1",
    "Visa_mlimitecompra_lag2",
    "Visa_mlimitecompra_delta2",
    "Visa_mlimitecompra_ols6",
    "Visa_mlimitecompra_sma3",
    "Visa_mlimitecompra_sma6",
    "Visa_mlimitecompra_min6",
    "Visa_mlimitecompra_max6",
    "Visa_mlimitecompra_ema3",
    "Visa_mlimitecompra_ema6",
    "Visa_madelantopesos_lag1",
    "Visa_madelantopesos_delta1",
    "Visa_madelantopesos_lag2",
    "Visa_madelantopesos_delta2",
    "Visa_madelantopesos_ols6",
    "Visa_madelantopesos_sma3",
    "Visa_madelantopesos_sma6",
    "Visa_madelantopesos_min6",
    "Visa_madelantopesos_max6",
    "Visa_madelantopesos_ema3",
    "Visa_madelantopesos_ema6",
    "Visa_madelantodolares_lag1",
    "Visa_madelantodolares_delta1",
    "Visa_madelantodolares_lag2",
    "Visa_madelantodolares_delta2",
    "Visa_madelantodolares_ols6",
    "Visa_madelantodolares_sma3",
    "Visa_madelantodolares_sma6",
    "Visa_madelantodolares_min6",
    "Visa_madelantodolares_max6",
    "Visa_madelantodolares_ema3",
    "Visa_madelantodolares_ema6",
    "Visa_fultimo_cierre_lag1",
    "Visa_fultimo_cierre_delta1",
    "Visa_fultimo_cierre_lag2",
    "Visa_fultimo_cierre_delta2",
    "Visa_fultimo_cierre_ols6",
    "Visa_fultimo_cierre_sma3",
    "Visa_fultimo_cierre_sma6",
    "Visa_fultimo_cierre_min6",
    "Visa_fultimo_cierre_max6",
    "Visa_fultimo_cierre_ema3",
    "Visa_fultimo_cierre_ema6",
    "Visa_mpagado_lag1",
    "Visa_mpagado_delta1",
    "Visa_mpagado_lag2",
    "Visa_mpagado_delta2",
    "Visa_mpagado_ols6",
    "Visa_mpagado_sma3",
    "Visa_mpagado_sma6",
    "Visa_mpagado_min6",
    "Visa_mpagado_max6",
    "Visa_mpagado_ema3",
    "Visa_mpagado_ema6",
    "Visa_mpagospesos_lag1",
    "Visa_mpagospesos_delta1",
    "Visa_mpagospesos_lag2",
    "Visa_mpagospesos_delta2",
    "Visa_mpagospesos_ols6",
    "Visa_mpagospesos_sma3",
    "Visa_mpagospesos_sma6",
    "Visa_mpagospesos_min6",
    "Visa_mpagospesos_max6",
    "Visa_mpagospesos_ema3",
    "Visa_mpagospesos_ema6",
    "Visa_mpagosdolares_lag1",
    "Visa_mpagosdolares_delta1",
    "Visa_mpagosdolares_lag2",
    "Visa_mpagosdolares_delta2",
    "Visa_mpagosdolares_ols6",
    "Visa_mpagosdolares_sma3",
    "Visa_mpagosdolares_sma6",
    "Visa_mpagosdolares_min6",
    "Visa_mpagosdolares_max6",
    "Visa_mpagosdolares_ema3",
    "Visa_mpagosdolares_ema6",
    "Visa_fechaalta_lag1",
    "Visa_fechaalta_delta1",
    "Visa_fechaalta_lag2",
    "Visa_fechaalta_delta2",
    "Visa_fechaalta_ols6",
    "Visa_fechaalta_sma3",
    "Visa_fechaalta_sma6",
    "Visa_fechaalta_min6",
    "Visa_fechaalta_max6",
    "Visa_fechaalta_ema3",
    "Visa_fechaalta_ema6",
    "Visa_mconsumototal_lag1",
    "Visa_mconsumototal_delta1",
    "Visa_mconsumototal_lag2",
    "Visa_mconsumototal_delta2",
    "Visa_mconsumototal_ols6",
    "Visa_mconsumototal_sma3",
    "Visa_mconsumototal_sma6",
    "Visa_mconsumototal_min6",
    "Visa_mconsumototal_max6",
    "Visa_mconsumototal_ema3",
    "Visa_mconsumototal_ema6",
    "Visa_cconsumos_lag1",
    "Visa_cconsumos_delta1",
    "Visa_cconsumos_lag2",
    "Visa_cconsumos_delta2",
    "Visa_cconsumos_ols6",
    "Visa_cconsumos_sma3",
    "Visa_cconsumos_sma6",
    "Visa_cconsumos_min6",
    "Visa_cconsumos_max6",
    "Visa_cconsumos_ema3",
    "Visa_cconsumos_ema6",
    "Visa_cadelantosefectivo_lag1",
    "Visa_cadelantosefectivo_delta1",
    "Visa_cadelantosefectivo_lag2",
    "Visa_cadelantosefectivo_delta2",
    "Visa_cadelantosefectivo_ols6",
    "Visa_cadelantosefectivo_sma3",
    "Visa_cadelantosefectivo_sma6",
    "Visa_cadelantosefectivo_min6",
    "Visa_cadelantosefectivo_max6",
    "Visa_cadelantosefectivo_ema3",
    "Visa_cadelantosefectivo_ema6",
    "Visa_mpagominimo_lag1",
    "Visa_mpagominimo_delta1",
    "Visa_mpagominimo_lag2",
    "Visa_mpagominimo_delta2",
    "Visa_mpagominimo_ols6",
    "Visa_mpagominimo_sma3",
    "Visa_mpagominimo_sma6",
    "Visa_mpagominimo_min6",
    "Visa_mpagominimo_max6",
    "Visa_mpagominimo_ema3",
    "Visa_mpagominimo_ema6",
    "fe67_mcuentas_saldo_median_prior6",
    "fe67_mcuentas_saldo_mad_prior6",
    "fe67_mcuentas_saldo_robust_deviation6",
    "fe67_mpayroll_median_prior6",
    "fe67_mpayroll_mad_prior6",
    "fe67_mpayroll_robust_deviation6"
  ],
  "feature_code": {
    "t7_month_index": "t7_month_index <- function (month) \n{\n    stopifnot(all(month%%100L %in% 1:12))\n    (month%/%100L) * 12L + month%%100L - 1L\n}",
    "t7_month_value": "t7_month_value <- function (index) \n(index%/%12L) * 100L + index%%12L + 1L",
    "t7_features": "t7_features <- function (panel, history_columns, history_panel = panel) \n{\n    keys <- c(\"numero_de_cliente\", \"foto_mes\", \"clase_ternaria\")\n    stopifnot(all(c(keys, history_columns) %in% names(panel)), !anyDuplicated(panel, by = c(\"numero_de_cliente\", \"foto_mes\")), !any(history_columns %in% keys))\n    snapshot <- setdiff(names(panel), keys)\n    stopifnot(all(vapply(panel[, snapshot, with = FALSE], is.numeric, logical(1))))\n    out <- data.table::copy(panel)\n    stopifnot(!anyDuplicated(history_panel, by = c(\"numero_de_cliente\", \"foto_mes\")))\n    index <- history_panel[, .(numero_de_cliente, month_index = t7_month_index(foto_mes), row_id = .I)]\n    data.table::setkey(index, numero_de_cliente, month_index)\n    matches <- lapply(1:6, function(lag) {\n        query <- panel[, .(numero_de_cliente, month_index = t7_month_index(foto_mes) - lag)]\n        index[query, on = .(numero_de_cliente, month_index), row_id]\n    })\n    for (lag in 1:6) out[, `:=`((paste0(\"history_month\", lag, \"_present\")), as.integer(!is.na(matches[[lag]])))]\n    for (column in history_columns) {\n        prior <- matrix(vapply(matches, function(rows) as.double(history_panel[[column]][rows]), numeric(nrow(panel))), nrow = nrow(panel), ncol = 6L)\n        for (lag in c(1L, 2L, 3L, 6L)) out[, `:=`((paste0(column, \"_lag\", lag)), prior[, lag])]\n        for (lag in c(1L, 3L)) out[, `:=`((paste0(column, \"_delta\", lag)), as.double(panel[[column]]) - prior[, lag])]\n        for (window in c(3L, 6L)) {\n            values <- prior[, seq_len(window), drop = FALSE]\n            count <- rowSums(!is.na(values))\n            mean <- rowMeans(values, na.rm = TRUE)\n            mean[count == 0L] <- NA_real_\n            out[, `:=`((paste0(column, \"_prior\", window, \"_mean\")), mean)]\n            out[, `:=`((paste0(column, \"_prior\", window, \"_observed\")), count)]\n        }\n    }\n    list(data = out, snapshot = snapshot, history = setdiff(names(out), keys))\n}",
    "o54_history_kernel": "o54_history_kernel <- function () \nlist(window = \"Rcpp::NumericMatrix ht5954_window_summary(Rcpp::NumericVector cur, Rcpp::NumericMatrix prior) {\\n    int n=cur.size(); Rcpp::NumericMatrix out(n,9);\\n    for(int i=0;i<n;i++) {\\n      double a=prior(i,0),b=prior(i,1);\\n      out(i,0)=a;out(i,1)=cur[i]-a;out(i,2)=b;out(i,3)=cur[i]-b;\\n      double sx=0,sy=0,sxy=0,sxx=0,lo=0,hi=0,s3=0;int k=0,k3=0;\\n      for(int lag=0;lag<6;lag++) {\\n        double v=lag==0?cur[i]:prior(i,lag-1);\\n        if(!R_finite(v))continue;\\n        double x=-lag;sx+=x;sy+=v;sxy+=x*v;sxx+=x*x;\\n        if(k==0){lo=v;hi=v;}else{if(v<lo)lo=v;if(v>hi)hi=v;}\\n        k++;if(lag<3){s3+=v;k3++;}\\n      }\\n      double den=k*sxx-sx*sx;\\n      out(i,4)=(k>=2&&den>0)?(k*sxy-sx*sy)/den:NA_REAL;\\n      out(i,5)=k3>=2?s3/k3:NA_REAL;out(i,6)=k>=2?sy/k:NA_REAL;\\n      out(i,7)=k>=2?lo:NA_REAL;out(i,8)=k>=2?hi:NA_REAL;\\n    } return out;\\n  }\", \n    ema = \"Rcpp::NumericMatrix ht5954_calendar_ema(Rcpp::NumericVector value,Rcpp::NumericVector id,Rcpp::IntegerVector month) {\\n    int n=value.size();Rcpp::NumericMatrix out(n,2);double num[2]={0,0},den[2]={0,0};int known=0;\\n    double decay[2]={0.5,5.0/7.0};\\n    for(int i=0;i<n;i++) {\\n      bool fresh=i==0||id[i]!=id[i-1];int gap=fresh?0:month[i]-month[i-1];\\n      if(!fresh&&gap<=0)Rcpp::stop(\\\"EMA requires unique increasing customer/month keys\\\");\\n      if(fresh){known=0;num[0]=num[1]=den[0]=den[1]=0;}\\n      bool valid=R_finite(value[i]);if(valid)known++;\\n      for(int j=0;j<2;j++) {\\n        double factor=std::pow(decay[j],gap);num[j]*=factor;den[j]*=factor;\\n        if(valid){num[j]+=value[i];den[j]+=1;}\\n        out(i,j)=known>=2&&den[j]>0?num[j]/den[j]:NA_REAL;\\n      }\\n    }return out;\\n  }\")",
    "o54_compile_history": "o54_compile_history <- function () \n{\n    code <- o54_history_kernel()\n    signature <- digest::digest(code, algo = \"sha256\")\n    if (!identical(get0(\".ht5954_kernel_signature\", envir = .GlobalEnv), signature)) {\n        Rcpp::cppFunction(code$window, env = .GlobalEnv)\n        Rcpp::cppFunction(code$ema, env = .GlobalEnv)\n        assign(\".ht5954_kernel_signature\", signature, envir = .GlobalEnv)\n    }\n}",
    "o54_full_history_extra": "o54_full_history_extra <- function (d, panel, index, fields) \n{\n    stopifnot(!\"clase_ternaria\" %in% names(panel), !any(c(\"numero_de_cliente\", \"foto_mes\", \"clase_ternaria\") %in% fields))\n    positions <- lapply(1:5, function(lag) index[d[, .(numero_de_cliente, month_index = t7_month_index(foto_mes) - lag)], on = .(numero_de_cliente, month_index), row_id])\n    current_pos <- index[d[, .(numero_de_cliente, month_index = t7_month_index(foto_mes))], on = .(numero_de_cliente, month_index), row_id]\n    stopifnot(!anyNA(current_pos))\n    out <- data.table::data.table(numero_de_cliente = d$numero_de_cliente)\n    for (field in fields) {\n        previous <- matrix(vapply(positions, function(i) as.double(panel[[field]][i]), numeric(nrow(d))), nrow = nrow(d), ncol = 5L)\n        values <- ht5954_window_summary(as.double(d[[field]]), previous)\n        names <- paste0(field, c(\"_lag1\", \"_delta1\", \"_lag2\", \"_delta2\", \"_ols6\", \"_sma3\", \"_sma6\", \"_min6\", \"_max6\"))\n        for (j in seq_along(names)) if (!names[j] %in% colnames(d)) \n            out[, `:=`((names[j]), values[, j])]\n        out[, `:=`((paste0(field, \"_ema3\")), panel[[paste0(\".ema3_\", field)]][current_pos])]\n        out[, `:=`((paste0(field, \"_ema6\")), panel[[paste0(\".ema6_\", field)]][current_pos])]\n    }\n    out\n}",
    "f67_values": "f67_values <- function (panel, rows, index) \n{\n    d <- panel[rows]\n    out <- d[, .(numero_de_cliente, foto_mes)]\n    at <- lapply(1:6, function(k) {\n        query <- data.table::data.table(id = d$numero_de_cliente, month = d$.month - k)\n        index[query, on = .(id, month), row]\n    })\n    for (f in c(\"mcuentas_saldo\", \"mpayroll\")) {\n        x <- matrix(vapply(at, function(i) as.double(panel[[f]][i]), numeric(nrow(d))), nrow = nrow(d), ncol = 6L)\n        x[!is.finite(x)] <- NA_real_\n        known <- is.finite(x)\n        n <- rowSums(known)\n        p3 <- x[, 1:3, drop = FALSE]\n        k3 <- is.finite(p3)\n        n3 <- rowSums(k3)\n        low <- matrixStats::rowMins(x, na.rm = TRUE)\n        high <- matrixStats::rowMaxs(x, na.rm = TRUE)\n        center <- matrixStats::rowMedians(x, na.rm = TRUE)\n        scale <- matrixStats::rowMedians(abs(x - center), na.rm = TRUE)\n        for (name in c(\"low\", \"high\", \"center\", \"scale\")) {\n            v <- get(name)\n            v[n < 3L] <- NA_real_\n            assign(name, v)\n        }\n        time <- -seq_len(3L)\n        y <- p3\n        y[!k3] <- 0\n        sx <- rowSums(sweep(k3, 2L, time, `*`))\n        sy <- rowSums(y)\n        sxx <- rowSums(sweep(k3, 2L, time^2, `*`))\n        sxy <- rowSums(sweep(y, 2L, time, `*`))\n        slope <- (n3 * sxy - sx * sy)/(n3 * sxx - sx^2)\n        projection <- (sy - slope * sx)/pmax(n3, 1L)\n        bad <- n3 < 2L | !is.finite(slope)\n        slope[bad] <- NA_real_\n        projection[bad] <- NA_real_\n        relative <- function(value, ref) pmax(-10, pmin(10, (value - ref)/pmax(abs(ref), 1)))\n        values <- list(min_prior6 = low, max_prior6 = high, range_prior6 = high - low, slope_prior3 = slope, relative_trend3 = relative(as.double(d[[f]]), projection), median_prior6 = center, mad_prior6 = scale, robust_deviation6 = pmax(-10, pmin(10, (as.double(d[[f]]) - center)/pmax(scale, 1))), known_prior3 = n3, known_prior6 = n)\n        for (name in names(values)) data.table::set(out, j = paste0(\"fe67_\", f, \"_\", name), value = values[[name]])\n    }\n    stopifnot(!any(vapply(out, function(v) any(is.infinite(v) | is.nan(v)), logical(1))))\n    out\n}",
    "f67_views": "f67_views <- function (base) \n{\n    fields <- c(\"mcuentas_saldo\", \"mpayroll\")\n    cols <- function(suffix) unlist(lapply(fields, function(f) paste0(\"fe67_\", f, \"_\", suffix)))\n    ext <- cols(c(\"min_prior6\", \"max_prior6\", \"range_prior6\"))\n    slope <- cols(c(\"slope_prior3\", \"relative_trend3\"))\n    mad <- cols(c(\"median_prior6\", \"mad_prior6\", \"robust_deviation6\"))\n    list(E_extremes = c(base, ext), S_slope = c(base, slope), M_mad = c(base, mad), U_union = c(base, ext, slope, mad))\n}",
    "r66_index": "r66_index <- function (month) \n{\n    stopifnot(all(month%%100L %in% 1:12))\n    month%/%100L * 12L + month%%100L - 1L\n}",
    "r66_month": "r66_month <- function (index) \nindex%/%12L * 100L + index%%12L + 1L",
    "r66_block": "r66_block <- function (month) \nmonth >= 202003L & month <= 202103L",
    "r66_eligible": "r66_eligible <- function (origin) \nr66_month(seq.int(r66_index(201901L), r66_index(origin) - 2L))",
    "r66_weights": "r66_weights <- function (month) \n{\n    raw <- ifelse(r66_block(month), 0.25, 1)\n    raw/mean(raw)\n}"
  },
  "params": {
    "BASE_333803": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "multi_logloss",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 333803,
      "feature_fraction_seed": 333803,
      "bagging_seed": 333803,
      "data_random_seed": 333803,
      "bagging_fraction": 0.702,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 28.7807,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.07591,
      "num_leaves": 48,
      "min_data_in_leaf": 914,
      "feature_fraction": 0.815,
      "num_class": 3
    },
    "BASE_333903": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "multi_logloss",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 333903,
      "feature_fraction_seed": 333903,
      "bagging_seed": 333903,
      "data_random_seed": 333903,
      "bagging_fraction": 0.702,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 28.7807,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.07591,
      "num_leaves": 48,
      "min_data_in_leaf": 914,
      "feature_fraction": 0.815,
      "num_class": 3
    },
    "BASE_334003": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "multi_logloss",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 334003,
      "feature_fraction_seed": 334003,
      "bagging_seed": 334003,
      "data_random_seed": 334003,
      "bagging_fraction": 0.702,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 28.7807,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.07591,
      "num_leaves": 48,
      "min_data_in_leaf": 914,
      "feature_fraction": 0.815,
      "num_class": 3
    },
    "MAD_333803": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "None",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 333803,
      "feature_fraction_seed": 333803,
      "bagging_seed": 333803,
      "data_random_seed": 333803,
      "bagging_fraction": 0.8541457988583311,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 8.809972383953268,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.04,
      "num_leaves": 71,
      "min_data_in_leaf": 201,
      "feature_fraction": 0.6683411362336381,
      "num_class": 3
    },
    "MAD_334003": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "None",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 334003,
      "feature_fraction_seed": 334003,
      "bagging_seed": 334003,
      "data_random_seed": 334003,
      "bagging_fraction": 0.8541457988583311,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 8.809972383953268,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.04,
      "num_leaves": 71,
      "min_data_in_leaf": 201,
      "feature_fraction": 0.6683411362336381,
      "num_class": 3
    },
    "MAD_333903": {
      "boosting": "gbdt",
      "objective": "multiclass",
      "metric": "None",
      "first_metric_only": true,
      "max_depth": -1,
      "max_bin": 31,
      "feature_pre_filter": false,
      "deterministic": true,
      "force_col_wise": true,
      "num_threads": 4,
      "verbosity": -1,
      "seed": 333903,
      "feature_fraction_seed": 333903,
      "bagging_seed": 333903,
      "data_random_seed": 333903,
      "bagging_fraction": 0.8541457988583311,
      "bagging_freq": 1,
      "lambda_l1": 0,
      "lambda_l2": 8.809972383953268,
      "min_gain_to_split": 0,
      "min_sum_hessian_in_leaf": 0.001,
      "is_unbalance": false,
      "scale_pos_weight": 1,
      "learning_rate": 0.04,
      "num_leaves": 71,
      "min_data_in_leaf": 201,
      "feature_fraction": 0.6683411362336381,
      "num_class": 3
    }
  },
  "construction": {
    "333803": {
      "max_bin": 31,
      "feature_pre_filter": false,
      "data_random_seed": 333803,
      "num_threads": 4
    },
    "333903": {
      "max_bin": 31,
      "feature_pre_filter": false,
      "data_random_seed": 333903,
      "num_threads": 4
    },
    "334003": {
      "max_bin": 31,
      "feature_pre_filter": false,
      "data_random_seed": 334003,
      "num_threads": 4
    }
  },
  "rounds": {
    "BASE": 192,
    "MAD": 384
  },
  "blend": {
    "BASE": 0.75,
    "MAD": 0.25,
    "contacts": 8250
  },
  "expected_models": {
    "BASE_333803": {
      "model_sha": "05beb043c25fed5ceb17aecaf4a6cec67bb2f0cfcfcdf78b2f59b7c05809209d",
      "original_path": "/content/buckets/b1/exp/HT5966/training_regimes_v1/models/2ac5c09fe46acef6ddbc4c4e3311efe74ba0dac59804e320fec068d29c45b3fb.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "3e9ed25078cfd5108a5fd7764c1cce282115d61cd4dbbb8ae560baa5eb12424b",
      "prediction_hash": "9b7cbd31583507139105b83b5fb4c53aff14f0ff69ba1596c972e826a7667918"
    },
    "BASE_333903": {
      "model_sha": "f9c84a0783437323f42ef1b4efe0c2c32dbff6055a1aab6648adbda4c69593cc",
      "original_path": "/content/buckets/b1/exp/HT5966/training_regimes_v1/models/3689667b0885fd3e3f163a3781aeee5eb23dbdc54f354f609df48936698d7880.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "3e9ed25078cfd5108a5fd7764c1cce282115d61cd4dbbb8ae560baa5eb12424b",
      "prediction_hash": "ab423751eb41c0b604f39d3c76ec042daff671e299e7c048dd54d8ab19df4b14"
    },
    "BASE_334003": {
      "model_sha": "5d0de295cb575ea4fc547c4d81596d2b2d4a6e1ee7ed38dd6ed314dfab466e0d",
      "original_path": "/content/buckets/b1/exp/HT5966/training_regimes_v1/models/6fe21e1a245b4eaca9534a13a9ac744913490724fc0b5f3de28d193f5948bac8.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "3e9ed25078cfd5108a5fd7764c1cce282115d61cd4dbbb8ae560baa5eb12424b",
      "prediction_hash": "7df758eb9faa042e38004774c6a0705b733871283a9677e7ca05f7534687b95c"
    },
    "MAD_333803": {
      "model_sha": "9fb608342b9da9c336eabdeca87387bcd4bbda51fce42179e5853fe5242af170",
      "original_path": "/content/buckets/b1/exp/HT5973/rich_hpo_v1/models/70536b2cecafd8b69435362889186d183b481cd494c1468e702a7fe8d4ae1f7e.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "03caec0c248806015c447d041cd895a2d728b82cc0ecb6859a463c113dab7be5",
      "prediction_hash": "d9e02be020bf93eb15dfe23da1848bb32b87c9287524c23574956ca611562677"
    },
    "MAD_334003": {
      "model_sha": "fd5ec7f4152a91595df8c905f4a498ac7e423a90b18d408892e719eead95095a",
      "original_path": "/content/buckets/b1/exp/HT5973/rich_hpo_v1/models/93fe9b28606e7f15439127f00d7fed66c53cd10e07a7f00547766f08227913d4.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "03caec0c248806015c447d041cd895a2d728b82cc0ecb6859a463c113dab7be5",
      "prediction_hash": "4d2616a8cc667eece2359512b8ef0c23cc7ab03dc12e32b30120f4c3c670c084"
    },
    "MAD_333903": {
      "model_sha": "43515472ee1003d5344ac9e27f8fba071244fa3fea8d0490857a5bd00344f1a9",
      "original_path": "/content/buckets/b1/exp/HT5973/rich_hpo_v1/models/dec5dcfec04fec1aa8f30daff5a15b25d395222d7f7cb81a9e2e5b7810f29b21.rds",
      "rows": 4547505,
      "row_hash": "943c60376d07286d6b0fa3a480a76b85157ffeaf789cec9fa76cd1cb31f6d828",
      "weight_hash": "af16a86a7fd109fa0c1df3b5fd9cde906a0ec1fa2d25b53090588b3fbc356b3c",
      "columns_hash": "03caec0c248806015c447d041cd895a2d728b82cc0ecb6859a463c113dab7be5",
      "prediction_hash": "b169e5284ed4fd484c978b766df805c671a5164a5b8f9cf11e95f76a26911641"
    }
  },
  "R": "R version 4.6.1 (2026-06-24)",
  "native_sha256": "bfdf4459d77121213d7f5187f1825d950a549ab42e54c976dfdd147044b3d6fb",
  "package_versions": {
    "lightgbm": "4.7.0.99",
    "data.table": "1.18.6.1",
    "digest": "0.6.39",
    "jsonlite": "2.0.0",
    "matrixStats": "1.5.0",
    "Rcpp": "1.1.2",
    "IRkernel": "1.3.2.9000"
  },
  "lightgbm_description": {
    "Version": "4.7.0.99",
    "URL": "https://github.com/lightgbm-org/LightGBM",
    "Packaged": "2026-09-15 04:24:05 UTC; ds",
    "Built": "R 4.6.1; x86_64-pc-linux-gnu; 2026-09-15 04:24:10 UTC; unix"
  },
  "compiler": null,
  "platform": "x86_64-pc-linux-gnu"
})---", simplifyVector=TRUE)
golden <- jsonlite::fromJSON(r"---({
  "state": "reference_fingerprints_complete",
  "R": "R version 4.6.1 (2026-06-24)",
  "columns": [
    "active_quarter",
    "cliente_vip",
    "internet",
    "cliente_edad",
    "cliente_antiguedad",
    "mrentabilidad",
    "mrentabilidad_annual",
    "mcomisiones",
    "mactivos_margen",
    "mpasivos_margen",
    "cproductos",
    "tcuentas",
    "ccuenta_corriente",
    "mcuenta_corriente_adicional",
    "mcuenta_corriente",
    "ccaja_ahorro",
    "mcaja_ahorro",
    "mcaja_ahorro_adicional",
    "mcaja_ahorro_dolares",
    "cdescubierto_preacordado",
    "mcuentas_saldo",
    "ctarjeta_debito",
    "ctarjeta_debito_transacciones",
    "mautoservicio",
    "ctarjeta_visa",
    "ctarjeta_visa_transacciones",
    "mtarjeta_visa_consumo",
    "ctarjeta_master",
    "ctarjeta_master_transacciones",
    "mtarjeta_master_consumo",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_prendarios",
    "mprestamos_prendarios",
    "cprestamos_hipotecarios",
    "mprestamos_hipotecarios",
    "cplazo_fijo",
    "mplazo_fijo_dolares",
    "mplazo_fijo_pesos",
    "cinversion1",
    "minversion1_pesos",
    "minversion1_dolares",
    "cinversion2",
    "minversion2",
    "cseguro_vida",
    "cseguro_auto",
    "cseguro_vivienda",
    "cseguro_accidentes_personales",
    "ccaja_seguridad",
    "cpayroll_trx",
    "mpayroll",
    "mpayroll2",
    "cpayroll2_trx",
    "ccuenta_debitos_automaticos",
    "mcuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos",
    "mttarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos",
    "mttarjeta_master_debitos_automaticos",
    "cpagodeservicios",
    "mpagodeservicios",
    "cpagomiscuentas",
    "mpagomiscuentas",
    "ccajeros_propios_descuentos",
    "mcajeros_propios_descuentos",
    "ctarjeta_visa_descuentos",
    "mtarjeta_visa_descuentos",
    "ctarjeta_master_descuentos",
    "mtarjeta_master_descuentos",
    "ccomisiones_mantenimiento",
    "mcomisiones_mantenimiento",
    "ccomisiones_otras",
    "mcomisiones_otras",
    "cforex",
    "cforex_buy",
    "mforex_buy",
    "cforex_sell",
    "mforex_sell",
    "ctransferencias_recibidas",
    "mtransferencias_recibidas",
    "ctransferencias_emitidas",
    "mtransferencias_emitidas",
    "cextraccion_autoservicio",
    "mextraccion_autoservicio",
    "ccheques_depositados",
    "mcheques_depositados",
    "ccheques_emitidos",
    "mcheques_emitidos",
    "ccheques_depositados_rechazados",
    "mcheques_depositados_rechazados",
    "ccheques_emitidos_rechazados",
    "mcheques_emitidos_rechazados",
    "tcallcenter",
    "ccallcenter_transacciones",
    "thomebanking",
    "chomebanking_transacciones",
    "ccajas_transacciones",
    "ccajas_consultas",
    "ccajas_depositos",
    "ccajas_extracciones",
    "ccajas_otras",
    "catm_trx",
    "matm",
    "catm_trx_other",
    "matm_other",
    "ctrx_quarter",
    "tmobile_app",
    "cmobile_app_trx",
    "Master_delinquency",
    "Master_status",
    "Master_mfinanciacion_limite",
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_msaldototal",
    "Master_msaldopesos",
    "Master_msaldodolares",
    "Master_mconsumospesos",
    "Master_mconsumosdolares",
    "Master_mlimitecompra",
    "Master_madelantopesos",
    "Master_madelantodolares",
    "Master_fultimo_cierre",
    "Master_mpagado",
    "Master_mpagospesos",
    "Master_mpagosdolares",
    "Master_fechaalta",
    "Master_mconsumototal",
    "Master_cconsumos",
    "Master_cadelantosefectivo",
    "Master_mpagominimo",
    "Visa_delinquency",
    "Visa_status",
    "Visa_mfinanciacion_limite",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_msaldototal",
    "Visa_msaldopesos",
    "Visa_msaldodolares",
    "Visa_mconsumospesos",
    "Visa_mconsumosdolares",
    "Visa_mlimitecompra",
    "Visa_madelantopesos",
    "Visa_madelantodolares",
    "Visa_fultimo_cierre",
    "Visa_mpagado",
    "Visa_mpagospesos",
    "Visa_mpagosdolares",
    "Visa_fechaalta",
    "Visa_mconsumototal",
    "Visa_cconsumos",
    "Visa_cadelantosefectivo",
    "Visa_mpagominimo",
    "history_month1_present",
    "history_month2_present",
    "history_month3_present",
    "history_month4_present",
    "history_month5_present",
    "history_month6_present",
    "mcuentas_saldo_lag1",
    "mcuentas_saldo_lag2",
    "mcuentas_saldo_lag3",
    "mcuentas_saldo_lag6",
    "mcuentas_saldo_delta1",
    "mcuentas_saldo_delta3",
    "mcuentas_saldo_prior3_mean",
    "mcuentas_saldo_prior3_observed",
    "mcuentas_saldo_prior6_mean",
    "mcuentas_saldo_prior6_observed",
    "mcuenta_corriente_lag1",
    "mcuenta_corriente_lag2",
    "mcuenta_corriente_lag3",
    "mcuenta_corriente_lag6",
    "mcuenta_corriente_delta1",
    "mcuenta_corriente_delta3",
    "mcuenta_corriente_prior3_mean",
    "mcuenta_corriente_prior3_observed",
    "mcuenta_corriente_prior6_mean",
    "mcuenta_corriente_prior6_observed",
    "mcaja_ahorro_lag1",
    "mcaja_ahorro_lag2",
    "mcaja_ahorro_lag3",
    "mcaja_ahorro_lag6",
    "mcaja_ahorro_delta1",
    "mcaja_ahorro_delta3",
    "mcaja_ahorro_prior3_mean",
    "mcaja_ahorro_prior3_observed",
    "mcaja_ahorro_prior6_mean",
    "mcaja_ahorro_prior6_observed",
    "mpayroll_lag1",
    "mpayroll_lag2",
    "mpayroll_lag3",
    "mpayroll_lag6",
    "mpayroll_delta1",
    "mpayroll_delta3",
    "mpayroll_prior3_mean",
    "mpayroll_prior3_observed",
    "mpayroll_prior6_mean",
    "mpayroll_prior6_observed",
    "mtarjeta_visa_consumo_lag1",
    "mtarjeta_visa_consumo_lag2",
    "mtarjeta_visa_consumo_lag3",
    "mtarjeta_visa_consumo_lag6",
    "mtarjeta_visa_consumo_delta1",
    "mtarjeta_visa_consumo_delta3",
    "mtarjeta_visa_consumo_prior3_mean",
    "mtarjeta_visa_consumo_prior3_observed",
    "mtarjeta_visa_consumo_prior6_mean",
    "mtarjeta_visa_consumo_prior6_observed",
    "mtarjeta_master_consumo_lag1",
    "mtarjeta_master_consumo_lag2",
    "mtarjeta_master_consumo_lag3",
    "mtarjeta_master_consumo_lag6",
    "mtarjeta_master_consumo_delta1",
    "mtarjeta_master_consumo_delta3",
    "mtarjeta_master_consumo_prior3_mean",
    "mtarjeta_master_consumo_prior3_observed",
    "mtarjeta_master_consumo_prior6_mean",
    "mtarjeta_master_consumo_prior6_observed",
    "ctrx_quarter_lag1",
    "ctrx_quarter_lag2",
    "ctrx_quarter_lag3",
    "ctrx_quarter_lag6",
    "ctrx_quarter_delta1",
    "ctrx_quarter_delta3",
    "ctrx_quarter_prior3_mean",
    "ctrx_quarter_prior3_observed",
    "ctrx_quarter_prior6_mean",
    "ctrx_quarter_prior6_observed",
    "ctarjeta_debito_transacciones_lag1",
    "ctarjeta_debito_transacciones_lag2",
    "ctarjeta_debito_transacciones_lag3",
    "ctarjeta_debito_transacciones_lag6",
    "ctarjeta_debito_transacciones_delta1",
    "ctarjeta_debito_transacciones_delta3",
    "ctarjeta_debito_transacciones_prior3_mean",
    "ctarjeta_debito_transacciones_prior3_observed",
    "ctarjeta_debito_transacciones_prior6_mean",
    "ctarjeta_debito_transacciones_prior6_observed",
    "mtransferencias_recibidas_lag1",
    "mtransferencias_recibidas_lag2",
    "mtransferencias_recibidas_lag3",
    "mtransferencias_recibidas_lag6",
    "mtransferencias_recibidas_delta1",
    "mtransferencias_recibidas_delta3",
    "mtransferencias_recibidas_prior3_mean",
    "mtransferencias_recibidas_prior3_observed",
    "mtransferencias_recibidas_prior6_mean",
    "mtransferencias_recibidas_prior6_observed",
    "mtransferencias_emitidas_lag1",
    "mtransferencias_emitidas_lag2",
    "mtransferencias_emitidas_lag3",
    "mtransferencias_emitidas_lag6",
    "mtransferencias_emitidas_delta1",
    "mtransferencias_emitidas_delta3",
    "mtransferencias_emitidas_prior3_mean",
    "mtransferencias_emitidas_prior3_observed",
    "mtransferencias_emitidas_prior6_mean",
    "mtransferencias_emitidas_prior6_observed",
    "mprestamos_personales_lag1",
    "mprestamos_personales_lag2",
    "mprestamos_personales_lag3",
    "mprestamos_personales_lag6",
    "mprestamos_personales_delta1",
    "mprestamos_personales_delta3",
    "mprestamos_personales_prior3_mean",
    "mprestamos_personales_prior3_observed",
    "mprestamos_personales_prior6_mean",
    "mprestamos_personales_prior6_observed",
    "mrentabilidad_lag1",
    "mrentabilidad_lag2",
    "mrentabilidad_lag3",
    "mrentabilidad_lag6",
    "mrentabilidad_delta1",
    "mrentabilidad_delta3",
    "mrentabilidad_prior3_mean",
    "mrentabilidad_prior3_observed",
    "mrentabilidad_prior6_mean",
    "mrentabilidad_prior6_observed",
    "active_quarter_lag1",
    "active_quarter_delta1",
    "active_quarter_lag2",
    "active_quarter_delta2",
    "active_quarter_ols6",
    "active_quarter_sma3",
    "active_quarter_sma6",
    "active_quarter_min6",
    "active_quarter_max6",
    "active_quarter_ema3",
    "active_quarter_ema6",
    "cliente_vip_lag1",
    "cliente_vip_delta1",
    "cliente_vip_lag2",
    "cliente_vip_delta2",
    "cliente_vip_ols6",
    "cliente_vip_sma3",
    "cliente_vip_sma6",
    "cliente_vip_min6",
    "cliente_vip_max6",
    "cliente_vip_ema3",
    "cliente_vip_ema6",
    "internet_lag1",
    "internet_delta1",
    "internet_lag2",
    "internet_delta2",
    "internet_ols6",
    "internet_sma3",
    "internet_sma6",
    "internet_min6",
    "internet_max6",
    "internet_ema3",
    "internet_ema6",
    "cliente_edad_lag1",
    "cliente_edad_delta1",
    "cliente_edad_lag2",
    "cliente_edad_delta2",
    "cliente_edad_ols6",
    "cliente_edad_sma3",
    "cliente_edad_sma6",
    "cliente_edad_min6",
    "cliente_edad_max6",
    "cliente_edad_ema3",
    "cliente_edad_ema6",
    "cliente_antiguedad_lag1",
    "cliente_antiguedad_delta1",
    "cliente_antiguedad_lag2",
    "cliente_antiguedad_delta2",
    "cliente_antiguedad_ols6",
    "cliente_antiguedad_sma3",
    "cliente_antiguedad_sma6",
    "cliente_antiguedad_min6",
    "cliente_antiguedad_max6",
    "cliente_antiguedad_ema3",
    "cliente_antiguedad_ema6",
    "mrentabilidad_delta2",
    "mrentabilidad_ols6",
    "mrentabilidad_sma3",
    "mrentabilidad_sma6",
    "mrentabilidad_min6",
    "mrentabilidad_max6",
    "mrentabilidad_ema3",
    "mrentabilidad_ema6",
    "mrentabilidad_annual_lag1",
    "mrentabilidad_annual_delta1",
    "mrentabilidad_annual_lag2",
    "mrentabilidad_annual_delta2",
    "mrentabilidad_annual_ols6",
    "mrentabilidad_annual_sma3",
    "mrentabilidad_annual_sma6",
    "mrentabilidad_annual_min6",
    "mrentabilidad_annual_max6",
    "mrentabilidad_annual_ema3",
    "mrentabilidad_annual_ema6",
    "mcomisiones_lag1",
    "mcomisiones_delta1",
    "mcomisiones_lag2",
    "mcomisiones_delta2",
    "mcomisiones_ols6",
    "mcomisiones_sma3",
    "mcomisiones_sma6",
    "mcomisiones_min6",
    "mcomisiones_max6",
    "mcomisiones_ema3",
    "mcomisiones_ema6",
    "mactivos_margen_lag1",
    "mactivos_margen_delta1",
    "mactivos_margen_lag2",
    "mactivos_margen_delta2",
    "mactivos_margen_ols6",
    "mactivos_margen_sma3",
    "mactivos_margen_sma6",
    "mactivos_margen_min6",
    "mactivos_margen_max6",
    "mactivos_margen_ema3",
    "mactivos_margen_ema6",
    "mpasivos_margen_lag1",
    "mpasivos_margen_delta1",
    "mpasivos_margen_lag2",
    "mpasivos_margen_delta2",
    "mpasivos_margen_ols6",
    "mpasivos_margen_sma3",
    "mpasivos_margen_sma6",
    "mpasivos_margen_min6",
    "mpasivos_margen_max6",
    "mpasivos_margen_ema3",
    "mpasivos_margen_ema6",
    "cproductos_lag1",
    "cproductos_delta1",
    "cproductos_lag2",
    "cproductos_delta2",
    "cproductos_ols6",
    "cproductos_sma3",
    "cproductos_sma6",
    "cproductos_min6",
    "cproductos_max6",
    "cproductos_ema3",
    "cproductos_ema6",
    "tcuentas_lag1",
    "tcuentas_delta1",
    "tcuentas_lag2",
    "tcuentas_delta2",
    "tcuentas_ols6",
    "tcuentas_sma3",
    "tcuentas_sma6",
    "tcuentas_min6",
    "tcuentas_max6",
    "tcuentas_ema3",
    "tcuentas_ema6",
    "ccuenta_corriente_lag1",
    "ccuenta_corriente_delta1",
    "ccuenta_corriente_lag2",
    "ccuenta_corriente_delta2",
    "ccuenta_corriente_ols6",
    "ccuenta_corriente_sma3",
    "ccuenta_corriente_sma6",
    "ccuenta_corriente_min6",
    "ccuenta_corriente_max6",
    "ccuenta_corriente_ema3",
    "ccuenta_corriente_ema6",
    "mcuenta_corriente_adicional_lag1",
    "mcuenta_corriente_adicional_delta1",
    "mcuenta_corriente_adicional_lag2",
    "mcuenta_corriente_adicional_delta2",
    "mcuenta_corriente_adicional_ols6",
    "mcuenta_corriente_adicional_sma3",
    "mcuenta_corriente_adicional_sma6",
    "mcuenta_corriente_adicional_min6",
    "mcuenta_corriente_adicional_max6",
    "mcuenta_corriente_adicional_ema3",
    "mcuenta_corriente_adicional_ema6",
    "mcuenta_corriente_delta2",
    "mcuenta_corriente_ols6",
    "mcuenta_corriente_sma3",
    "mcuenta_corriente_sma6",
    "mcuenta_corriente_min6",
    "mcuenta_corriente_max6",
    "mcuenta_corriente_ema3",
    "mcuenta_corriente_ema6",
    "ccaja_ahorro_lag1",
    "ccaja_ahorro_delta1",
    "ccaja_ahorro_lag2",
    "ccaja_ahorro_delta2",
    "ccaja_ahorro_ols6",
    "ccaja_ahorro_sma3",
    "ccaja_ahorro_sma6",
    "ccaja_ahorro_min6",
    "ccaja_ahorro_max6",
    "ccaja_ahorro_ema3",
    "ccaja_ahorro_ema6",
    "mcaja_ahorro_delta2",
    "mcaja_ahorro_ols6",
    "mcaja_ahorro_sma3",
    "mcaja_ahorro_sma6",
    "mcaja_ahorro_min6",
    "mcaja_ahorro_max6",
    "mcaja_ahorro_ema3",
    "mcaja_ahorro_ema6",
    "mcaja_ahorro_adicional_lag1",
    "mcaja_ahorro_adicional_delta1",
    "mcaja_ahorro_adicional_lag2",
    "mcaja_ahorro_adicional_delta2",
    "mcaja_ahorro_adicional_ols6",
    "mcaja_ahorro_adicional_sma3",
    "mcaja_ahorro_adicional_sma6",
    "mcaja_ahorro_adicional_min6",
    "mcaja_ahorro_adicional_max6",
    "mcaja_ahorro_adicional_ema3",
    "mcaja_ahorro_adicional_ema6",
    "mcaja_ahorro_dolares_lag1",
    "mcaja_ahorro_dolares_delta1",
    "mcaja_ahorro_dolares_lag2",
    "mcaja_ahorro_dolares_delta2",
    "mcaja_ahorro_dolares_ols6",
    "mcaja_ahorro_dolares_sma3",
    "mcaja_ahorro_dolares_sma6",
    "mcaja_ahorro_dolares_min6",
    "mcaja_ahorro_dolares_max6",
    "mcaja_ahorro_dolares_ema3",
    "mcaja_ahorro_dolares_ema6",
    "cdescubierto_preacordado_lag1",
    "cdescubierto_preacordado_delta1",
    "cdescubierto_preacordado_lag2",
    "cdescubierto_preacordado_delta2",
    "cdescubierto_preacordado_ols6",
    "cdescubierto_preacordado_sma3",
    "cdescubierto_preacordado_sma6",
    "cdescubierto_preacordado_min6",
    "cdescubierto_preacordado_max6",
    "cdescubierto_preacordado_ema3",
    "cdescubierto_preacordado_ema6",
    "mcuentas_saldo_delta2",
    "mcuentas_saldo_ols6",
    "mcuentas_saldo_sma3",
    "mcuentas_saldo_sma6",
    "mcuentas_saldo_min6",
    "mcuentas_saldo_max6",
    "mcuentas_saldo_ema3",
    "mcuentas_saldo_ema6",
    "ctarjeta_debito_lag1",
    "ctarjeta_debito_delta1",
    "ctarjeta_debito_lag2",
    "ctarjeta_debito_delta2",
    "ctarjeta_debito_ols6",
    "ctarjeta_debito_sma3",
    "ctarjeta_debito_sma6",
    "ctarjeta_debito_min6",
    "ctarjeta_debito_max6",
    "ctarjeta_debito_ema3",
    "ctarjeta_debito_ema6",
    "ctarjeta_debito_transacciones_delta2",
    "ctarjeta_debito_transacciones_ols6",
    "ctarjeta_debito_transacciones_sma3",
    "ctarjeta_debito_transacciones_sma6",
    "ctarjeta_debito_transacciones_min6",
    "ctarjeta_debito_transacciones_max6",
    "ctarjeta_debito_transacciones_ema3",
    "ctarjeta_debito_transacciones_ema6",
    "mautoservicio_lag1",
    "mautoservicio_delta1",
    "mautoservicio_lag2",
    "mautoservicio_delta2",
    "mautoservicio_ols6",
    "mautoservicio_sma3",
    "mautoservicio_sma6",
    "mautoservicio_min6",
    "mautoservicio_max6",
    "mautoservicio_ema3",
    "mautoservicio_ema6",
    "ctarjeta_visa_lag1",
    "ctarjeta_visa_delta1",
    "ctarjeta_visa_lag2",
    "ctarjeta_visa_delta2",
    "ctarjeta_visa_ols6",
    "ctarjeta_visa_sma3",
    "ctarjeta_visa_sma6",
    "ctarjeta_visa_min6",
    "ctarjeta_visa_max6",
    "ctarjeta_visa_ema3",
    "ctarjeta_visa_ema6",
    "ctarjeta_visa_transacciones_lag1",
    "ctarjeta_visa_transacciones_delta1",
    "ctarjeta_visa_transacciones_lag2",
    "ctarjeta_visa_transacciones_delta2",
    "ctarjeta_visa_transacciones_ols6",
    "ctarjeta_visa_transacciones_sma3",
    "ctarjeta_visa_transacciones_sma6",
    "ctarjeta_visa_transacciones_min6",
    "ctarjeta_visa_transacciones_max6",
    "ctarjeta_visa_transacciones_ema3",
    "ctarjeta_visa_transacciones_ema6",
    "mtarjeta_visa_consumo_delta2",
    "mtarjeta_visa_consumo_ols6",
    "mtarjeta_visa_consumo_sma3",
    "mtarjeta_visa_consumo_sma6",
    "mtarjeta_visa_consumo_min6",
    "mtarjeta_visa_consumo_max6",
    "mtarjeta_visa_consumo_ema3",
    "mtarjeta_visa_consumo_ema6",
    "ctarjeta_master_lag1",
    "ctarjeta_master_delta1",
    "ctarjeta_master_lag2",
    "ctarjeta_master_delta2",
    "ctarjeta_master_ols6",
    "ctarjeta_master_sma3",
    "ctarjeta_master_sma6",
    "ctarjeta_master_min6",
    "ctarjeta_master_max6",
    "ctarjeta_master_ema3",
    "ctarjeta_master_ema6",
    "ctarjeta_master_transacciones_lag1",
    "ctarjeta_master_transacciones_delta1",
    "ctarjeta_master_transacciones_lag2",
    "ctarjeta_master_transacciones_delta2",
    "ctarjeta_master_transacciones_ols6",
    "ctarjeta_master_transacciones_sma3",
    "ctarjeta_master_transacciones_sma6",
    "ctarjeta_master_transacciones_min6",
    "ctarjeta_master_transacciones_max6",
    "ctarjeta_master_transacciones_ema3",
    "ctarjeta_master_transacciones_ema6",
    "mtarjeta_master_consumo_delta2",
    "mtarjeta_master_consumo_ols6",
    "mtarjeta_master_consumo_sma3",
    "mtarjeta_master_consumo_sma6",
    "mtarjeta_master_consumo_min6",
    "mtarjeta_master_consumo_max6",
    "mtarjeta_master_consumo_ema3",
    "mtarjeta_master_consumo_ema6",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "cprestamos_personales_lag2",
    "cprestamos_personales_delta2",
    "cprestamos_personales_ols6",
    "cprestamos_personales_sma3",
    "cprestamos_personales_sma6",
    "cprestamos_personales_min6",
    "cprestamos_personales_max6",
    "cprestamos_personales_ema3",
    "cprestamos_personales_ema6",
    "mprestamos_personales_delta2",
    "mprestamos_personales_ols6",
    "mprestamos_personales_sma3",
    "mprestamos_personales_sma6",
    "mprestamos_personales_min6",
    "mprestamos_personales_max6",
    "mprestamos_personales_ema3",
    "mprestamos_personales_ema6",
    "cprestamos_prendarios_lag1",
    "cprestamos_prendarios_delta1",
    "cprestamos_prendarios_lag2",
    "cprestamos_prendarios_delta2",
    "cprestamos_prendarios_ols6",
    "cprestamos_prendarios_sma3",
    "cprestamos_prendarios_sma6",
    "cprestamos_prendarios_min6",
    "cprestamos_prendarios_max6",
    "cprestamos_prendarios_ema3",
    "cprestamos_prendarios_ema6",
    "mprestamos_prendarios_lag1",
    "mprestamos_prendarios_delta1",
    "mprestamos_prendarios_lag2",
    "mprestamos_prendarios_delta2",
    "mprestamos_prendarios_ols6",
    "mprestamos_prendarios_sma3",
    "mprestamos_prendarios_sma6",
    "mprestamos_prendarios_min6",
    "mprestamos_prendarios_max6",
    "mprestamos_prendarios_ema3",
    "mprestamos_prendarios_ema6",
    "cprestamos_hipotecarios_lag1",
    "cprestamos_hipotecarios_delta1",
    "cprestamos_hipotecarios_lag2",
    "cprestamos_hipotecarios_delta2",
    "cprestamos_hipotecarios_ols6",
    "cprestamos_hipotecarios_sma3",
    "cprestamos_hipotecarios_sma6",
    "cprestamos_hipotecarios_min6",
    "cprestamos_hipotecarios_max6",
    "cprestamos_hipotecarios_ema3",
    "cprestamos_hipotecarios_ema6",
    "mprestamos_hipotecarios_lag1",
    "mprestamos_hipotecarios_delta1",
    "mprestamos_hipotecarios_lag2",
    "mprestamos_hipotecarios_delta2",
    "mprestamos_hipotecarios_ols6",
    "mprestamos_hipotecarios_sma3",
    "mprestamos_hipotecarios_sma6",
    "mprestamos_hipotecarios_min6",
    "mprestamos_hipotecarios_max6",
    "mprestamos_hipotecarios_ema3",
    "mprestamos_hipotecarios_ema6",
    "cplazo_fijo_lag1",
    "cplazo_fijo_delta1",
    "cplazo_fijo_lag2",
    "cplazo_fijo_delta2",
    "cplazo_fijo_ols6",
    "cplazo_fijo_sma3",
    "cplazo_fijo_sma6",
    "cplazo_fijo_min6",
    "cplazo_fijo_max6",
    "cplazo_fijo_ema3",
    "cplazo_fijo_ema6",
    "mplazo_fijo_dolares_lag1",
    "mplazo_fijo_dolares_delta1",
    "mplazo_fijo_dolares_lag2",
    "mplazo_fijo_dolares_delta2",
    "mplazo_fijo_dolares_ols6",
    "mplazo_fijo_dolares_sma3",
    "mplazo_fijo_dolares_sma6",
    "mplazo_fijo_dolares_min6",
    "mplazo_fijo_dolares_max6",
    "mplazo_fijo_dolares_ema3",
    "mplazo_fijo_dolares_ema6",
    "mplazo_fijo_pesos_lag1",
    "mplazo_fijo_pesos_delta1",
    "mplazo_fijo_pesos_lag2",
    "mplazo_fijo_pesos_delta2",
    "mplazo_fijo_pesos_ols6",
    "mplazo_fijo_pesos_sma3",
    "mplazo_fijo_pesos_sma6",
    "mplazo_fijo_pesos_min6",
    "mplazo_fijo_pesos_max6",
    "mplazo_fijo_pesos_ema3",
    "mplazo_fijo_pesos_ema6",
    "cinversion1_lag1",
    "cinversion1_delta1",
    "cinversion1_lag2",
    "cinversion1_delta2",
    "cinversion1_ols6",
    "cinversion1_sma3",
    "cinversion1_sma6",
    "cinversion1_min6",
    "cinversion1_max6",
    "cinversion1_ema3",
    "cinversion1_ema6",
    "minversion1_pesos_lag1",
    "minversion1_pesos_delta1",
    "minversion1_pesos_lag2",
    "minversion1_pesos_delta2",
    "minversion1_pesos_ols6",
    "minversion1_pesos_sma3",
    "minversion1_pesos_sma6",
    "minversion1_pesos_min6",
    "minversion1_pesos_max6",
    "minversion1_pesos_ema3",
    "minversion1_pesos_ema6",
    "minversion1_dolares_lag1",
    "minversion1_dolares_delta1",
    "minversion1_dolares_lag2",
    "minversion1_dolares_delta2",
    "minversion1_dolares_ols6",
    "minversion1_dolares_sma3",
    "minversion1_dolares_sma6",
    "minversion1_dolares_min6",
    "minversion1_dolares_max6",
    "minversion1_dolares_ema3",
    "minversion1_dolares_ema6",
    "cinversion2_lag1",
    "cinversion2_delta1",
    "cinversion2_lag2",
    "cinversion2_delta2",
    "cinversion2_ols6",
    "cinversion2_sma3",
    "cinversion2_sma6",
    "cinversion2_min6",
    "cinversion2_max6",
    "cinversion2_ema3",
    "cinversion2_ema6",
    "minversion2_lag1",
    "minversion2_delta1",
    "minversion2_lag2",
    "minversion2_delta2",
    "minversion2_ols6",
    "minversion2_sma3",
    "minversion2_sma6",
    "minversion2_min6",
    "minversion2_max6",
    "minversion2_ema3",
    "minversion2_ema6",
    "cseguro_vida_lag1",
    "cseguro_vida_delta1",
    "cseguro_vida_lag2",
    "cseguro_vida_delta2",
    "cseguro_vida_ols6",
    "cseguro_vida_sma3",
    "cseguro_vida_sma6",
    "cseguro_vida_min6",
    "cseguro_vida_max6",
    "cseguro_vida_ema3",
    "cseguro_vida_ema6",
    "cseguro_auto_lag1",
    "cseguro_auto_delta1",
    "cseguro_auto_lag2",
    "cseguro_auto_delta2",
    "cseguro_auto_ols6",
    "cseguro_auto_sma3",
    "cseguro_auto_sma6",
    "cseguro_auto_min6",
    "cseguro_auto_max6",
    "cseguro_auto_ema3",
    "cseguro_auto_ema6",
    "cseguro_vivienda_lag1",
    "cseguro_vivienda_delta1",
    "cseguro_vivienda_lag2",
    "cseguro_vivienda_delta2",
    "cseguro_vivienda_ols6",
    "cseguro_vivienda_sma3",
    "cseguro_vivienda_sma6",
    "cseguro_vivienda_min6",
    "cseguro_vivienda_max6",
    "cseguro_vivienda_ema3",
    "cseguro_vivienda_ema6",
    "cseguro_accidentes_personales_lag1",
    "cseguro_accidentes_personales_delta1",
    "cseguro_accidentes_personales_lag2",
    "cseguro_accidentes_personales_delta2",
    "cseguro_accidentes_personales_ols6",
    "cseguro_accidentes_personales_sma3",
    "cseguro_accidentes_personales_sma6",
    "cseguro_accidentes_personales_min6",
    "cseguro_accidentes_personales_max6",
    "cseguro_accidentes_personales_ema3",
    "cseguro_accidentes_personales_ema6",
    "ccaja_seguridad_lag1",
    "ccaja_seguridad_delta1",
    "ccaja_seguridad_lag2",
    "ccaja_seguridad_delta2",
    "ccaja_seguridad_ols6",
    "ccaja_seguridad_sma3",
    "ccaja_seguridad_sma6",
    "ccaja_seguridad_min6",
    "ccaja_seguridad_max6",
    "ccaja_seguridad_ema3",
    "ccaja_seguridad_ema6",
    "cpayroll_trx_lag1",
    "cpayroll_trx_delta1",
    "cpayroll_trx_lag2",
    "cpayroll_trx_delta2",
    "cpayroll_trx_ols6",
    "cpayroll_trx_sma3",
    "cpayroll_trx_sma6",
    "cpayroll_trx_min6",
    "cpayroll_trx_max6",
    "cpayroll_trx_ema3",
    "cpayroll_trx_ema6",
    "mpayroll_delta2",
    "mpayroll_ols6",
    "mpayroll_sma3",
    "mpayroll_sma6",
    "mpayroll_min6",
    "mpayroll_max6",
    "mpayroll_ema3",
    "mpayroll_ema6",
    "mpayroll2_lag1",
    "mpayroll2_delta1",
    "mpayroll2_lag2",
    "mpayroll2_delta2",
    "mpayroll2_ols6",
    "mpayroll2_sma3",
    "mpayroll2_sma6",
    "mpayroll2_min6",
    "mpayroll2_max6",
    "mpayroll2_ema3",
    "mpayroll2_ema6",
    "cpayroll2_trx_lag1",
    "cpayroll2_trx_delta1",
    "cpayroll2_trx_lag2",
    "cpayroll2_trx_delta2",
    "cpayroll2_trx_ols6",
    "cpayroll2_trx_sma3",
    "cpayroll2_trx_sma6",
    "cpayroll2_trx_min6",
    "cpayroll2_trx_max6",
    "cpayroll2_trx_ema3",
    "cpayroll2_trx_ema6",
    "ccuenta_debitos_automaticos_lag1",
    "ccuenta_debitos_automaticos_delta1",
    "ccuenta_debitos_automaticos_lag2",
    "ccuenta_debitos_automaticos_delta2",
    "ccuenta_debitos_automaticos_ols6",
    "ccuenta_debitos_automaticos_sma3",
    "ccuenta_debitos_automaticos_sma6",
    "ccuenta_debitos_automaticos_min6",
    "ccuenta_debitos_automaticos_max6",
    "ccuenta_debitos_automaticos_ema3",
    "ccuenta_debitos_automaticos_ema6",
    "mcuenta_debitos_automaticos_lag1",
    "mcuenta_debitos_automaticos_delta1",
    "mcuenta_debitos_automaticos_lag2",
    "mcuenta_debitos_automaticos_delta2",
    "mcuenta_debitos_automaticos_ols6",
    "mcuenta_debitos_automaticos_sma3",
    "mcuenta_debitos_automaticos_sma6",
    "mcuenta_debitos_automaticos_min6",
    "mcuenta_debitos_automaticos_max6",
    "mcuenta_debitos_automaticos_ema3",
    "mcuenta_debitos_automaticos_ema6",
    "ctarjeta_visa_debitos_automaticos_lag1",
    "ctarjeta_visa_debitos_automaticos_delta1",
    "ctarjeta_visa_debitos_automaticos_lag2",
    "ctarjeta_visa_debitos_automaticos_delta2",
    "ctarjeta_visa_debitos_automaticos_ols6",
    "ctarjeta_visa_debitos_automaticos_sma3",
    "ctarjeta_visa_debitos_automaticos_sma6",
    "ctarjeta_visa_debitos_automaticos_min6",
    "ctarjeta_visa_debitos_automaticos_max6",
    "ctarjeta_visa_debitos_automaticos_ema3",
    "ctarjeta_visa_debitos_automaticos_ema6",
    "mttarjeta_visa_debitos_automaticos_lag1",
    "mttarjeta_visa_debitos_automaticos_delta1",
    "mttarjeta_visa_debitos_automaticos_lag2",
    "mttarjeta_visa_debitos_automaticos_delta2",
    "mttarjeta_visa_debitos_automaticos_ols6",
    "mttarjeta_visa_debitos_automaticos_sma3",
    "mttarjeta_visa_debitos_automaticos_sma6",
    "mttarjeta_visa_debitos_automaticos_min6",
    "mttarjeta_visa_debitos_automaticos_max6",
    "mttarjeta_visa_debitos_automaticos_ema3",
    "mttarjeta_visa_debitos_automaticos_ema6",
    "ctarjeta_master_debitos_automaticos_lag1",
    "ctarjeta_master_debitos_automaticos_delta1",
    "ctarjeta_master_debitos_automaticos_lag2",
    "ctarjeta_master_debitos_automaticos_delta2",
    "ctarjeta_master_debitos_automaticos_ols6",
    "ctarjeta_master_debitos_automaticos_sma3",
    "ctarjeta_master_debitos_automaticos_sma6",
    "ctarjeta_master_debitos_automaticos_min6",
    "ctarjeta_master_debitos_automaticos_max6",
    "ctarjeta_master_debitos_automaticos_ema3",
    "ctarjeta_master_debitos_automaticos_ema6",
    "mttarjeta_master_debitos_automaticos_lag1",
    "mttarjeta_master_debitos_automaticos_delta1",
    "mttarjeta_master_debitos_automaticos_lag2",
    "mttarjeta_master_debitos_automaticos_delta2",
    "mttarjeta_master_debitos_automaticos_ols6",
    "mttarjeta_master_debitos_automaticos_sma3",
    "mttarjeta_master_debitos_automaticos_sma6",
    "mttarjeta_master_debitos_automaticos_min6",
    "mttarjeta_master_debitos_automaticos_max6",
    "mttarjeta_master_debitos_automaticos_ema3",
    "mttarjeta_master_debitos_automaticos_ema6",
    "cpagodeservicios_lag1",
    "cpagodeservicios_delta1",
    "cpagodeservicios_lag2",
    "cpagodeservicios_delta2",
    "cpagodeservicios_ols6",
    "cpagodeservicios_sma3",
    "cpagodeservicios_sma6",
    "cpagodeservicios_min6",
    "cpagodeservicios_max6",
    "cpagodeservicios_ema3",
    "cpagodeservicios_ema6",
    "mpagodeservicios_lag1",
    "mpagodeservicios_delta1",
    "mpagodeservicios_lag2",
    "mpagodeservicios_delta2",
    "mpagodeservicios_ols6",
    "mpagodeservicios_sma3",
    "mpagodeservicios_sma6",
    "mpagodeservicios_min6",
    "mpagodeservicios_max6",
    "mpagodeservicios_ema3",
    "mpagodeservicios_ema6",
    "cpagomiscuentas_lag1",
    "cpagomiscuentas_delta1",
    "cpagomiscuentas_lag2",
    "cpagomiscuentas_delta2",
    "cpagomiscuentas_ols6",
    "cpagomiscuentas_sma3",
    "cpagomiscuentas_sma6",
    "cpagomiscuentas_min6",
    "cpagomiscuentas_max6",
    "cpagomiscuentas_ema3",
    "cpagomiscuentas_ema6",
    "mpagomiscuentas_lag1",
    "mpagomiscuentas_delta1",
    "mpagomiscuentas_lag2",
    "mpagomiscuentas_delta2",
    "mpagomiscuentas_ols6",
    "mpagomiscuentas_sma3",
    "mpagomiscuentas_sma6",
    "mpagomiscuentas_min6",
    "mpagomiscuentas_max6",
    "mpagomiscuentas_ema3",
    "mpagomiscuentas_ema6",
    "ccajeros_propios_descuentos_lag1",
    "ccajeros_propios_descuentos_delta1",
    "ccajeros_propios_descuentos_lag2",
    "ccajeros_propios_descuentos_delta2",
    "ccajeros_propios_descuentos_ols6",
    "ccajeros_propios_descuentos_sma3",
    "ccajeros_propios_descuentos_sma6",
    "ccajeros_propios_descuentos_min6",
    "ccajeros_propios_descuentos_max6",
    "ccajeros_propios_descuentos_ema3",
    "ccajeros_propios_descuentos_ema6",
    "mcajeros_propios_descuentos_lag1",
    "mcajeros_propios_descuentos_delta1",
    "mcajeros_propios_descuentos_lag2",
    "mcajeros_propios_descuentos_delta2",
    "mcajeros_propios_descuentos_ols6",
    "mcajeros_propios_descuentos_sma3",
    "mcajeros_propios_descuentos_sma6",
    "mcajeros_propios_descuentos_min6",
    "mcajeros_propios_descuentos_max6",
    "mcajeros_propios_descuentos_ema3",
    "mcajeros_propios_descuentos_ema6",
    "ctarjeta_visa_descuentos_lag1",
    "ctarjeta_visa_descuentos_delta1",
    "ctarjeta_visa_descuentos_lag2",
    "ctarjeta_visa_descuentos_delta2",
    "ctarjeta_visa_descuentos_ols6",
    "ctarjeta_visa_descuentos_sma3",
    "ctarjeta_visa_descuentos_sma6",
    "ctarjeta_visa_descuentos_min6",
    "ctarjeta_visa_descuentos_max6",
    "ctarjeta_visa_descuentos_ema3",
    "ctarjeta_visa_descuentos_ema6",
    "mtarjeta_visa_descuentos_lag1",
    "mtarjeta_visa_descuentos_delta1",
    "mtarjeta_visa_descuentos_lag2",
    "mtarjeta_visa_descuentos_delta2",
    "mtarjeta_visa_descuentos_ols6",
    "mtarjeta_visa_descuentos_sma3",
    "mtarjeta_visa_descuentos_sma6",
    "mtarjeta_visa_descuentos_min6",
    "mtarjeta_visa_descuentos_max6",
    "mtarjeta_visa_descuentos_ema3",
    "mtarjeta_visa_descuentos_ema6",
    "ctarjeta_master_descuentos_lag1",
    "ctarjeta_master_descuentos_delta1",
    "ctarjeta_master_descuentos_lag2",
    "ctarjeta_master_descuentos_delta2",
    "ctarjeta_master_descuentos_ols6",
    "ctarjeta_master_descuentos_sma3",
    "ctarjeta_master_descuentos_sma6",
    "ctarjeta_master_descuentos_min6",
    "ctarjeta_master_descuentos_max6",
    "ctarjeta_master_descuentos_ema3",
    "ctarjeta_master_descuentos_ema6",
    "mtarjeta_master_descuentos_lag1",
    "mtarjeta_master_descuentos_delta1",
    "mtarjeta_master_descuentos_lag2",
    "mtarjeta_master_descuentos_delta2",
    "mtarjeta_master_descuentos_ols6",
    "mtarjeta_master_descuentos_sma3",
    "mtarjeta_master_descuentos_sma6",
    "mtarjeta_master_descuentos_min6",
    "mtarjeta_master_descuentos_max6",
    "mtarjeta_master_descuentos_ema3",
    "mtarjeta_master_descuentos_ema6",
    "ccomisiones_mantenimiento_lag1",
    "ccomisiones_mantenimiento_delta1",
    "ccomisiones_mantenimiento_lag2",
    "ccomisiones_mantenimiento_delta2",
    "ccomisiones_mantenimiento_ols6",
    "ccomisiones_mantenimiento_sma3",
    "ccomisiones_mantenimiento_sma6",
    "ccomisiones_mantenimiento_min6",
    "ccomisiones_mantenimiento_max6",
    "ccomisiones_mantenimiento_ema3",
    "ccomisiones_mantenimiento_ema6",
    "mcomisiones_mantenimiento_lag1",
    "mcomisiones_mantenimiento_delta1",
    "mcomisiones_mantenimiento_lag2",
    "mcomisiones_mantenimiento_delta2",
    "mcomisiones_mantenimiento_ols6",
    "mcomisiones_mantenimiento_sma3",
    "mcomisiones_mantenimiento_sma6",
    "mcomisiones_mantenimiento_min6",
    "mcomisiones_mantenimiento_max6",
    "mcomisiones_mantenimiento_ema3",
    "mcomisiones_mantenimiento_ema6",
    "ccomisiones_otras_lag1",
    "ccomisiones_otras_delta1",
    "ccomisiones_otras_lag2",
    "ccomisiones_otras_delta2",
    "ccomisiones_otras_ols6",
    "ccomisiones_otras_sma3",
    "ccomisiones_otras_sma6",
    "ccomisiones_otras_min6",
    "ccomisiones_otras_max6",
    "ccomisiones_otras_ema3",
    "ccomisiones_otras_ema6",
    "mcomisiones_otras_lag1",
    "mcomisiones_otras_delta1",
    "mcomisiones_otras_lag2",
    "mcomisiones_otras_delta2",
    "mcomisiones_otras_ols6",
    "mcomisiones_otras_sma3",
    "mcomisiones_otras_sma6",
    "mcomisiones_otras_min6",
    "mcomisiones_otras_max6",
    "mcomisiones_otras_ema3",
    "mcomisiones_otras_ema6",
    "cforex_lag1",
    "cforex_delta1",
    "cforex_lag2",
    "cforex_delta2",
    "cforex_ols6",
    "cforex_sma3",
    "cforex_sma6",
    "cforex_min6",
    "cforex_max6",
    "cforex_ema3",
    "cforex_ema6",
    "cforex_buy_lag1",
    "cforex_buy_delta1",
    "cforex_buy_lag2",
    "cforex_buy_delta2",
    "cforex_buy_ols6",
    "cforex_buy_sma3",
    "cforex_buy_sma6",
    "cforex_buy_min6",
    "cforex_buy_max6",
    "cforex_buy_ema3",
    "cforex_buy_ema6",
    "mforex_buy_lag1",
    "mforex_buy_delta1",
    "mforex_buy_lag2",
    "mforex_buy_delta2",
    "mforex_buy_ols6",
    "mforex_buy_sma3",
    "mforex_buy_sma6",
    "mforex_buy_min6",
    "mforex_buy_max6",
    "mforex_buy_ema3",
    "mforex_buy_ema6",
    "cforex_sell_lag1",
    "cforex_sell_delta1",
    "cforex_sell_lag2",
    "cforex_sell_delta2",
    "cforex_sell_ols6",
    "cforex_sell_sma3",
    "cforex_sell_sma6",
    "cforex_sell_min6",
    "cforex_sell_max6",
    "cforex_sell_ema3",
    "cforex_sell_ema6",
    "mforex_sell_lag1",
    "mforex_sell_delta1",
    "mforex_sell_lag2",
    "mforex_sell_delta2",
    "mforex_sell_ols6",
    "mforex_sell_sma3",
    "mforex_sell_sma6",
    "mforex_sell_min6",
    "mforex_sell_max6",
    "mforex_sell_ema3",
    "mforex_sell_ema6",
    "ctransferencias_recibidas_lag1",
    "ctransferencias_recibidas_delta1",
    "ctransferencias_recibidas_lag2",
    "ctransferencias_recibidas_delta2",
    "ctransferencias_recibidas_ols6",
    "ctransferencias_recibidas_sma3",
    "ctransferencias_recibidas_sma6",
    "ctransferencias_recibidas_min6",
    "ctransferencias_recibidas_max6",
    "ctransferencias_recibidas_ema3",
    "ctransferencias_recibidas_ema6",
    "mtransferencias_recibidas_delta2",
    "mtransferencias_recibidas_ols6",
    "mtransferencias_recibidas_sma3",
    "mtransferencias_recibidas_sma6",
    "mtransferencias_recibidas_min6",
    "mtransferencias_recibidas_max6",
    "mtransferencias_recibidas_ema3",
    "mtransferencias_recibidas_ema6",
    "ctransferencias_emitidas_lag1",
    "ctransferencias_emitidas_delta1",
    "ctransferencias_emitidas_lag2",
    "ctransferencias_emitidas_delta2",
    "ctransferencias_emitidas_ols6",
    "ctransferencias_emitidas_sma3",
    "ctransferencias_emitidas_sma6",
    "ctransferencias_emitidas_min6",
    "ctransferencias_emitidas_max6",
    "ctransferencias_emitidas_ema3",
    "ctransferencias_emitidas_ema6",
    "mtransferencias_emitidas_delta2",
    "mtransferencias_emitidas_ols6",
    "mtransferencias_emitidas_sma3",
    "mtransferencias_emitidas_sma6",
    "mtransferencias_emitidas_min6",
    "mtransferencias_emitidas_max6",
    "mtransferencias_emitidas_ema3",
    "mtransferencias_emitidas_ema6",
    "cextraccion_autoservicio_lag1",
    "cextraccion_autoservicio_delta1",
    "cextraccion_autoservicio_lag2",
    "cextraccion_autoservicio_delta2",
    "cextraccion_autoservicio_ols6",
    "cextraccion_autoservicio_sma3",
    "cextraccion_autoservicio_sma6",
    "cextraccion_autoservicio_min6",
    "cextraccion_autoservicio_max6",
    "cextraccion_autoservicio_ema3",
    "cextraccion_autoservicio_ema6",
    "mextraccion_autoservicio_lag1",
    "mextraccion_autoservicio_delta1",
    "mextraccion_autoservicio_lag2",
    "mextraccion_autoservicio_delta2",
    "mextraccion_autoservicio_ols6",
    "mextraccion_autoservicio_sma3",
    "mextraccion_autoservicio_sma6",
    "mextraccion_autoservicio_min6",
    "mextraccion_autoservicio_max6",
    "mextraccion_autoservicio_ema3",
    "mextraccion_autoservicio_ema6",
    "ccheques_depositados_lag1",
    "ccheques_depositados_delta1",
    "ccheques_depositados_lag2",
    "ccheques_depositados_delta2",
    "ccheques_depositados_ols6",
    "ccheques_depositados_sma3",
    "ccheques_depositados_sma6",
    "ccheques_depositados_min6",
    "ccheques_depositados_max6",
    "ccheques_depositados_ema3",
    "ccheques_depositados_ema6",
    "mcheques_depositados_lag1",
    "mcheques_depositados_delta1",
    "mcheques_depositados_lag2",
    "mcheques_depositados_delta2",
    "mcheques_depositados_ols6",
    "mcheques_depositados_sma3",
    "mcheques_depositados_sma6",
    "mcheques_depositados_min6",
    "mcheques_depositados_max6",
    "mcheques_depositados_ema3",
    "mcheques_depositados_ema6",
    "ccheques_emitidos_lag1",
    "ccheques_emitidos_delta1",
    "ccheques_emitidos_lag2",
    "ccheques_emitidos_delta2",
    "ccheques_emitidos_ols6",
    "ccheques_emitidos_sma3",
    "ccheques_emitidos_sma6",
    "ccheques_emitidos_min6",
    "ccheques_emitidos_max6",
    "ccheques_emitidos_ema3",
    "ccheques_emitidos_ema6",
    "mcheques_emitidos_lag1",
    "mcheques_emitidos_delta1",
    "mcheques_emitidos_lag2",
    "mcheques_emitidos_delta2",
    "mcheques_emitidos_ols6",
    "mcheques_emitidos_sma3",
    "mcheques_emitidos_sma6",
    "mcheques_emitidos_min6",
    "mcheques_emitidos_max6",
    "mcheques_emitidos_ema3",
    "mcheques_emitidos_ema6",
    "ccheques_depositados_rechazados_lag1",
    "ccheques_depositados_rechazados_delta1",
    "ccheques_depositados_rechazados_lag2",
    "ccheques_depositados_rechazados_delta2",
    "ccheques_depositados_rechazados_ols6",
    "ccheques_depositados_rechazados_sma3",
    "ccheques_depositados_rechazados_sma6",
    "ccheques_depositados_rechazados_min6",
    "ccheques_depositados_rechazados_max6",
    "ccheques_depositados_rechazados_ema3",
    "ccheques_depositados_rechazados_ema6",
    "mcheques_depositados_rechazados_lag1",
    "mcheques_depositados_rechazados_delta1",
    "mcheques_depositados_rechazados_lag2",
    "mcheques_depositados_rechazados_delta2",
    "mcheques_depositados_rechazados_ols6",
    "mcheques_depositados_rechazados_sma3",
    "mcheques_depositados_rechazados_sma6",
    "mcheques_depositados_rechazados_min6",
    "mcheques_depositados_rechazados_max6",
    "mcheques_depositados_rechazados_ema3",
    "mcheques_depositados_rechazados_ema6",
    "ccheques_emitidos_rechazados_lag1",
    "ccheques_emitidos_rechazados_delta1",
    "ccheques_emitidos_rechazados_lag2",
    "ccheques_emitidos_rechazados_delta2",
    "ccheques_emitidos_rechazados_ols6",
    "ccheques_emitidos_rechazados_sma3",
    "ccheques_emitidos_rechazados_sma6",
    "ccheques_emitidos_rechazados_min6",
    "ccheques_emitidos_rechazados_max6",
    "ccheques_emitidos_rechazados_ema3",
    "ccheques_emitidos_rechazados_ema6",
    "mcheques_emitidos_rechazados_lag1",
    "mcheques_emitidos_rechazados_delta1",
    "mcheques_emitidos_rechazados_lag2",
    "mcheques_emitidos_rechazados_delta2",
    "mcheques_emitidos_rechazados_ols6",
    "mcheques_emitidos_rechazados_sma3",
    "mcheques_emitidos_rechazados_sma6",
    "mcheques_emitidos_rechazados_min6",
    "mcheques_emitidos_rechazados_max6",
    "mcheques_emitidos_rechazados_ema3",
    "mcheques_emitidos_rechazados_ema6",
    "tcallcenter_lag1",
    "tcallcenter_delta1",
    "tcallcenter_lag2",
    "tcallcenter_delta2",
    "tcallcenter_ols6",
    "tcallcenter_sma3",
    "tcallcenter_sma6",
    "tcallcenter_min6",
    "tcallcenter_max6",
    "tcallcenter_ema3",
    "tcallcenter_ema6",
    "ccallcenter_transacciones_lag1",
    "ccallcenter_transacciones_delta1",
    "ccallcenter_transacciones_lag2",
    "ccallcenter_transacciones_delta2",
    "ccallcenter_transacciones_ols6",
    "ccallcenter_transacciones_sma3",
    "ccallcenter_transacciones_sma6",
    "ccallcenter_transacciones_min6",
    "ccallcenter_transacciones_max6",
    "ccallcenter_transacciones_ema3",
    "ccallcenter_transacciones_ema6",
    "thomebanking_lag1",
    "thomebanking_delta1",
    "thomebanking_lag2",
    "thomebanking_delta2",
    "thomebanking_ols6",
    "thomebanking_sma3",
    "thomebanking_sma6",
    "thomebanking_min6",
    "thomebanking_max6",
    "thomebanking_ema3",
    "thomebanking_ema6",
    "chomebanking_transacciones_lag1",
    "chomebanking_transacciones_delta1",
    "chomebanking_transacciones_lag2",
    "chomebanking_transacciones_delta2",
    "chomebanking_transacciones_ols6",
    "chomebanking_transacciones_sma3",
    "chomebanking_transacciones_sma6",
    "chomebanking_transacciones_min6",
    "chomebanking_transacciones_max6",
    "chomebanking_transacciones_ema3",
    "chomebanking_transacciones_ema6",
    "ccajas_transacciones_lag1",
    "ccajas_transacciones_delta1",
    "ccajas_transacciones_lag2",
    "ccajas_transacciones_delta2",
    "ccajas_transacciones_ols6",
    "ccajas_transacciones_sma3",
    "ccajas_transacciones_sma6",
    "ccajas_transacciones_min6",
    "ccajas_transacciones_max6",
    "ccajas_transacciones_ema3",
    "ccajas_transacciones_ema6",
    "ccajas_consultas_lag1",
    "ccajas_consultas_delta1",
    "ccajas_consultas_lag2",
    "ccajas_consultas_delta2",
    "ccajas_consultas_ols6",
    "ccajas_consultas_sma3",
    "ccajas_consultas_sma6",
    "ccajas_consultas_min6",
    "ccajas_consultas_max6",
    "ccajas_consultas_ema3",
    "ccajas_consultas_ema6",
    "ccajas_depositos_lag1",
    "ccajas_depositos_delta1",
    "ccajas_depositos_lag2",
    "ccajas_depositos_delta2",
    "ccajas_depositos_ols6",
    "ccajas_depositos_sma3",
    "ccajas_depositos_sma6",
    "ccajas_depositos_min6",
    "ccajas_depositos_max6",
    "ccajas_depositos_ema3",
    "ccajas_depositos_ema6",
    "ccajas_extracciones_lag1",
    "ccajas_extracciones_delta1",
    "ccajas_extracciones_lag2",
    "ccajas_extracciones_delta2",
    "ccajas_extracciones_ols6",
    "ccajas_extracciones_sma3",
    "ccajas_extracciones_sma6",
    "ccajas_extracciones_min6",
    "ccajas_extracciones_max6",
    "ccajas_extracciones_ema3",
    "ccajas_extracciones_ema6",
    "ccajas_otras_lag1",
    "ccajas_otras_delta1",
    "ccajas_otras_lag2",
    "ccajas_otras_delta2",
    "ccajas_otras_ols6",
    "ccajas_otras_sma3",
    "ccajas_otras_sma6",
    "ccajas_otras_min6",
    "ccajas_otras_max6",
    "ccajas_otras_ema3",
    "ccajas_otras_ema6",
    "catm_trx_lag1",
    "catm_trx_delta1",
    "catm_trx_lag2",
    "catm_trx_delta2",
    "catm_trx_ols6",
    "catm_trx_sma3",
    "catm_trx_sma6",
    "catm_trx_min6",
    "catm_trx_max6",
    "catm_trx_ema3",
    "catm_trx_ema6",
    "matm_lag1",
    "matm_delta1",
    "matm_lag2",
    "matm_delta2",
    "matm_ols6",
    "matm_sma3",
    "matm_sma6",
    "matm_min6",
    "matm_max6",
    "matm_ema3",
    "matm_ema6",
    "catm_trx_other_lag1",
    "catm_trx_other_delta1",
    "catm_trx_other_lag2",
    "catm_trx_other_delta2",
    "catm_trx_other_ols6",
    "catm_trx_other_sma3",
    "catm_trx_other_sma6",
    "catm_trx_other_min6",
    "catm_trx_other_max6",
    "catm_trx_other_ema3",
    "catm_trx_other_ema6",
    "matm_other_lag1",
    "matm_other_delta1",
    "matm_other_lag2",
    "matm_other_delta2",
    "matm_other_ols6",
    "matm_other_sma3",
    "matm_other_sma6",
    "matm_other_min6",
    "matm_other_max6",
    "matm_other_ema3",
    "matm_other_ema6",
    "ctrx_quarter_delta2",
    "ctrx_quarter_ols6",
    "ctrx_quarter_sma3",
    "ctrx_quarter_sma6",
    "ctrx_quarter_min6",
    "ctrx_quarter_max6",
    "ctrx_quarter_ema3",
    "ctrx_quarter_ema6",
    "tmobile_app_lag1",
    "tmobile_app_delta1",
    "tmobile_app_lag2",
    "tmobile_app_delta2",
    "tmobile_app_ols6",
    "tmobile_app_sma3",
    "tmobile_app_sma6",
    "tmobile_app_min6",
    "tmobile_app_max6",
    "tmobile_app_ema3",
    "tmobile_app_ema6",
    "cmobile_app_trx_lag1",
    "cmobile_app_trx_delta1",
    "cmobile_app_trx_lag2",
    "cmobile_app_trx_delta2",
    "cmobile_app_trx_ols6",
    "cmobile_app_trx_sma3",
    "cmobile_app_trx_sma6",
    "cmobile_app_trx_min6",
    "cmobile_app_trx_max6",
    "cmobile_app_trx_ema3",
    "cmobile_app_trx_ema6",
    "Master_delinquency_lag1",
    "Master_delinquency_delta1",
    "Master_delinquency_lag2",
    "Master_delinquency_delta2",
    "Master_delinquency_ols6",
    "Master_delinquency_sma3",
    "Master_delinquency_sma6",
    "Master_delinquency_min6",
    "Master_delinquency_max6",
    "Master_delinquency_ema3",
    "Master_delinquency_ema6",
    "Master_status_lag1",
    "Master_status_delta1",
    "Master_status_lag2",
    "Master_status_delta2",
    "Master_status_ols6",
    "Master_status_sma3",
    "Master_status_sma6",
    "Master_status_min6",
    "Master_status_max6",
    "Master_status_ema3",
    "Master_status_ema6",
    "Master_mfinanciacion_limite_lag1",
    "Master_mfinanciacion_limite_delta1",
    "Master_mfinanciacion_limite_lag2",
    "Master_mfinanciacion_limite_delta2",
    "Master_mfinanciacion_limite_ols6",
    "Master_mfinanciacion_limite_sma3",
    "Master_mfinanciacion_limite_sma6",
    "Master_mfinanciacion_limite_min6",
    "Master_mfinanciacion_limite_max6",
    "Master_mfinanciacion_limite_ema3",
    "Master_mfinanciacion_limite_ema6",
    "Master_Fvencimiento_lag1",
    "Master_Fvencimiento_delta1",
    "Master_Fvencimiento_lag2",
    "Master_Fvencimiento_delta2",
    "Master_Fvencimiento_ols6",
    "Master_Fvencimiento_sma3",
    "Master_Fvencimiento_sma6",
    "Master_Fvencimiento_min6",
    "Master_Fvencimiento_max6",
    "Master_Fvencimiento_ema3",
    "Master_Fvencimiento_ema6",
    "Master_Finiciomora_lag1",
    "Master_Finiciomora_delta1",
    "Master_Finiciomora_lag2",
    "Master_Finiciomora_delta2",
    "Master_Finiciomora_ols6",
    "Master_Finiciomora_sma3",
    "Master_Finiciomora_sma6",
    "Master_Finiciomora_min6",
    "Master_Finiciomora_max6",
    "Master_Finiciomora_ema3",
    "Master_Finiciomora_ema6",
    "Master_msaldototal_lag1",
    "Master_msaldototal_delta1",
    "Master_msaldototal_lag2",
    "Master_msaldototal_delta2",
    "Master_msaldototal_ols6",
    "Master_msaldototal_sma3",
    "Master_msaldototal_sma6",
    "Master_msaldototal_min6",
    "Master_msaldototal_max6",
    "Master_msaldototal_ema3",
    "Master_msaldototal_ema6",
    "Master_msaldopesos_lag1",
    "Master_msaldopesos_delta1",
    "Master_msaldopesos_lag2",
    "Master_msaldopesos_delta2",
    "Master_msaldopesos_ols6",
    "Master_msaldopesos_sma3",
    "Master_msaldopesos_sma6",
    "Master_msaldopesos_min6",
    "Master_msaldopesos_max6",
    "Master_msaldopesos_ema3",
    "Master_msaldopesos_ema6",
    "Master_msaldodolares_lag1",
    "Master_msaldodolares_delta1",
    "Master_msaldodolares_lag2",
    "Master_msaldodolares_delta2",
    "Master_msaldodolares_ols6",
    "Master_msaldodolares_sma3",
    "Master_msaldodolares_sma6",
    "Master_msaldodolares_min6",
    "Master_msaldodolares_max6",
    "Master_msaldodolares_ema3",
    "Master_msaldodolares_ema6",
    "Master_mconsumospesos_lag1",
    "Master_mconsumospesos_delta1",
    "Master_mconsumospesos_lag2",
    "Master_mconsumospesos_delta2",
    "Master_mconsumospesos_ols6",
    "Master_mconsumospesos_sma3",
    "Master_mconsumospesos_sma6",
    "Master_mconsumospesos_min6",
    "Master_mconsumospesos_max6",
    "Master_mconsumospesos_ema3",
    "Master_mconsumospesos_ema6",
    "Master_mconsumosdolares_lag1",
    "Master_mconsumosdolares_delta1",
    "Master_mconsumosdolares_lag2",
    "Master_mconsumosdolares_delta2",
    "Master_mconsumosdolares_ols6",
    "Master_mconsumosdolares_sma3",
    "Master_mconsumosdolares_sma6",
    "Master_mconsumosdolares_min6",
    "Master_mconsumosdolares_max6",
    "Master_mconsumosdolares_ema3",
    "Master_mconsumosdolares_ema6",
    "Master_mlimitecompra_lag1",
    "Master_mlimitecompra_delta1",
    "Master_mlimitecompra_lag2",
    "Master_mlimitecompra_delta2",
    "Master_mlimitecompra_ols6",
    "Master_mlimitecompra_sma3",
    "Master_mlimitecompra_sma6",
    "Master_mlimitecompra_min6",
    "Master_mlimitecompra_max6",
    "Master_mlimitecompra_ema3",
    "Master_mlimitecompra_ema6",
    "Master_madelantopesos_lag1",
    "Master_madelantopesos_delta1",
    "Master_madelantopesos_lag2",
    "Master_madelantopesos_delta2",
    "Master_madelantopesos_ols6",
    "Master_madelantopesos_sma3",
    "Master_madelantopesos_sma6",
    "Master_madelantopesos_min6",
    "Master_madelantopesos_max6",
    "Master_madelantopesos_ema3",
    "Master_madelantopesos_ema6",
    "Master_madelantodolares_lag1",
    "Master_madelantodolares_delta1",
    "Master_madelantodolares_lag2",
    "Master_madelantodolares_delta2",
    "Master_madelantodolares_ols6",
    "Master_madelantodolares_sma3",
    "Master_madelantodolares_sma6",
    "Master_madelantodolares_min6",
    "Master_madelantodolares_max6",
    "Master_madelantodolares_ema3",
    "Master_madelantodolares_ema6",
    "Master_fultimo_cierre_lag1",
    "Master_fultimo_cierre_delta1",
    "Master_fultimo_cierre_lag2",
    "Master_fultimo_cierre_delta2",
    "Master_fultimo_cierre_ols6",
    "Master_fultimo_cierre_sma3",
    "Master_fultimo_cierre_sma6",
    "Master_fultimo_cierre_min6",
    "Master_fultimo_cierre_max6",
    "Master_fultimo_cierre_ema3",
    "Master_fultimo_cierre_ema6",
    "Master_mpagado_lag1",
    "Master_mpagado_delta1",
    "Master_mpagado_lag2",
    "Master_mpagado_delta2",
    "Master_mpagado_ols6",
    "Master_mpagado_sma3",
    "Master_mpagado_sma6",
    "Master_mpagado_min6",
    "Master_mpagado_max6",
    "Master_mpagado_ema3",
    "Master_mpagado_ema6",
    "Master_mpagospesos_lag1",
    "Master_mpagospesos_delta1",
    "Master_mpagospesos_lag2",
    "Master_mpagospesos_delta2",
    "Master_mpagospesos_ols6",
    "Master_mpagospesos_sma3",
    "Master_mpagospesos_sma6",
    "Master_mpagospesos_min6",
    "Master_mpagospesos_max6",
    "Master_mpagospesos_ema3",
    "Master_mpagospesos_ema6",
    "Master_mpagosdolares_lag1",
    "Master_mpagosdolares_delta1",
    "Master_mpagosdolares_lag2",
    "Master_mpagosdolares_delta2",
    "Master_mpagosdolares_ols6",
    "Master_mpagosdolares_sma3",
    "Master_mpagosdolares_sma6",
    "Master_mpagosdolares_min6",
    "Master_mpagosdolares_max6",
    "Master_mpagosdolares_ema3",
    "Master_mpagosdolares_ema6",
    "Master_fechaalta_lag1",
    "Master_fechaalta_delta1",
    "Master_fechaalta_lag2",
    "Master_fechaalta_delta2",
    "Master_fechaalta_ols6",
    "Master_fechaalta_sma3",
    "Master_fechaalta_sma6",
    "Master_fechaalta_min6",
    "Master_fechaalta_max6",
    "Master_fechaalta_ema3",
    "Master_fechaalta_ema6",
    "Master_mconsumototal_lag1",
    "Master_mconsumototal_delta1",
    "Master_mconsumototal_lag2",
    "Master_mconsumototal_delta2",
    "Master_mconsumototal_ols6",
    "Master_mconsumototal_sma3",
    "Master_mconsumototal_sma6",
    "Master_mconsumototal_min6",
    "Master_mconsumototal_max6",
    "Master_mconsumototal_ema3",
    "Master_mconsumototal_ema6",
    "Master_cconsumos_lag1",
    "Master_cconsumos_delta1",
    "Master_cconsumos_lag2",
    "Master_cconsumos_delta2",
    "Master_cconsumos_ols6",
    "Master_cconsumos_sma3",
    "Master_cconsumos_sma6",
    "Master_cconsumos_min6",
    "Master_cconsumos_max6",
    "Master_cconsumos_ema3",
    "Master_cconsumos_ema6",
    "Master_cadelantosefectivo_lag1",
    "Master_cadelantosefectivo_delta1",
    "Master_cadelantosefectivo_lag2",
    "Master_cadelantosefectivo_delta2",
    "Master_cadelantosefectivo_ols6",
    "Master_cadelantosefectivo_sma3",
    "Master_cadelantosefectivo_sma6",
    "Master_cadelantosefectivo_min6",
    "Master_cadelantosefectivo_max6",
    "Master_cadelantosefectivo_ema3",
    "Master_cadelantosefectivo_ema6",
    "Master_mpagominimo_lag1",
    "Master_mpagominimo_delta1",
    "Master_mpagominimo_lag2",
    "Master_mpagominimo_delta2",
    "Master_mpagominimo_ols6",
    "Master_mpagominimo_sma3",
    "Master_mpagominimo_sma6",
    "Master_mpagominimo_min6",
    "Master_mpagominimo_max6",
    "Master_mpagominimo_ema3",
    "Master_mpagominimo_ema6",
    "Visa_delinquency_lag1",
    "Visa_delinquency_delta1",
    "Visa_delinquency_lag2",
    "Visa_delinquency_delta2",
    "Visa_delinquency_ols6",
    "Visa_delinquency_sma3",
    "Visa_delinquency_sma6",
    "Visa_delinquency_min6",
    "Visa_delinquency_max6",
    "Visa_delinquency_ema3",
    "Visa_delinquency_ema6",
    "Visa_status_lag1",
    "Visa_status_delta1",
    "Visa_status_lag2",
    "Visa_status_delta2",
    "Visa_status_ols6",
    "Visa_status_sma3",
    "Visa_status_sma6",
    "Visa_status_min6",
    "Visa_status_max6",
    "Visa_status_ema3",
    "Visa_status_ema6",
    "Visa_mfinanciacion_limite_lag1",
    "Visa_mfinanciacion_limite_delta1",
    "Visa_mfinanciacion_limite_lag2",
    "Visa_mfinanciacion_limite_delta2",
    "Visa_mfinanciacion_limite_ols6",
    "Visa_mfinanciacion_limite_sma3",
    "Visa_mfinanciacion_limite_sma6",
    "Visa_mfinanciacion_limite_min6",
    "Visa_mfinanciacion_limite_max6",
    "Visa_mfinanciacion_limite_ema3",
    "Visa_mfinanciacion_limite_ema6",
    "Visa_Fvencimiento_lag1",
    "Visa_Fvencimiento_delta1",
    "Visa_Fvencimiento_lag2",
    "Visa_Fvencimiento_delta2",
    "Visa_Fvencimiento_ols6",
    "Visa_Fvencimiento_sma3",
    "Visa_Fvencimiento_sma6",
    "Visa_Fvencimiento_min6",
    "Visa_Fvencimiento_max6",
    "Visa_Fvencimiento_ema3",
    "Visa_Fvencimiento_ema6",
    "Visa_Finiciomora_lag1",
    "Visa_Finiciomora_delta1",
    "Visa_Finiciomora_lag2",
    "Visa_Finiciomora_delta2",
    "Visa_Finiciomora_ols6",
    "Visa_Finiciomora_sma3",
    "Visa_Finiciomora_sma6",
    "Visa_Finiciomora_min6",
    "Visa_Finiciomora_max6",
    "Visa_Finiciomora_ema3",
    "Visa_Finiciomora_ema6",
    "Visa_msaldototal_lag1",
    "Visa_msaldototal_delta1",
    "Visa_msaldototal_lag2",
    "Visa_msaldototal_delta2",
    "Visa_msaldototal_ols6",
    "Visa_msaldototal_sma3",
    "Visa_msaldototal_sma6",
    "Visa_msaldototal_min6",
    "Visa_msaldototal_max6",
    "Visa_msaldototal_ema3",
    "Visa_msaldototal_ema6",
    "Visa_msaldopesos_lag1",
    "Visa_msaldopesos_delta1",
    "Visa_msaldopesos_lag2",
    "Visa_msaldopesos_delta2",
    "Visa_msaldopesos_ols6",
    "Visa_msaldopesos_sma3",
    "Visa_msaldopesos_sma6",
    "Visa_msaldopesos_min6",
    "Visa_msaldopesos_max6",
    "Visa_msaldopesos_ema3",
    "Visa_msaldopesos_ema6",
    "Visa_msaldodolares_lag1",
    "Visa_msaldodolares_delta1",
    "Visa_msaldodolares_lag2",
    "Visa_msaldodolares_delta2",
    "Visa_msaldodolares_ols6",
    "Visa_msaldodolares_sma3",
    "Visa_msaldodolares_sma6",
    "Visa_msaldodolares_min6",
    "Visa_msaldodolares_max6",
    "Visa_msaldodolares_ema3",
    "Visa_msaldodolares_ema6",
    "Visa_mconsumospesos_lag1",
    "Visa_mconsumospesos_delta1",
    "Visa_mconsumospesos_lag2",
    "Visa_mconsumospesos_delta2",
    "Visa_mconsumospesos_ols6",
    "Visa_mconsumospesos_sma3",
    "Visa_mconsumospesos_sma6",
    "Visa_mconsumospesos_min6",
    "Visa_mconsumospesos_max6",
    "Visa_mconsumospesos_ema3",
    "Visa_mconsumospesos_ema6",
    "Visa_mconsumosdolares_lag1",
    "Visa_mconsumosdolares_delta1",
    "Visa_mconsumosdolares_lag2",
    "Visa_mconsumosdolares_delta2",
    "Visa_mconsumosdolares_ols6",
    "Visa_mconsumosdolares_sma3",
    "Visa_mconsumosdolares_sma6",
    "Visa_mconsumosdolares_min6",
    "Visa_mconsumosdolares_max6",
    "Visa_mconsumosdolares_ema3",
    "Visa_mconsumosdolares_ema6",
    "Visa_mlimitecompra_lag1",
    "Visa_mlimitecompra_delta1",
    "Visa_mlimitecompra_lag2",
    "Visa_mlimitecompra_delta2",
    "Visa_mlimitecompra_ols6",
    "Visa_mlimitecompra_sma3",
    "Visa_mlimitecompra_sma6",
    "Visa_mlimitecompra_min6",
    "Visa_mlimitecompra_max6",
    "Visa_mlimitecompra_ema3",
    "Visa_mlimitecompra_ema6",
    "Visa_madelantopesos_lag1",
    "Visa_madelantopesos_delta1",
    "Visa_madelantopesos_lag2",
    "Visa_madelantopesos_delta2",
    "Visa_madelantopesos_ols6",
    "Visa_madelantopesos_sma3",
    "Visa_madelantopesos_sma6",
    "Visa_madelantopesos_min6",
    "Visa_madelantopesos_max6",
    "Visa_madelantopesos_ema3",
    "Visa_madelantopesos_ema6",
    "Visa_madelantodolares_lag1",
    "Visa_madelantodolares_delta1",
    "Visa_madelantodolares_lag2",
    "Visa_madelantodolares_delta2",
    "Visa_madelantodolares_ols6",
    "Visa_madelantodolares_sma3",
    "Visa_madelantodolares_sma6",
    "Visa_madelantodolares_min6",
    "Visa_madelantodolares_max6",
    "Visa_madelantodolares_ema3",
    "Visa_madelantodolares_ema6",
    "Visa_fultimo_cierre_lag1",
    "Visa_fultimo_cierre_delta1",
    "Visa_fultimo_cierre_lag2",
    "Visa_fultimo_cierre_delta2",
    "Visa_fultimo_cierre_ols6",
    "Visa_fultimo_cierre_sma3",
    "Visa_fultimo_cierre_sma6",
    "Visa_fultimo_cierre_min6",
    "Visa_fultimo_cierre_max6",
    "Visa_fultimo_cierre_ema3",
    "Visa_fultimo_cierre_ema6",
    "Visa_mpagado_lag1",
    "Visa_mpagado_delta1",
    "Visa_mpagado_lag2",
    "Visa_mpagado_delta2",
    "Visa_mpagado_ols6",
    "Visa_mpagado_sma3",
    "Visa_mpagado_sma6",
    "Visa_mpagado_min6",
    "Visa_mpagado_max6",
    "Visa_mpagado_ema3",
    "Visa_mpagado_ema6",
    "Visa_mpagospesos_lag1",
    "Visa_mpagospesos_delta1",
    "Visa_mpagospesos_lag2",
    "Visa_mpagospesos_delta2",
    "Visa_mpagospesos_ols6",
    "Visa_mpagospesos_sma3",
    "Visa_mpagospesos_sma6",
    "Visa_mpagospesos_min6",
    "Visa_mpagospesos_max6",
    "Visa_mpagospesos_ema3",
    "Visa_mpagospesos_ema6",
    "Visa_mpagosdolares_lag1",
    "Visa_mpagosdolares_delta1",
    "Visa_mpagosdolares_lag2",
    "Visa_mpagosdolares_delta2",
    "Visa_mpagosdolares_ols6",
    "Visa_mpagosdolares_sma3",
    "Visa_mpagosdolares_sma6",
    "Visa_mpagosdolares_min6",
    "Visa_mpagosdolares_max6",
    "Visa_mpagosdolares_ema3",
    "Visa_mpagosdolares_ema6",
    "Visa_fechaalta_lag1",
    "Visa_fechaalta_delta1",
    "Visa_fechaalta_lag2",
    "Visa_fechaalta_delta2",
    "Visa_fechaalta_ols6",
    "Visa_fechaalta_sma3",
    "Visa_fechaalta_sma6",
    "Visa_fechaalta_min6",
    "Visa_fechaalta_max6",
    "Visa_fechaalta_ema3",
    "Visa_fechaalta_ema6",
    "Visa_mconsumototal_lag1",
    "Visa_mconsumototal_delta1",
    "Visa_mconsumototal_lag2",
    "Visa_mconsumototal_delta2",
    "Visa_mconsumototal_ols6",
    "Visa_mconsumototal_sma3",
    "Visa_mconsumototal_sma6",
    "Visa_mconsumototal_min6",
    "Visa_mconsumototal_max6",
    "Visa_mconsumototal_ema3",
    "Visa_mconsumototal_ema6",
    "Visa_cconsumos_lag1",
    "Visa_cconsumos_delta1",
    "Visa_cconsumos_lag2",
    "Visa_cconsumos_delta2",
    "Visa_cconsumos_ols6",
    "Visa_cconsumos_sma3",
    "Visa_cconsumos_sma6",
    "Visa_cconsumos_min6",
    "Visa_cconsumos_max6",
    "Visa_cconsumos_ema3",
    "Visa_cconsumos_ema6",
    "Visa_cadelantosefectivo_lag1",
    "Visa_cadelantosefectivo_delta1",
    "Visa_cadelantosefectivo_lag2",
    "Visa_cadelantosefectivo_delta2",
    "Visa_cadelantosefectivo_ols6",
    "Visa_cadelantosefectivo_sma3",
    "Visa_cadelantosefectivo_sma6",
    "Visa_cadelantosefectivo_min6",
    "Visa_cadelantosefectivo_max6",
    "Visa_cadelantosefectivo_ema3",
    "Visa_cadelantosefectivo_ema6",
    "Visa_mpagominimo_lag1",
    "Visa_mpagominimo_delta1",
    "Visa_mpagominimo_lag2",
    "Visa_mpagominimo_delta2",
    "Visa_mpagominimo_ols6",
    "Visa_mpagominimo_sma3",
    "Visa_mpagominimo_sma6",
    "Visa_mpagominimo_min6",
    "Visa_mpagominimo_max6",
    "Visa_mpagominimo_ema3",
    "Visa_mpagominimo_ema6",
    "fe67_mcuentas_saldo_median_prior6",
    "fe67_mcuentas_saldo_mad_prior6",
    "fe67_mcuentas_saldo_robust_deviation6",
    "fe67_mpayroll_median_prior6",
    "fe67_mpayroll_mad_prior6",
    "fe67_mpayroll_robust_deviation6"
  ],
  "entries": {
    "201901": {
      "rows": 124031,
      "hash": "cbbd7433785b0d2c2dd7cc90aacaf34c4720633e40f57de1ca1f86e05a89f8ce"
    },
    "201902": {
      "rows": 125124,
      "hash": "1974893c477918dff283bd92edc414400302ce89637375019029d836cbf77317"
    },
    "201903": {
      "rows": 125732,
      "hash": "3ed69ef6ceef17a41600277aace06afcfab8b7d088d01964a0417d2ad4ec5e17"
    },
    "201904": {
      "rows": 126246,
      "hash": "e8503d6306411ad243efbe60ace54fe2353bb0d37ddd898200e263f4eb9e4885"
    },
    "201905": {
      "rows": 126876,
      "hash": "6cf7dbb6573e6eb968603df818bb19fff812413536a351106e0184d8e33a11e1"
    },
    "201906": {
      "rows": 128474,
      "hash": "60342338b39e3921bfc1f6f3d70fa075ea7f2a71d6475d3f761c28cea01cba91"
    },
    "201907": {
      "rows": 129957,
      "hash": "636bf5413aa004cbc9cd1bad5c3e058909a345675299023d1933a8ec0ddcd3b2"
    },
    "201908": {
      "rows": 131887,
      "hash": "b89e52b38c1be4d394f657a6ca4617c11c2c5c352aa5d5ec6f69379699599a5b"
    },
    "201909": {
      "rows": 133538,
      "hash": "5f593654e05c282a94c92b9367ab4a1a4b246955034f2b0e5daccf7d43dab56c"
    },
    "201910": {
      "rows": 135754,
      "hash": "a01f83e3599124a4de55b169e2a37b8ab033aecb106ac4b0e88c2f64e8869d45"
    },
    "201911": {
      "rows": 137657,
      "hash": "4cd99bdb69f756c94eed30b79bcdf8904ec399e4003842ed96291c8b0c49c7ac"
    },
    "201912": {
      "rows": 139705,
      "hash": "40dada8c64133e80621c846566ad3f6a76fd8827097a8fdb1b71db1fe4c5354e"
    },
    "202001": {
      "rows": 143077,
      "hash": "af711cec814cad4cfc7125074614b4a70c7f97785d6e8cc2726156297641e165"
    },
    "202002": {
      "rows": 146347,
      "hash": "036af5a128299cf73455d100ba75a50e77a953753ae7b22fcea7fddb4d95facf"
    },
    "202003": {
      "rows": 148500,
      "hash": "b54c5ffcf453a9387dcf5dd8f2662cc885753a9417067f16787819a8197e7cb3"
    },
    "202004": {
      "rows": 149014,
      "hash": "b56b36f1d86e09dd28bc9de995c88cc3c9ee7991c0f5dea2ba912a8c9a1ea25d"
    },
    "202005": {
      "rows": 150447,
      "hash": "777c0ccf23d362572a3e058f2e633d6edc7ab125f412db3765a50c85b5df01bc"
    },
    "202006": {
      "rows": 152905,
      "hash": "17a6925fc88a1d41dd8ae6c458b72bfb95cdea37879643c2ad31c2dfebcd5184"
    },
    "202007": {
      "rows": 154960,
      "hash": "547bd0ddc750e00445e696157784ec20e3051a8a10143280ff3c421792a9b63a"
    },
    "202008": {
      "rows": 156413,
      "hash": "e56245dbf89d8334c712905115746a77828d246d068b2dab7d1467583f5e0bfd"
    },
    "202009": {
      "rows": 157722,
      "hash": "703f4148a2ec6c161c19175409ba96cf9a6ab851391c83c2fc17cd446210d141"
    },
    "202010": {
      "rows": 159140,
      "hash": "86e0a9cd72b07a07a025524ba9dea8fbcd51c439f54d30c707db6556ff42c13e"
    },
    "202011": {
      "rows": 160252,
      "hash": "9fd1d0b45bd61881ed0b89655e949704549d2b8b8a1091dbb6ff68c407e7b804"
    },
    "202012": {
      "rows": 161003,
      "hash": "d8927c39131942d80466a47f684204f8e4cfd6208f1153c5580bdd9d874f5376"
    },
    "202101": {
      "rows": 161520,
      "hash": "d57b598f8cacae6b4bda7a0c13920178bbcd7ae42ac7b56d60318b8c6c6de7a7"
    },
    "202102": {
      "rows": 162091,
      "hash": "09ceab9c57ea6a975c4b4705de0b933f45583326d60f64120c697c1ebff36f27"
    },
    "202103": {
      "rows": 163051,
      "hash": "6273c181fef9c9e8c7921b9bbc2e3ec807a4881fde4d103abeaa1d1c672e65f3"
    },
    "202104": {
      "rows": 163445,
      "hash": "2cac4d28b819d39153adca850035833c576786bdbb96238a4b293c1bc3eeb778"
    },
    "202105": {
      "rows": 163906,
      "hash": "a2e23b74bfca751ed27ad9c7e7594d1506da23037bc21c37865b7a706c92daeb"
    },
    "202106": {
      "rows": 164252,
      "hash": "3f32e602f0c594312d03f18687253f7716010f550cb230641ff9cc0090a4957d"
    },
    "202107": {
      "rows": 164479,
      "hash": "af3174ae7e127f259270a1fcba4265eeb922aff2b7050d73afd0a16085ccc437"
    },
    "202109": {
      "rows": 165076,
      "hash": "1d7947cde2963c2bb2d8964cc2fe65b2856acb903b6747115e5fb3bb35a5d3bc"
    }
  }
})---", simplifyVector=FALSE)


## Original feature definitions

Features are generated causally using the entire available covariate panel.
Calendar gaps remain gaps. Baseline history features retain their definitions;
full-history kernels add only names absent from the baseline. EMA and rolling
summaries include the current observation where specified by the original code.
The six MAD additions use strictly prior observations. August covariates remain
available when scoring September. August labels are incomplete (some BAJA+1
exits are observed); all August labels and all unknown September labels are
excluded from training.


In [ ]:
t7_month_index <- function (month) 
{
    stopifnot(all(month%%100L %in% 1:12))
    (month%/%100L) * 12L + month%%100L - 1L
}


In [ ]:
t7_month_value <- function (index) 
(index%/%12L) * 100L + index%%12L + 1L


In [ ]:
t7_features <- function (panel, history_columns, history_panel = panel) 
{
    keys <- c("numero_de_cliente", "foto_mes", "clase_ternaria")
    stopifnot(all(c(keys, history_columns) %in% names(panel)), !anyDuplicated(panel, by = c("numero_de_cliente", "foto_mes")), !any(history_columns %in% keys))
    snapshot <- setdiff(names(panel), keys)
    stopifnot(all(vapply(panel[, snapshot, with = FALSE], is.numeric, logical(1))))
    out <- data.table::copy(panel)
    stopifnot(!anyDuplicated(history_panel, by = c("numero_de_cliente", "foto_mes")))
    index <- history_panel[, .(numero_de_cliente, month_index = t7_month_index(foto_mes), row_id = .I)]
    data.table::setkey(index, numero_de_cliente, month_index)
    matches <- lapply(1:6, function(lag) {
        query <- panel[, .(numero_de_cliente, month_index = t7_month_index(foto_mes) - lag)]
        index[query, on = .(numero_de_cliente, month_index), row_id]
    })
    for (lag in 1:6) out[, `:=`((paste0("history_month", lag, "_present")), as.integer(!is.na(matches[[lag]])))]
    for (column in history_columns) {
        prior <- matrix(vapply(matches, function(rows) as.double(history_panel[[column]][rows]), numeric(nrow(panel))), nrow = nrow(panel), ncol = 6L)
        for (lag in c(1L, 2L, 3L, 6L)) out[, `:=`((paste0(column, "_lag", lag)), prior[, lag])]
        for (lag in c(1L, 3L)) out[, `:=`((paste0(column, "_delta", lag)), as.double(panel[[column]]) - prior[, lag])]
        for (window in c(3L, 6L)) {
            values <- prior[, seq_len(window), drop = FALSE]
            count <- rowSums(!is.na(values))
            mean <- rowMeans(values, na.rm = TRUE)
            mean[count == 0L] <- NA_real_
            out[, `:=`((paste0(column, "_prior", window, "_mean")), mean)]
            out[, `:=`((paste0(column, "_prior", window, "_observed")), count)]
        }
    }
    list(data = out, snapshot = snapshot, history = setdiff(names(out), keys))
}


In [ ]:
o54_history_kernel <- function () 
list(window = "Rcpp::NumericMatrix ht5954_window_summary(Rcpp::NumericVector cur, Rcpp::NumericMatrix prior) {\n    int n=cur.size(); Rcpp::NumericMatrix out(n,9);\n    for(int i=0;i<n;i++) {\n      double a=prior(i,0),b=prior(i,1);\n      out(i,0)=a;out(i,1)=cur[i]-a;out(i,2)=b;out(i,3)=cur[i]-b;\n      double sx=0,sy=0,sxy=0,sxx=0,lo=0,hi=0,s3=0;int k=0,k3=0;\n      for(int lag=0;lag<6;lag++) {\n        double v=lag==0?cur[i]:prior(i,lag-1);\n        if(!R_finite(v))continue;\n        double x=-lag;sx+=x;sy+=v;sxy+=x*v;sxx+=x*x;\n        if(k==0){lo=v;hi=v;}else{if(v<lo)lo=v;if(v>hi)hi=v;}\n        k++;if(lag<3){s3+=v;k3++;}\n      }\n      double den=k*sxx-sx*sx;\n      out(i,4)=(k>=2&&den>0)?(k*sxy-sx*sy)/den:NA_REAL;\n      out(i,5)=k3>=2?s3/k3:NA_REAL;out(i,6)=k>=2?sy/k:NA_REAL;\n      out(i,7)=k>=2?lo:NA_REAL;out(i,8)=k>=2?hi:NA_REAL;\n    } return out;\n  }", 
    ema = "Rcpp::NumericMatrix ht5954_calendar_ema(Rcpp::NumericVector value,Rcpp::NumericVector id,Rcpp::IntegerVector month) {\n    int n=value.size();Rcpp::NumericMatrix out(n,2);double num[2]={0,0},den[2]={0,0};int known=0;\n    double decay[2]={0.5,5.0/7.0};\n    for(int i=0;i<n;i++) {\n      bool fresh=i==0||id[i]!=id[i-1];int gap=fresh?0:month[i]-month[i-1];\n      if(!fresh&&gap<=0)Rcpp::stop(\"EMA requires unique increasing customer/month keys\");\n      if(fresh){known=0;num[0]=num[1]=den[0]=den[1]=0;}\n      bool valid=R_finite(value[i]);if(valid)known++;\n      for(int j=0;j<2;j++) {\n        double factor=std::pow(decay[j],gap);num[j]*=factor;den[j]*=factor;\n        if(valid){num[j]+=value[i];den[j]+=1;}\n        out(i,j)=known>=2&&den[j]>0?num[j]/den[j]:NA_REAL;\n      }\n    }return out;\n  }")


In [ ]:
o54_compile_history <- function () 
{
    code <- o54_history_kernel()
    signature <- digest::digest(code, algo = "sha256")
    if (!identical(get0(".ht5954_kernel_signature", envir = .GlobalEnv), signature)) {
        Rcpp::cppFunction(code$window, env = .GlobalEnv)
        Rcpp::cppFunction(code$ema, env = .GlobalEnv)
        assign(".ht5954_kernel_signature", signature, envir = .GlobalEnv)
    }
}


In [ ]:
o54_full_history_extra <- function (d, panel, index, fields) 
{
    stopifnot(!"clase_ternaria" %in% names(panel), !any(c("numero_de_cliente", "foto_mes", "clase_ternaria") %in% fields))
    positions <- lapply(1:5, function(lag) index[d[, .(numero_de_cliente, month_index = t7_month_index(foto_mes) - lag)], on = .(numero_de_cliente, month_index), row_id])
    current_pos <- index[d[, .(numero_de_cliente, month_index = t7_month_index(foto_mes))], on = .(numero_de_cliente, month_index), row_id]
    stopifnot(!anyNA(current_pos))
    out <- data.table::data.table(numero_de_cliente = d$numero_de_cliente)
    for (field in fields) {
        previous <- matrix(vapply(positions, function(i) as.double(panel[[field]][i]), numeric(nrow(d))), nrow = nrow(d), ncol = 5L)
        values <- ht5954_window_summary(as.double(d[[field]]), previous)
        names <- paste0(field, c("_lag1", "_delta1", "_lag2", "_delta2", "_ols6", "_sma3", "_sma6", "_min6", "_max6"))
        for (j in seq_along(names)) if (!names[j] %in% colnames(d)) 
            out[, `:=`((names[j]), values[, j])]
        out[, `:=`((paste0(field, "_ema3")), panel[[paste0(".ema3_", field)]][current_pos])]
        out[, `:=`((paste0(field, "_ema6")), panel[[paste0(".ema6_", field)]][current_pos])]
    }
    out
}


In [ ]:
f67_values <- function (panel, rows, index) 
{
    d <- panel[rows]
    out <- d[, .(numero_de_cliente, foto_mes)]
    at <- lapply(1:6, function(k) {
        query <- data.table::data.table(id = d$numero_de_cliente, month = d$.month - k)
        index[query, on = .(id, month), row]
    })
    for (f in c("mcuentas_saldo", "mpayroll")) {
        x <- matrix(vapply(at, function(i) as.double(panel[[f]][i]), numeric(nrow(d))), nrow = nrow(d), ncol = 6L)
        x[!is.finite(x)] <- NA_real_
        known <- is.finite(x)
        n <- rowSums(known)
        p3 <- x[, 1:3, drop = FALSE]
        k3 <- is.finite(p3)
        n3 <- rowSums(k3)
        low <- matrixStats::rowMins(x, na.rm = TRUE)
        high <- matrixStats::rowMaxs(x, na.rm = TRUE)
        center <- matrixStats::rowMedians(x, na.rm = TRUE)
        scale <- matrixStats::rowMedians(abs(x - center), na.rm = TRUE)
        for (name in c("low", "high", "center", "scale")) {
            v <- get(name)
            v[n < 3L] <- NA_real_
            assign(name, v)
        }
        time <- -seq_len(3L)
        y <- p3
        y[!k3] <- 0
        sx <- rowSums(sweep(k3, 2L, time, `*`))
        sy <- rowSums(y)
        sxx <- rowSums(sweep(k3, 2L, time^2, `*`))
        sxy <- rowSums(sweep(y, 2L, time, `*`))
        slope <- (n3 * sxy - sx * sy)/(n3 * sxx - sx^2)
        projection <- (sy - slope * sx)/pmax(n3, 1L)
        bad <- n3 < 2L | !is.finite(slope)
        slope[bad] <- NA_real_
        projection[bad] <- NA_real_
        relative <- function(value, ref) pmax(-10, pmin(10, (value - ref)/pmax(abs(ref), 1)))
        values <- list(min_prior6 = low, max_prior6 = high, range_prior6 = high - low, slope_prior3 = slope, relative_trend3 = relative(as.double(d[[f]]), projection), median_prior6 = center, mad_prior6 = scale, robust_deviation6 = pmax(-10, pmin(10, (as.double(d[[f]]) - center)/pmax(scale, 1))), known_prior3 = n3, known_prior6 = n)
        for (name in names(values)) data.table::set(out, j = paste0("fe67_", f, "_", name), value = values[[name]])
    }
    stopifnot(!any(vapply(out, function(v) any(is.infinite(v) | is.nan(v)), logical(1))))
    out
}


In [ ]:
f67_views <- function (base) 
{
    fields <- c("mcuentas_saldo", "mpayroll")
    cols <- function(suffix) unlist(lapply(fields, function(f) paste0("fe67_", f, "_", suffix)))
    ext <- cols(c("min_prior6", "max_prior6", "range_prior6"))
    slope <- cols(c("slope_prior3", "relative_trend3"))
    mad <- cols(c("median_prior6", "mad_prior6", "robust_deviation6"))
    list(E_extremes = c(base, ext), S_slope = c(base, slope), M_mad = c(base, mad), U_union = c(base, ext, slope, mad))
}


In [ ]:
r66_index <- function (month) 
{
    stopifnot(all(month%%100L %in% 1:12))
    month%/%100L * 12L + month%%100L - 1L
}


In [ ]:
r66_month <- function (index) 
index%/%12L * 100L + index%%12L + 1L


In [ ]:
r66_block <- function (month) 
month >= 202003L & month <= 202103L


In [ ]:
r66_eligible <- function (origin) 
r66_month(seq.int(r66_index(201901L), r66_index(origin) - 2L))


In [ ]:
r66_weights <- function (month) 
{
    raw <- ifelse(r66_block(month), 0.25, 1)
    raw/mean(raw)
}


## Reconstruction, training, and verification helpers


In [ ]:
# Self-contained notebook helpers. The builder embeds these definitions verbatim.
ht76_sha <- function(path) digest::digest(file=path,algo="sha256")
ht76_hash <- function(x) digest::digest(x,algo="sha256")

ht76_json <- function(x,path) {
  dir.create(dirname(path),recursive=TRUE,showWarnings=FALSE)
  tmp<-paste0(path,".pending_",Sys.getpid())
  jsonlite::write_json(x,tmp,auto_unbox=TRUE,pretty=TRUE,digits=17,null="null",na="null")
  stopifnot(file.rename(tmp,path));invisible(x)
}

ht76_save <- function(x,path) {
  dir.create(dirname(path),recursive=TRUE,showWarnings=FALSE)
  tmp<-paste0(path,".pending_",Sys.getpid());on.exit(unlink(tmp),add=TRUE)
  con<-gzfile(tmp,"wb",compression=1L)
  tryCatch(saveRDS(x,con,version=3L),finally=close(con))
  stopifnot(file.rename(tmp,path));invisible(NULL)
}

ht76_state <- function(cfg,phase,...) {
  x<-c(list(phase=phase,utc=format(Sys.time(),tz="UTC",usetz=TRUE),pid=Sys.getpid()),list(...))
  ht76_json(x,file.path(cfg$output,"status.json"));cat(x$utc,phase,"\n");flush.console()
}

ht76_memory <- function() {
  line<-readLines("/proc/meminfo",warn=FALSE)
  as.double(sub("^MemAvailable: +([0-9]+) kB$","\\1",line[grepl("^MemAvailable:",line)]))/1024^2
}

ht76_disk <- function(path) {
  lines<-system2("df",c("-Pk",shQuote(path)),stdout=TRUE)
  parts<-strsplit(trimws(lines[length(lines)])," +")[[1]]
  as.double(parts[4])/1024^2
}

ht76_preflight <- function(cfg,recipe) {
  packages<-c("data.table","digest","jsonlite","matrixStats","Rcpp","lightgbm")
  stopifnot(all(vapply(packages,requireNamespace,logical(1),quietly=TRUE)))
  suppressPackageStartupMessages(library(data.table));suppressPackageStartupMessages(library(lightgbm))
  data.table::setDTthreads(4L)
  stopifnot(identical(R.version.string,recipe$R),identical(R.version$platform,recipe$platform))
  for(p in packages)stopifnot(as.character(utils::packageVersion(p))==recipe$package_versions[[p]])
  stopifnot(ht76_sha(getLoadedDLLs()[["lightgbm"]][["path"]])==recipe$native_sha256)
  stopifnot(file.exists(cfg$raw),ht76_sha(cfg$raw)==recipe$dataset_sha256)
  stopifnot(cfg$workers%in%1:3,normalizePath(cfg$output,mustWork=FALSE)!=dirname(normalizePath(cfg$raw)))
  dir.create(cfg$output,recursive=TRUE,showWarnings=FALSE)
  dir.create(cfg$scratch,recursive=TRUE,showWarnings=FALSE)
  for(folder in c("frames","models","predictions","workers"))dir.create(file.path(cfg$output,folder),showWarnings=FALSE)
  stopifnot(ht76_disk(cfg$output)>cfg$disk_reserve_gib,ht76_disk(tempdir())>cfg$disk_reserve_gib,
    ht76_disk(cfg$scratch)>cfg$disk_reserve_gib,ht76_memory()>cfg$construction_gib)
  fs<-system2("df",c("-PTk",shQuote(cfg$scratch)),stdout=TRUE)
  if(grepl("fuse|tmpfs|ramfs",strsplit(trimws(fs[length(fs)])," +")[[1]][2]))
    stop("Constructor scratch must be a local disk directory, not FUSE or RAM")
  manifest<-list(recipe_hash=ht76_hash(recipe),reference_hash=cfg$reference_hash,
    engine_hash=cfg$engine_hash,raw_sha256=ht76_sha(cfg$raw),
    R=R.version.string,native_sha256=recipe$native_sha256)
  path<-file.path(cfg$output,"contract.rds")
  if(file.exists(path))stopifnot(identical(readRDS(path),manifest))else ht76_save(manifest,path)
  compiler<-list(CXX=system2(file.path(R.home("bin"),"R"),c("CMD","config","CXX"),stdout=TRUE),
    CXXFLAGS=system2(file.path(R.home("bin"),"R"),c("CMD","config","CXXFLAGS"),stdout=TRUE))
  ht76_json(list(contract=manifest,compiler=compiler,packages=recipe$package_versions),file.path(cfg$output,"runtime.json"))
  ht76_state(cfg,"preflight_passed",contract_sha256=ht76_sha(path));invisible(manifest)
}

ht76_fingerprint <- function(frame) {
  ht76_hash(list(ids=frame$ids,label=frame$label,x=unname(frame$x)))
}

ht76_check_frame <- function(frame,month,recipe,golden) {
  expected<-golden$entries[[as.character(month)]]
  stopifnot(!is.null(expected),nrow(frame$x)==expected$rows,ncol(frame$x)==length(recipe$mad_columns),
    identical(colnames(frame$x),unlist(recipe$mad_columns)),length(frame$ids)==nrow(frame$x),
    !anyNA(frame$ids),!anyDuplicated(frame$ids),ht76_fingerprint(frame)==expected$hash)
  invisible(TRUE)
}

ht76_context <- function(raw,recipe) {
  stopifnot(nrow(raw)==recipe$rows,data.table::uniqueN(raw$numero_de_cliente)==recipe$customers,
    !anyDuplicated(raw,by=c("numero_de_cliente","foto_mes")),
    identical(setdiff(names(raw),c("numero_de_cliente","foto_mes","clase_ternaria")),unlist(recipe$raw_fields)))
  stopifnot(all(raw$foto_mes%in%r66_month(seq.int(r66_index(201901L),r66_index(202109L)))))
  train<-raw$foto_mes%in%unlist(recipe$train_months);future<-raw$foto_mes==202109L
  stopifnot(sum(train)==recipe$training_rows,sum(raw$foto_mes==202109L)==recipe$prediction_rows,
    all(raw$clase_ternaria[train]%in%c("CONTINUA","BAJA+1","BAJA+2")),
    all(is.na(raw$clase_ternaria[future])|raw$clase_ternaria[future]==""))
  # August includes some observed BAJA+1 exits, but not a complete two-month
  # horizon. Every August label is excluded, including those partial positives.
  history<-raw[,c("numero_de_cliente","foto_mes",unlist(recipe$history_fields)),with=FALSE]
  panel<-raw[,c("numero_de_cliente","foto_mes",unlist(recipe$raw_fields)),with=FALSE]
  data.table::setorder(panel,numero_de_cliente,foto_mes)
  o54_compile_history()
  months<-as.integer(t7_month_index(panel$foto_mes));ids<-as.double(panel$numero_de_cliente)
  for(field in unlist(recipe$raw_fields)) {
    ema<-ht5954_calendar_ema(as.double(panel[[field]]),ids,months)
    data.table::set(panel,j=paste0(".ema3_",field),value=ema[,1L])
    data.table::set(panel,j=paste0(".ema6_",field),value=ema[,2L])
  }
  index<-panel[,.(numero_de_cliente,month_index=t7_month_index(foto_mes),row_id=.I)]
  data.table::setkey(index,numero_de_cliente,month_index)
  mad_panel<-panel[,c("numero_de_cliente","foto_mes","mcuentas_saldo","mpayroll"),with=FALSE]
  mad_panel[,.month:=r66_index(foto_mes)]
  mad_index<-mad_panel[,.(id=numero_de_cliente,month=.month,row=.I)];data.table::setkey(mad_index,id,month)
  list(raw=raw,history=history,panel=panel,index=index,mad_panel=mad_panel,mad_index=mad_index)
}

ht76_month <- function(ctx,month,recipe) {
  f<-t7_features(ctx$raw[foto_mes==month],unlist(recipe$history_fields),ctx$history)
  d<-f$data
  extra<-o54_full_history_extra(d,ctx$panel,ctx$index,unlist(recipe$raw_fields))
  for(name in setdiff(names(extra),"numero_de_cliente"))data.table::set(d,j=name,value=extra[[name]])
  stopifnot(identical(c(f$history,setdiff(names(extra),"numero_de_cliente")),unlist(recipe$base_columns)))
  added<-f67_values(ctx$mad_panel,which(ctx$mad_panel$foto_mes==month),ctx$mad_index)
  at<-match(d$numero_de_cliente,added$numero_de_cliente);stopifnot(!anyNA(at),nrow(d)==nrow(added))
  for(name in setdiff(unlist(recipe$mad_columns),unlist(recipe$base_columns)))data.table::set(d,j=name,value=added[[name]][at])
  y<-if(month==202109L)NULL else match(as.character(d$clase_ternaria),c("CONTINUA","BAJA+1","BAJA+2"))-1L
  list(ids=d$numero_de_cliente,label=y,x=data.matrix(d[,unlist(recipe$mad_columns),with=FALSE]))
}

ht76_prepare <- function(cfg,recipe,golden) {
  months<-as.integer(c(unlist(recipe$train_months),recipe$target_month));ctx<-NULL;entries<-list()
  for(month in months) {
    path<-file.path(cfg$output,"frames",paste0(month,".rds"))
    if(file.exists(path)) {
      frame<-readRDS(path);ht76_check_frame(frame,month,recipe,golden)
    } else {
      if(is.null(ctx)) {
        ht76_state(cfg,"reading_raw_and_preparing_history")
        ctx<-ht76_context(data.table::fread(cfg$raw,showProgress=FALSE),recipe)
      }
      stopifnot(ht76_disk(cfg$output)>cfg$disk_reserve_gib,ht76_disk(tempdir())>cfg$disk_reserve_gib)
      ht76_state(cfg,"reconstructing_month",month=month)
      frame<-ht76_month(ctx,month,recipe);ht76_check_frame(frame,month,recipe,golden)
      ht76_save(frame,path)
    }
    entries[[as.character(month)]]<-list(rows=nrow(frame$x),sha256=ht76_sha(path),fingerprint=golden$entries[[as.character(month)]]$hash)
    ht76_json(list(state="validating_features",months=entries),file.path(cfg$output,"features.json"))
    ht76_state(cfg,"feature_month_verified",month=month,completed=length(entries),total=length(months))
    rm(frame);gc()
  }
  stopifnot(length(entries)==32L)
  result<-list(state="all_32_raw_feature_frames_match",months=entries)
  ht76_json(result,file.path(cfg$output,"features.json"));result
}

ht76_frame <- function(cfg,month) {
  manifest<-jsonlite::read_json(file.path(cfg$output,"features.json"),simplifyVector=FALSE)
  stopifnot(manifest$state=="all_32_raw_feature_frames_match")
  path<-file.path(cfg$output,"frames",paste0(month,".rds"))
  stopifnot(ht76_sha(path)==manifest$months[[as.character(month)]]$sha256)
  readRDS(path)
}

ht76_construct <- function(cfg,recipe,family,seed) {
  mutex<-file.path(cfg$scratch,"constructor.lock")
  repeat {
    if(dir.create(mutex,showWarnings=FALSE))break
    owner<-file.path(mutex,"owner.rds")
    if(file.exists(owner)) {
      pid<-readRDS(owner)$pid
      if(!file.exists(file.path("/proc",as.character(pid))))stop("Constructor owner exited; inspect and explicitly recover its lock")
    }
    Sys.sleep(2)
  }
  ht76_save(list(pid=Sys.getpid(),host=Sys.info()[["nodename"]]),file.path(mutex,"owner.rds"))
  on.exit(unlink(mutex,recursive=TRUE),add=TRUE)
  stopifnot(ht76_memory()>cfg$construction_gib)
  columns<-unlist(if(family=="BASE")recipe$base_columns else recipe$mad_columns)
  n<-as.integer(recipe$training_rows)
  x<-matrix(NA_real_,n,length(columns),dimnames=list(NULL,columns));y<-integer(n);m<-integer(n);ids<-integer(n);offset<-0L
  for(month in as.integer(unlist(recipe$train_months))) {
    f<-ht76_frame(cfg,month);at<-seq.int(offset+1L,offset+length(f$ids))
    x[at,]<-f$x[,columns,drop=FALSE];y[at]<-f$label;ids[at]<-f$ids;m[at]<-month
    offset<-offset+length(f$ids);rm(f);gc()
  }
  weights<-r66_weights(m)
  proof<-list(rows=offset,row_hash=ht76_hash(list(ids=ids,month=m,label=y)),weight_hash=ht76_hash(weights),columns_hash=ht76_hash(columns))
  expected<-recipe$expected_models[[paste(family,seed,sep="_")]]
  stopifnot(offset==n,!anyNA(y))
  # Check by column instead of allocating an extra 8.7-billion-element mask.
  stopifnot(!any(vapply(seq_len(ncol(x)),function(j)any(is.infinite(x[,j])),logical(1))))
  for(key in names(proof))stopifnot(identical(proof[[key]],expected[[key]]))
  # Bins are constructed before weights, exactly as in both original model families.
  construction<-recipe$construction[[as.character(seed)]]
  ds<-lightgbm::lgb.Dataset(x,label=y,params=construction,free_raw_data=TRUE)
  lightgbm::lgb.Dataset.construct(ds);lightgbm::set_field(ds,"weight",weights)
  native_weights<-lightgbm::get_field(ds,"weight")
  stopifnot(length(native_weights)==n,max(abs(native_weights-weights))<1e-6)
  rm(x,y,m,ids,weights,native_weights);gc()
  list(dataset=ds,proof=proof)
}

ht76_fit <- function(task,cfg,recipe) {
  suppressPackageStartupMessages(library(data.table));suppressPackageStartupMessages(library(lightgbm));data.table::setDTthreads(4L)
  key<-paste(task$family,task$seed,sep="_");expected<-recipe$expected_models[[key]]
  state<-function(phase,...)ht76_json(c(list(task=key,pid=Sys.getpid(),phase=phase,utc=format(Sys.time(),tz="UTC",usetz=TRUE)),list(...)),file.path(cfg$output,"workers",paste0(key,".json")))
  tryCatch({
    model_path<-file.path(cfg$output,"models",paste0(key,".rds"));pred_path<-file.path(cfg$output,"predictions",paste0(key,".rds"))
    receipt<-paste0(pred_path,".receipt.json")
    if(file.exists(receipt)) {
      r<-jsonlite::read_json(receipt);stopifnot(ht76_sha(model_path)==r$model_sha256,ht76_sha(pred_path)==r$prediction_sha256)
      v<-readRDS(pred_path);stopifnot(ht76_hash(list(ids=v$ids,score=v$score))==expected$prediction_hash)
      state("complete_reused");return(list(task=key,prediction_hash=expected$prediction_hash))
    }
    p<-recipe$params[[key]];rounds<-as.integer(recipe$rounds[[task$family]])
    if(file.exists(model_path)) {
      saved<-readRDS(model_path);stopifnot(saved$key==key,identical(saved$params,p),saved$rounds==rounds)
    } else {
      state("constructing");data<-ht76_construct(cfg,recipe,task$family,as.integer(task$seed))
      state("training",rows=data$proof$rows);set.seed(as.integer(task$seed),kind="L'Ecuyer-CMRG")
      start<-Sys.time()
      # BASE used the API default; MAD explicitly retained its Dataset.
      if(task$family=="BASE")model<-lightgbm::lgb.train(p,data$dataset,nrounds=rounds,verbose=-1L)
      else model<-lightgbm::lgb.train(p,data$dataset,nrounds=rounds,verbose=-1L,reset_data=FALSE)
      temp<-tempfile("ht76_native_",tmpdir=cfg$scratch);on.exit(unlink(temp),add=TRUE)
      lightgbm::lgb.save(model,temp,num_iteration=-1L);payload<-paste(readLines(temp,warn=FALSE),collapse="\n")
      stopifnot(sum(grepl("^Tree=[0-9]+$",strsplit(payload,"\n",fixed=TRUE)[[1]]))==3L*rounds)
      saved<-list(key=key,params=p,rounds=rounds,proof=data$proof,payload=payload,native_seconds=as.numeric(difftime(Sys.time(),start,units="secs")))
      ht76_save(saved,model_path);rm(model,data);gc()
    }
    state("predicting");f<-ht76_frame(cfg,as.integer(recipe$target_month))
    columns<-unlist(if(task$family=="BASE")recipe$base_columns else recipe$mad_columns)
    model<-lightgbm::lgb.load(model_str=saved$payload)
    p<-predict(model,f$x[,columns,drop=FALSE],num_iteration=rounds,params=list(num_threads=4L))
    stopifnot(is.matrix(p),ncol(p)==3L,all(is.finite(p)),max(abs(rowSums(p)-1))<1e-7)
    v<-list(ids=f$ids,score=as.numeric(p[,3L]))
    got<-ht76_hash(v);stopifnot(got==expected$prediction_hash)
    sample<-unique(as.integer(round(seq(1,length(v$score),length.out=min(512L,length(v$score))))))
    reloaded<-lightgbm::lgb.load(model_str=saved$payload)
    replay<-predict(reloaded,f$x[sample,columns,drop=FALSE],num_iteration=rounds,params=list(num_threads=1L))
    replay_error<-max(abs(v$score[sample]-replay[,3L]));stopifnot(replay_error<1e-12)
    ht76_save(v,pred_path)
    ht76_json(list(state="exact_prediction_match",task=key,prediction_hash=got,replay_error=replay_error,
      model_sha256=ht76_sha(model_path),prediction_sha256=ht76_sha(pred_path)),receipt)
    state("complete",prediction_hash=got);list(task=key,prediction_hash=got)
  },error=function(error){state("failed",error=conditionMessage(error));stop(error)})
}

ht76_train <- function(cfg,recipe) {
  stopifnot(ht76_memory()>cfg$construction_gib)
  ht76_state(cfg,"training_six_models",workers=cfg$workers,threads_per_worker=4L)
  tasks<-unlist(lapply(c("BASE","MAD"),function(family)lapply(as.integer(unlist(recipe$seeds)),function(seed)list(family=family,seed=seed))),recursive=FALSE)
  cluster<-parallel::makePSOCKcluster(cfg$workers,outfile=file.path(cfg$output,"training.log"))
  on.exit(parallel::stopCluster(cluster),add=TRUE)
  symbols<-c(ls(.GlobalEnv,pattern="^ht76_"),"r66_weights","r66_block")
  symbols<-symbols[vapply(mget(symbols,.GlobalEnv),is.function,logical(1))]
  parallel::clusterExport(cluster,symbols,envir=.GlobalEnv)
  results<-parallel::parLapplyLB(cluster,tasks,ht76_fit,cfg=cfg,recipe=recipe)
  ht76_json(list(state="six_prediction_hashes_match",results=results),file.path(cfg$output,"training.json"))
  results
}

ht76_export <- function(cfg,recipe) {
  values<-list();ids<-NULL
  for(family in c("BASE","MAD")) {
    parts<-lapply(as.integer(unlist(recipe$seeds)),function(seed) {
      key<-paste(family,seed,sep="_");path<-file.path(cfg$output,"predictions",paste0(key,".rds"))
      r<-jsonlite::read_json(paste0(path,".receipt.json"));v<-readRDS(path)
      stopifnot(r$state=="exact_prediction_match",ht76_sha(path)==r$prediction_sha256,
        ht76_sha(file.path(cfg$output,"models",paste0(key,".rds")))==r$model_sha256,
        ht76_hash(v)==recipe$expected_models[[key]]$prediction_hash)
      v
    })
    if(is.null(ids))ids<-parts[[1L]]$ids
    values[[family]]<-Reduce(`+`,lapply(parts,function(v){at<-match(ids,v$ids);stopifnot(!anyNA(at));v$score[at]}))/3
  }
  percentile<-function(x)(base::rank(x,ties.method="average")-.5)/length(x)
  score<-.75*percentile(values$BASE)+.25*percentile(values$MAD)
  chosen<-order(-score,ids)[seq_len(as.integer(recipe$blend$contacts))]
  selected<-integer(length(ids));selected[chosen]<-1L
  d<-data.table::data.table(numero_de_cliente=ids,Predicted=selected);data.table::setorder(d,numero_de_cliente)
  path<-file.path(cfg$output,recipe$final_filename)
  if(file.exists(path))stopifnot(ht76_sha(path)==recipe$final_csv_sha256)else {
    tmp<-paste0(path,".pending");data.table::fwrite(d,tmp)
    stopifnot(ht76_sha(tmp)==recipe$final_csv_sha256,file.rename(tmp,path))
  }
  stopifnot(nrow(d)==recipe$prediction_rows,sum(d$Predicted)==recipe$blend$contacts,ht76_sha(path)==recipe$final_csv_sha256)
  result<-list(state="exact_raw_to_submission_reproduction",filename=recipe$final_filename,sha256=ht76_sha(path),rows=nrow(d),contacts=sum(d$Predicted),
    public_score=recipe$public_score,user_reported_final_score=recipe$user_reported_final_score,source="original dataset plus embedded code and frozen recipe",utc=format(Sys.time(),tz="UTC",usetz=TRUE))
  ht76_json(result,file.path(cfg$output,"verification.json"));ht76_state(cfg,"complete",sha256=result$sha256);result
}


In [ ]:
cfg <- list(raw=RAW_DATA, output=OUTPUT_DIR, scratch=SCRATCH_DIR, workers=WORKERS,
  construction_gib=136, disk_reserve_gib=8,
  reference_hash=digest::digest(golden, algo="sha256"),
  engine_hash="02c85e1b18c0b3ad2d6ef5d090749d473bbe5ffc5137ce2200a2a9aeddf1df51")
if (RUN_NOTEBOOK) ht76_preflight(cfg, recipe) else message("Definitions only; raw input was not opened.")


## 1 · Reconstruct and verify all monthly frames

Training months are January 2019–July 2021, in ascending order, with the original
within-month row order. Each of the 32 frames must match its independent reference
fingerprint before training is allowed. Compressed caches are created here from
raw input, never imported from earlier experiments.


In [ ]:
if (RUN_NOTEBOOK) {
  features <- ht76_prepare(cfg, recipe, golden)
  print(features$state)
  gc()
}


## 2 · Six fresh native fits

Three independent seeds for BASE (192 rounds) and MAD (384 rounds). A single
constructor assembles each matrix, checks ordered row/label, column, and weight
hashes, constructs unweighted bins, and then applies the quarter-period weights.
Each saved model is reloaded for prediction. Every September probability-vector
hash must equal its original counterpart before it is accepted. Checkpoints may
only be reused from this reconstruction's matching output contract.


In [ ]:
if (RUN_NOTEBOOK && RUN_TRAINING) {
  training <- ht76_train(cfg, recipe)
  print(training)
}


## 3 · Exact blend and submitted CSV

Average probabilities in seed order within each family. Convert each family mean
to average-tie percentile ranks, combine 75% BASE +25% MAD, and select the first
8,250 by descending blend and ascending customer ID. Write rows in ascending ID
order and assert the submitted file's byte hash. `verification.json` exists only
after the entire acceptance chain passes.


In [ ]:
if (RUN_NOTEBOOK && RUN_TRAINING) {
  verification <- ht76_export(cfg, recipe)
  print(verification)
} else if (RUN_NOTEBOOK) {
  message("Feature-only run: no claim of fitted-model or CSV reproduction.")
}
